In [ ]:
#pip install datasets pillow pandas tqdm

In [ ]:
## STEP 1: IMPORTS AND CONFIGURATION

import os
import json
from datasets import load_dataset
from PIL import Image
from collections import defaultdict
from tqdm import tqdm
import random

### 25 Selected Classes with COCO Category IDs

**Vehicles:** car(3), truck(8), bus(6), motorcycle(4), bicycle(2), airplane(5)  
**Person:** person(1)  
**Outdoor:** traffic light(10), stop sign(13), bench(15)  
**Animals:** dog(18), cat(17), horse(19), bird(16), cow(21), elephant(22)  
**Kitchen & Food:** bottle(44), cup(47), bowl(51), pizza(59), cake(61)  
**Furniture:** chair(62), couch(63), bed(65), potted plant(64)

In [ ]:
# 25 Selected Classes for SmartVision AI

SELECTED_CLASSES = {
    'person': 0,
    'bicycle': 1,
    'car': 2,
    'motorcycle': 3,
    'airplane': 4,
    'bus': 5,
    'truck': 7,
    'traffic light': 9,
    'stop sign': 11,
    'bench': 13,
    'bird': 14,
    'cat': 15,
    'dog': 16,
    'horse': 17,
    'cow': 19,
    'elephant': 20,
    'bottle': 39,
    'cup': 41,
    'bowl': 45,
    'pizza': 53,
    'cake': 55,
    'chair': 56,
    'couch': 57,
    'potted plant': 58,
    'bed': 59
}

IMAGES_PER_CLASS = 100

BASE_DIR = "smartvision_dataset"

print("Total selected classes:", len(SELECTED_CLASSES))

assert len(SELECTED_CLASSES) == 25

print("Correct 25-class configuration!")

In [ ]:
## STEP 2: LOAD COCO DATASET FROM HUGGING FACE

print("📥 Loading COCO dataset in STREAMING mode (no download)...")
dataset = load_dataset("detection-datasets/coco", split="train", streaming=True)
print("✅ Dataset loaded in streaming mode!")

In [ ]:
## STEP 3: COLLECT IMAGES FROM STREAM

print("\n🔍 Starting image collection from COCO dataset stream...")
print(f"🎯 Target: {IMAGES_PER_CLASS} images per class")
print()

# Initialize storage for collected images
class_images = {class_name: [] for class_name in SELECTED_CLASSES.keys()}
class_counts = {class_name: 0 for class_name in SELECTED_CLASSES.keys()}

# Progress tracking
total_collected = 0
images_processed = 0
max_iterations = 50000  # Safety limit

print("⏳ Processing images from stream...")
print("💡 Progress updates every 100 images collected")
print()

# Iterate through streaming dataset
for idx, item in enumerate(dataset):

    images_processed += 1

    # Progress update every 1000 images processed
    if images_processed % 1000 == 0:
        print(f"📊 Processed {images_processed} images | Collected {total_collected}/{len(SELECTED_CLASSES) * IMAGES_PER_CLASS}")

    # Safety check
    if images_processed >= max_iterations:
        print(f"⚠️ Reached safety limit of {max_iterations} iterations")
        break

    # Check if we have enough images for ALL classes
    if all(count >= IMAGES_PER_CLASS for count in class_counts.values()):
        print("🎉 Successfully collected 100 images for ALL classes!")
        break

    # Get annotations from current image
    annotations = item['objects']
    categories = annotations['category']

    # Check if any of our target classes are in this image
    for cat_id in categories:
        for class_name, class_id in SELECTED_CLASSES.items():
            if cat_id == class_id and class_counts[class_name] < IMAGES_PER_CLASS:

                # Store the ACTUAL image data (not just index!)
                class_images[class_name].append({
                    'image': item['image'],           # PIL Image object
                    'annotations': item['objects'],   # Annotations
                    'idx': images_processed           # For naming
                })

                class_counts[class_name] += 1
                total_collected += 1

                # Progress update every 100 collected
                if total_collected % 100 == 0:
                    print(f"✓ Collected {total_collected}/{len(SELECTED_CLASSES) * IMAGES_PER_CLASS} images")

                break  # Only count once per class

print()
print("="*60)
print("📊 COLLECTION COMPLETE:")
print("="*60)
print(f"Images Processed: {images_processed}")
print(f"Images Collected: {total_collected}")
print()
for class_name, count in sorted(class_counts.items()):
    status = "✅" if count >= IMAGES_PER_CLASS else "⚠️"
    print(f"{status} {class_name:20s}: {count:3d} images")
print("="*60)

In [ ]:
## STEP 4: CREATE FOLDER STRUCTURE

print("\n📁 Creating project folder structure...")
print()

# Create main directory
os.makedirs(BASE_DIR, exist_ok=True)

# Create subdirectories for Classification task
os.makedirs(f"{BASE_DIR}/classification/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/classification/val", exist_ok=True)
os.makedirs(f"{BASE_DIR}/classification/test", exist_ok=True)

# Create subdirectories for Detection task
os.makedirs(f"{BASE_DIR}/detection/images", exist_ok=True)
os.makedirs(f"{BASE_DIR}/detection/labels", exist_ok=True)

# Create class folders inside train/val/test
for class_name in SELECTED_CLASSES.keys():
    os.makedirs(f"{BASE_DIR}/classification/train/{class_name}", exist_ok=True)
    os.makedirs(f"{BASE_DIR}/classification/val/{class_name}", exist_ok=True)
    os.makedirs(f"{BASE_DIR}/classification/test/{class_name}", exist_ok=True)

print("✅ Folder structure created successfully!")
print()
print("📂 Structure:")
print(f"""
{BASE_DIR}/
├── classification/
│   ├── train/
│   │   ├── person/
│   │   ├── car/
│   │   └── ... (25 class folders)
│   ├── val/
│   │   └── ... (25 class folders)
│   └── test/
│       └── ... (25 class folders)
│
└── detection/
    ├── images/
    └── labels/
""")

In [ ]:
## STEP 5: TRAIN/VAL/TEST SPLIT (70/15/15)

print("="*70)
print("🔀 Preparing Train/Val/Test splits...")
print("📊 Split Ratio: 70% Train / 15% Val / 15% Test")
print("="*70)
print()

# Initialize metadata dictionary
metadata = {
    'total_images': 0,
    'classes': {},
    'splits': {'train': 0, 'val': 0, 'test': 0}
}

# Create split dictionaries for each class
train_data = {}
val_data = {}
test_data = {}

# Process each class
for class_name in SELECTED_CLASSES.keys():

    all_items = class_images.get(class_name, [])

    if not all_items:
        print(f"⚠️ Warning: No images found for {class_name}")
        continue

    # Calculate split indices
    n = len(all_items)
    train_split = int(0.7 * n)   # 70% for training
    val_split = int(0.85 * n)    # 15% for validation
    # Remaining 15% for test

    # Split the data
    train_data[class_name] = all_items[:train_split]
    val_data[class_name] = all_items[train_split:val_split]
    test_data[class_name] = all_items[val_split:]

    # Store split info in metadata
    metadata['classes'][class_name] = {
        'train': len(train_data[class_name]),
        'val': len(val_data[class_name]),
        'test': len(test_data[class_name]),
        'total': len(all_items)
    }

    metadata['splits']['train'] += len(train_data[class_name])
    metadata['splits']['val'] += len(val_data[class_name])
    metadata['splits']['test'] += len(test_data[class_name])
    metadata['total_images'] += len(all_items)

    print(f"{class_name:20s}: Train={len(train_data[class_name]):3d} | Val={len(val_data[class_name]):2d} | Test={len(test_data[class_name]):2d}")

In [ ]:
import os
from PIL import Image
from tqdm import tqdm
import json

print("="*70)
print("💾 STEP 6: SAVING IMAGES TO DISK")
print("="*70)
print()

# PART A: SAVE CLASSIFICATION IMAGES


print("📁 PART A: Saving Classification Images...")
print("   Format: Cropped objects, 224x224 pixels\n")

classification_stats = {'train': 0, 'val': 0, 'test': 0}

# Process each split
for split_name, split_data in [('train', train_data), ('val', val_data), ('test', test_data)]:

    print(f"📂 Processing {split_name.upper()} split...")

    # Process each class
    for class_name, items in tqdm(split_data.items(), desc=f"  {split_name}"):

        class_folder = f"{BASE_DIR}/classification/{split_name}/{class_name}"

        # Save each image
        for img_idx, item in enumerate(items):

            img = item['image']
            annotations = item['annotations']
            bboxes = annotations['bbox']
            categories = annotations['category']

            class_id = SELECTED_CLASSES[class_name]

            # Find bbox for this class
            for bbox, cat_id in zip(bboxes, categories):
                if cat_id == class_id:
                    x, y, w, h = bbox

                    try:
                        # Crop and resize
                        cropped_img = img.crop((x, y, x + w, y + h))
                        cropped_img = cropped_img.resize((224, 224), Image.LANCZOS)

                        # Save
                        img_filename = f"{class_name}_{split_name}_{img_idx:04d}.jpg"
                        img_path = os.path.join(class_folder, img_filename)
                        cropped_img.save(img_path, quality=95)

                        classification_stats[split_name] += 1

                    except Exception as e:
                        print(f"⚠️ Error: {class_name} image {img_idx}: {e}")

                    break

print()
print("="*70)
print("✅ CLASSIFICATION IMAGES SAVED!")
print("="*70)
print(f"📊 Train: {classification_stats['train']} images")
print(f"📊 Val:   {classification_stats['val']} images")
print(f"📊 Test:  {classification_stats['test']} images")
print(f"📊 Total: {sum(classification_stats.values())} images")
print()

In [ ]:
# PART B: SAVE DETECTION IMAGES (YOLO FORMAT)

print("="*70)
print("📁 PART B: Saving Detection Images & Annotations...")
print("   Format: Full images with YOLO .txt labels\n")

detection_stats = {'images': 0, 'annotations': 0, 'objects': 0}

# COCO to YOLO class mapping
coco_to_yolo = {class_id: idx for idx, class_id in enumerate(SELECTED_CLASSES.values())}

# Combine train + val for detection
all_detection_data = []
for class_name in SELECTED_CLASSES.keys():
    all_detection_data.extend(train_data.get(class_name, []))
    all_detection_data.extend(val_data.get(class_name, []))

print(f"📊 Total detection images: {len(all_detection_data)}\n")

# Save images and create YOLO labels
for img_idx, item in enumerate(tqdm(all_detection_data, desc="Saving detection data")):

    img = item['image']
    img_width, img_height = img.size

    # Save full image
    img_filename = f"image_{img_idx:06d}.jpg"
    img_path = os.path.join(f"{BASE_DIR}/detection/images", img_filename)
    img.save(img_path, quality=95)
    detection_stats['images'] += 1

    # Get annotations
    annotations = item['annotations']
    bboxes = annotations['bbox']
    categories = annotations['category']

    # Create YOLO annotation
    label_filename = f"image_{img_idx:06d}.txt"
    label_path = os.path.join(f"{BASE_DIR}/detection/labels", label_filename)

    yolo_annotations = []
    objects_count = 0

    for bbox, cat_id in zip(bboxes, categories):
        if cat_id in coco_to_yolo:
            x, y, w, h = bbox

            # Convert to YOLO format (normalized)
            x_center = (x + w/2) / img_width
            y_center = (y + h/2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height

            yolo_class_id = coco_to_yolo[cat_id]
            yolo_line = f"{yolo_class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
            yolo_annotations.append(yolo_line)
            objects_count += 1

    # Save label file
    if yolo_annotations:
        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_annotations))
        detection_stats['annotations'] += 1
        detection_stats['objects'] += objects_count

print()
print("="*70)
print("✅ DETECTION DATASET CREATED!")
print("="*70)
print(f"📊 Images:     {detection_stats['images']}")
print(f"📊 Labels:     {detection_stats['annotations']}")
print(f"📊 Objects:    {detection_stats['objects']}")
print(f"📊 Avg/image:  {detection_stats['objects']/detection_stats['images']:.2f}")
print()

In [ ]:
# PART C: CREATE YOLO CONFIG FILE

print("📝 Creating YOLO configuration file...\n")

yaml_content = f"""# SmartVision Dataset - YOLOv8 Configuration
path: {os.path.abspath(BASE_DIR)}/detection
train: images
val: images

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: traffic light
  9: stop sign
  10: bench
  11: bird
  12: cat
  13: dog
  14: horse
  15: cow
  16: elephant
  17: bottle
  18: cup
  19: bowl
  20: pizza
  21: cake
  22: chair
  23: couch
  24: potted plant
  25: bed

nc: 26
"""

yaml_path = f"{BASE_DIR}/detection/data.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✅ Created: {yaml_path}\n")

In [ ]:
# PART D: SAVE METADATA

print("📊 Saving metadata...\n")

metadata['classification'] = classification_stats
metadata['detection'] = detection_stats
metadata['dataset_path'] = os.path.abspath(BASE_DIR)

metadata_path = f"{BASE_DIR}/dataset_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, indent=2, fp=f)

print(f"✅ Saved: {metadata_path}\n")

In [ ]:
print("="*70)
print("🎉 DATASET SETUP COMPLETE!")
print("="*70)
print()
print(f"📁 Location: {os.path.abspath(BASE_DIR)}")
print()
print("📂 Classification Dataset:")
print(f"   ├─ Train:  {classification_stats['train']} images (70%)")
print(f"   ├─ Val:    {classification_stats['val']} images (15%)")
print(f"   ├─ Test:   {classification_stats['test']} images (15%)")
print(f"   └─ Total:  {sum(classification_stats.values())} cropped images (224x224)")
print()
print("📂 Detection Dataset:")
print(f"   ├─ Images: {detection_stats['images']} full images")
print(f"   ├─ Labels: {detection_stats['annotations']} YOLO .txt files")
print(f"   └─ Objects: {detection_stats['objects']} annotated objects")
print()
print("="*70)
print("✅ LEARNERS CAN NOW START:")
print("="*70)
print("Step 7:  Exploratory Data Analysis (EDA)")
print("Step 8:  Train Classification Models")
print("Step 9:  Train YOLO Detection Model")
print("Step 10: Build Streamlit Application")
print("Step 11: Deploy to Hugging Face Spaces")
print("="*70)

# Step 1.2: Exploratory Data Analysis (EDA)

In [ ]:
# Import libraries and configure dataset path

# Import required libraries

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw
from collections import Counter

# Dataset location
DATASET_DIR = "/content/smartvision_dataset"

# Classification dataset paths
CLASSIFICATION_DIR = os.path.join(DATASET_DIR, "classification")

TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

# Detection dataset paths
DETECTION_DIR = os.path.join(DATASET_DIR, "detection")

DETECTION_IMAGES_DIR = os.path.join(DETECTION_DIR, "images")
DETECTION_LABELS_DIR = os.path.join(DETECTION_DIR, "labels")

# Check whether dataset exists
if os.path.exists(DATASET_DIR):
    print("Dataset found successfully!")
else:
    raise FileNotFoundError("Dataset folder not found!")

print("Dataset path:", DATASET_DIR)

In [ ]:
# Analyze class distribution

# Get all class names from training folder

classes = sorted(os.listdir(TRAIN_DIR))

# Create an empty list to store class information
class_distribution = []

# Check image count for every class
for class_name in classes:

    # Count training images
    train_count = len(os.listdir(os.path.join(TRAIN_DIR, class_name)))

    # Count validation images
    val_count = len(os.listdir(os.path.join(VAL_DIR, class_name)))

    # Count testing images
    test_count = len(os.listdir(os.path.join(TEST_DIR, class_name)))

    # Calculate total images
    total_count = train_count + val_count + test_count

    # Store results
    class_distribution.append({
        "Class": class_name,
        "Train": train_count,
        "Validation": val_count,
        "Test": test_count,
        "Total": total_count
    })

# Convert results into a DataFrame
class_df = pd.DataFrame(class_distribution)

# Display class distribution
display(class_df)

# Display total images
print("Total Classes:", len(classes))
print("Total Images:", class_df["Total"].sum())

In [ ]:
# Visualize class distribution

# Plot total images for each class

plt.figure(figsize=(14, 6))

plt.bar(class_df["Class"], class_df["Total"])

plt.title("Image Distribution Across Classes")
plt.xlabel("Object Classes")
plt.ylabel("Number of Images")

plt.xticks(rotation=75)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze image characteristics and quality

# Store image information
image_information = []

# Store invalid image paths
invalid_images = []

# Check all training images
for class_name in classes:

    class_folder = os.path.join(TRAIN_DIR, class_name)

    for filename in os.listdir(class_folder):

        image_path = os.path.join(class_folder, filename)

        try:
            # Open image
            with Image.open(image_path) as img:

                # Get image information
                width, height = img.size

                image_information.append({
                    "Class": class_name,
                    "Filename": filename,
                    "Width": width,
                    "Height": height,
                    "Format": img.format,
                    "Color Mode": img.mode
                })

                # Verify that the image can be read
                img.verify()

        except Exception:
            invalid_images.append(image_path)

# Convert information into DataFrame
image_df = pd.DataFrame(image_information)

# Display first five records
display(image_df.head())

# Display image statistics
print("Total Valid Images:", len(image_df))
print("Invalid Images:", len(invalid_images))

print("\nImage Width Statistics:")
print(image_df["Width"].describe())

print("\nImage Height Statistics:")
print(image_df["Height"].describe())

print("\nImage Formats:")
print(image_df["Format"].value_counts())

print("\nColor Modes:")
print(image_df["Color Mode"].value_counts())

In [ ]:
# Visualize image dimensions

# Plot image width and height distribution

plt.figure(figsize=(10, 5))

plt.hist(image_df["Width"], bins=20, alpha=0.7, label="Width")
plt.hist(image_df["Height"], bins=20, alpha=0.7, label="Height")

plt.title("Distribution of Image Dimensions")
plt.xlabel("Pixels")
plt.ylabel("Number of Images")

plt.legend()
plt.show()

In [ ]:
# Analyze objects per image

# Store object counts for every image

objects_per_image = []

# Read all YOLO annotation files
label_files = [
    file for file in os.listdir(DETECTION_LABELS_DIR)
    if file.endswith(".txt")
]

for filename in label_files:

    label_path = os.path.join(DETECTION_LABELS_DIR, filename)

    # Read annotation file
    with open(label_path, "r") as file:
        annotations = file.readlines()

    # Count objects in this image
    object_count = len(annotations)

    objects_per_image.append(object_count)

# Convert to NumPy array
objects_per_image = np.array(objects_per_image)

# Display statistics
print("Total Annotated Images:", len(objects_per_image))

print("Average Objects per Image:",
      round(objects_per_image.mean(), 2))

print("Minimum Objects:", objects_per_image.min())

print("Maximum Objects:", objects_per_image.max())

print("Total Annotated Objects:", objects_per_image.sum())

In [ ]:
# Visualize objects per image

# Plot object distribution

plt.figure(figsize=(10, 5))

plt.hist(objects_per_image, bins=20, edgecolor="black")

plt.title("Objects Per Image Distribution")
plt.xlabel("Number of Objects")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
# # Plot object distribution

plt.figure(figsize=(10, 5))

plt.hist(objects_per_image, bins=20, edgecolor="black")

plt.title("Objects Per Image Distribution")
plt.xlabel("Number of Objects")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
# Visualize sample images with bounding boxes

# Class names in the same order as the original GUVI notebook

class_names = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "airplane",
    "bus",
    "train",
    "truck",
    "traffic light",
    "stop sign",
    "bench",
    "bird",
    "cat",
    "dog",
    "horse",
    "cow",
    "elephant",
    "bottle",
    "cup",
    "bowl",
    "pizza",
    "cake",
    "chair",
    "couch",
    "potted plant",
    "bed"
]

# Find images that have annotation files
sample_labels = [
    file for file in os.listdir(DETECTION_LABELS_DIR)
    if file.endswith(".txt")
]

# Select the first three samples
sample_labels = sample_labels[:3]

# Display sample images
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, label_file in zip(axes, sample_labels):

    # Get corresponding image filename
    image_file = label_file.replace(".txt", ".jpg")

    image_path = os.path.join(DETECTION_IMAGES_DIR, image_file)
    label_path = os.path.join(DETECTION_LABELS_DIR, label_file)

    # Open image
    img = Image.open(image_path).convert("RGB")

    # Get image dimensions
    width, height = img.size

    # Prepare drawing object
    draw = ImageDraw.Draw(img)

    # Read YOLO annotations
    with open(label_path, "r") as file:
        annotations = file.readlines()

    # Draw each bounding box
    for annotation in annotations:

        values = annotation.strip().split()

        # Read class ID and normalized coordinates
        class_id = int(values[0])

        x_center = float(values[1])
        y_center = float(values[2])
        box_width = float(values[3])
        box_height = float(values[4])

        # Convert normalized coordinates into pixel coordinates
        x1 = int((x_center - box_width / 2) * width)
        y1 = int((y_center - box_height / 2) * height)

        x2 = int((x_center + box_width / 2) * width)
        y2 = int((y_center + box_height / 2) * height)

        # Get object name
        class_name = class_names[class_id]

        # Draw bounding box
        draw.rectangle((x1, y1, x2, y2), outline="red", width=3)

        # Display class name
        draw.text((x1, y1 - 15), class_name, fill="yellow")

    # Display image
    ax.imshow(img)
    ax.set_title(image_file)
    ax.axis("off")

plt.tight_layout()
plt.show()

# Step 1.3: Data Preprocessing for Classification

In [ ]:
# Import required libraries

import os
import shutil
import random
import json

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm


# Set random seed for reproducibility
random.seed(42)


# Define the 25 classes mentioned in the GUVI PDF
# The class IDs are taken from the original GUVI notebook

SELECTED_CLASSES = {
    "person": 0,
    "bicycle": 1,
    "car": 2,
    "motorcycle": 3,
    "airplane": 4,
    "bus": 5,
    "truck": 7,
    "traffic light": 9,
    "stop sign": 11,
    "bench": 13,
    "bird": 14,
    "cat": 15,
    "dog": 16,
    "horse": 17,
    "cow": 19,
    "elephant": 20,
    "bottle": 39,
    "cup": 41,
    "bowl": 45,
    "pizza": 53,
    "cake": 55,
    "chair": 56,
    "couch": 57,
    "potted plant": 58,
    "bed": 59
}


# Original dataset location
SOURCE_DIR = "/content/smartvision_dataset"


# New corrected dataset location
OUTPUT_DIR = "/content/smartvision_dataset_clean"


# Classification dataset location
CLASSIFICATION_DIR = os.path.join(
    OUTPUT_DIR, "classification"
)


# Check whether original dataset is available
assert os.path.exists(SOURCE_DIR), "Original dataset not found!"

# Check whether original collected data is available
assert "class_images" in globals(), (
    "class_images not found. Run the original GUVI notebook first."
)


# Display configuration
print("Number of selected classes:", len(SELECTED_CLASSES))
print("Original dataset:", SOURCE_DIR)
print("New dataset:", OUTPUT_DIR)

print("\nSelected classes:")

for class_name in SELECTED_CLASSES:
    print("-", class_name)

In [ ]:
# Remove duplicate images and split the dataset

# Create a dictionary to store unique original images

unique_images = {}


# Collect images from all 25 selected classes
for class_name in SELECTED_CLASSES:

    # Get images belonging to this class
    images = class_images.get(class_name, [])

    for item in images:

        # Every original image has a unique index
        image_id = item["idx"]

        # Store each original image only once
        if image_id not in unique_images:

            unique_images[image_id] = item


# Convert dictionary into a list
all_images = list(unique_images.values())


# Shuffle images randomly
random.shuffle(all_images)


# Calculate split sizes
total_images = len(all_images)

train_size = int(total_images * 0.70)

val_size = int(total_images * 0.15)


# Split original images
train_images = all_images[:train_size]

val_images = all_images[train_size:train_size + val_size]

test_images = all_images[train_size + val_size:]


# Display split information
print("UNIQUE ORIGINAL IMAGES")
print("----------------------")

print("Total unique images:", total_images)

print("Training images:", len(train_images))

print("Validation images:", len(val_images))

print("Testing images:", len(test_images))

print("\nSplit completed successfully!")

In [ ]:
# Extract, resize and save classification images
# Only extract objects belonging to their selected class

from PIL import Image, ImageOps

# Remove previously generated classification dataset
if os.path.exists(CLASSIFICATION_DIR):
    shutil.rmtree(CLASSIFICATION_DIR)


# Create folders for train, validation and test

for split_name in ["train", "val", "test"]:

    for class_name in SELECTED_CLASSES:

        class_folder = os.path.join(
            CLASSIFICATION_DIR,
            split_name,
            class_name
        )

        os.makedirs(class_folder, exist_ok=True)


# Store image counts

image_counts = {
    "train": {},
    "val": {},
    "test": {}
}


# Create mapping using the latest stratified splits

image_split = {}

for item in train_images:
    image_split[item["idx"]] = "train"

for item in val_images:
    image_split[item["idx"]] = "val"

for item in test_images:
    image_split[item["idx"]] = "test"


print("Split mapping created successfully!")
print("Image ID 5 belongs to:", image_split.get(5))


# Process images class by class

for class_name, class_id in SELECTED_CLASSES.items():

    print(f"\nProcessing class: {class_name}")

    images = class_images.get(class_name, [])

    for item in tqdm(images):

        image_id = item["idx"]

        # Find the correct split
        split_name = image_split[image_id]

        # Convert original image to RGB
        original_image = item["image"].convert("RGB")

        image_width, image_height = original_image.size

        # Get annotations
        annotations = item["annotations"]

        bounding_boxes = annotations["bbox"]
        categories = annotations["category"]

        object_number = 0

        # Examine bounding boxes
        for bbox, category_id in zip(
            bounding_boxes, categories
        ):

            # Extract only the selected class
            if category_id != class_id:
                continue

            x, y, width, height = bbox

            # Keep coordinates inside image boundaries
            x1 = max(0, int(x))
            y1 = max(0, int(y))

            x2 = min(image_width, int(x + width))
            y2 = min(image_height, int(y + height))

            # Skip invalid bounding boxes
            if x2 <= x1 or y2 <= y1:
                continue

            # Crop individual object
            cropped_image = original_image.crop(
                (x1, y1, x2, y2)
            )

            # Resize while preserving aspect ratio
            cropped_image = ImageOps.pad(
                cropped_image,
                (224, 224),
                method=Image.Resampling.LANCZOS,
                color=(0, 0, 0)
            )

            # Save in the correct class folder
            class_folder = os.path.join(
                CLASSIFICATION_DIR,
                split_name,
                class_name
            )

            filename = f"{image_id}_{object_number:03d}.jpg"

            save_path = os.path.join(
                class_folder,
                filename
            )

            cropped_image.save(
                save_path,
                format="JPEG",
                quality=95
            )

            # Update image count
            image_counts[split_name][class_name] = (
                image_counts[split_name].get(class_name, 0) + 1
            )

            object_number += 1


print("\nClassification preprocessing completed!")

In [ ]:
# Create a list to store final dataset statistics

final_statistics = []


# Check all classes and splits
for class_name in SELECTED_CLASSES:

    train_count = len(
        os.listdir(
            os.path.join(CLASSIFICATION_DIR, "train", class_name)
        )
    )

    val_count = len(
        os.listdir(
            os.path.join(CLASSIFICATION_DIR, "val", class_name)
        )
    )

    test_count = len(
        os.listdir(
            os.path.join(CLASSIFICATION_DIR, "test", class_name)
        )
    )

    total_count = train_count + val_count + test_count

    final_statistics.append({
        "Class": class_name,
        "Train": train_count,
        "Validation": val_count,
        "Test": test_count,
        "Total": total_count
    })


# Convert statistics into DataFrame
final_df = pd.DataFrame(final_statistics)


# Display results
display(final_df)


# Display total counts
print("Total classes:", len(final_df))

print("Total training crops:", final_df["Train"].sum())

print("Total validation crops:", final_df["Validation"].sum())

print("Total testing crops:", final_df["Test"].sum())

print("Total classification crops:", final_df["Total"].sum())


# Verify that all 25 classes exist
assert len(final_df) == 25, "Incorrect number of classes!"



# Verify no original image exists in multiple splits

# Extract original image IDs from each split
train_ids = {item["idx"] for item in train_images}

val_ids = {item["idx"] for item in val_images}

test_ids = {item["idx"] for item in test_images}


# Check for overlapping images
assert train_ids.isdisjoint(val_ids), "Train-validation leakage!"

assert train_ids.isdisjoint(test_ids), "Train-test leakage!"

assert val_ids.isdisjoint(test_ids), "Validation-test leakage!"


# Display verification results
print("\nDATA LEAKAGE CHECK")
print("------------------")

print("Training and validation overlap:", len(train_ids & val_ids))

print("Training and testing overlap:", len(train_ids & test_ids))

print("Validation and testing overlap:", len(val_ids & test_ids))

print("\nAll leakage checks passed!")

In [ ]:
# normalization

# Load one sample image

sample_image_path = os.path.join(
    CLASSIFICATION_DIR,
    "train",
    "person",
    os.listdir(
        os.path.join(CLASSIFICATION_DIR, "train", "person")
    )[0]
)

sample_image = Image.open(sample_image_path).convert("RGB")

# Convert image into NumPy array
image_array = np.array(sample_image)

# Normalize pixel values between 0 and 1
normalized_image = image_array.astype("float32") / 255.0

print("Original pixel range:",
      image_array.min(), image_array.max())

print("Normalized pixel range:",
      normalized_image.min(), normalized_image.max())

# We used a 70:15:15 split on unique original images.

However, different original images contain different numbers of objects from different categories.

Consequently, the original-image split is approximately 70:15:15, but the resulting classification crops are not evenly distributed.

improving the splitting strategy using Stratified Group Splitting.

In [ ]:
# Install multilabel stratification library

!pip install -q iterative-stratification

In [ ]:
# Prepare unique images and class labels

import numpy as np

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Store every original image only once
unique_images = {}

for class_name in SELECTED_CLASSES:

    for item in class_images.get(class_name, []):

        image_id = item["idx"]

        if image_id not in unique_images:
            unique_images[image_id] = item

# Convert dictionary into a list
all_images = list(unique_images.values())

# Create a binary label matrix
# Each row represents one original image
# Each column represents one of our 25 classes

class_ids = list(SELECTED_CLASSES.values())

labels = []

for item in all_images:

    categories = item["annotations"]["category"]

    image_labels = [
        1 if class_id in categories else 0
        for class_id in class_ids
    ]

    labels.append(image_labels)

# Convert labels into NumPy array
Y = np.array(labels)

print("Unique original images:", len(all_images))
print("Number of classes:", len(class_ids))
print("Label matrix shape:", Y.shape)

In [ ]:
!pip install iterative-stratification

In [ ]:
# Multilabel Stratified Train / Validation / Test Split

import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Step 1: Collect unique original images

unique_images = {}

for class_name, images in class_images.items():

    if class_name not in SELECTED_CLASSES:
        continue

    for item in images:
        image_id = item["idx"]

        if image_id not in unique_images:
            unique_images[image_id] = item

print("Total unique original images:", len(unique_images))


# Step 2: Create multilabel matrix

image_list = list(unique_images.values())

class_names = list(SELECTED_CLASSES.keys())
class_ids = list(SELECTED_CLASSES.values())

Y = np.zeros(
    (len(image_list), len(class_names)),
    dtype=int
)

for i, item in enumerate(image_list):

    categories = item["annotations"]["category"]

    for j, class_id in enumerate(class_ids):

        if class_id in categories:
            Y[i, j] = 1


# Step 3: Split 70% training and 30% temporary data

splitter_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    splitter_1.split(np.zeros(len(Y)), Y)
)


# Step 4: Split temporary data equally into validation and test

Y_temp = Y[temp_idx]

splitter_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_relative_idx, test_relative_idx = next(
    splitter_2.split(np.zeros(len(Y_temp)), Y_temp)
)

val_idx = temp_idx[val_relative_idx]
test_idx = temp_idx[test_relative_idx]


# Step 5: Create final datasets

train_images = [image_list[i] for i in train_idx]

val_images = [image_list[i] for i in val_idx]

test_images = [image_list[i] for i in test_idx]


# Display split information

print("\nSTRATIFIED DATASET SPLIT")
print("------------------------")

print("Training images:", len(train_images))
print("Validation images:", len(val_images))
print("Testing images:", len(test_images))

print("Total images:", len(image_list))

In [ ]:
# Verify no original image appears in multiple splits

train_ids = {
    item["idx"] for item in train_images
}

val_ids = {
    item["idx"] for item in val_images
}

test_ids = {
    item["idx"] for item in test_images
}


# Check overlapping images

train_val_overlap = train_ids & val_ids

train_test_overlap = train_ids & test_ids

val_test_overlap = val_ids & test_ids


# Display results

print("DATA LEAKAGE VERIFICATION")
print("-------------------------")

print("Train-Val overlap:", len(train_val_overlap))

print("Train-Test overlap:", len(train_test_overlap))

print("Val-Test overlap:", len(val_test_overlap))


# Ensure no leakage exists

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print("\nAll leakage checks passed!")

In [ ]:
# Verify class distribution before cropping

# Calculate class distribution based on original images

split_data = {
    "Train": train_images,
    "Validation": val_images,
    "Test": test_images
}

distribution = []

for class_name, class_id in SELECTED_CLASSES.items():

    row = {"Class": class_name}

    for split_name, images in split_data.items():

        count = sum(
            class_id in item["annotations"]["category"]
            for item in images
        )

        row[split_name] = count

    distribution.append(row)

distribution_df = pd.DataFrame(distribution)

display(distribution_df)

print("\nTotal images per split:")
print("Train:", len(train_images))
print("Validation:", len(val_images))
print("Test:", len(test_images))

In [ ]:
# Verify data leakage

# Get original image IDs from each split

train_ids = {item["idx"] for item in train_images}

val_ids = {item["idx"] for item in val_images}

test_ids = {item["idx"] for item in test_images}

# Check overlapping images

print("Train vs Validation:", len(train_ids & val_ids))

print("Train vs Test:", len(train_ids & test_ids))

print("Validation vs Test:", len(val_ids & test_ids))

# Verify all splits are independent

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("\nAll data leakage checks passed!")

In [ ]:
# Generate classification dataset

# Step 1.3: Generate final classification dataset

# PHASE 1 - STEP 1.3
# Final Classification Dataset Generation

from PIL import Image, ImageOps
import os
import shutil
from tqdm import tqdm

# 1. Remove previously generated classification dataset

if os.path.exists(CLASSIFICATION_DIR):
    shutil.rmtree(CLASSIFICATION_DIR)


# 2. Create folders for all splits and classes

for split_name in ["train", "val", "test"]:

    for class_name in SELECTED_CLASSES:

        folder_path = os.path.join(
            CLASSIFICATION_DIR,
            split_name,
            class_name
        )

        os.makedirs(folder_path, exist_ok=True)


# 3. Initialize crop counts

crop_counts = {
    split: {
        class_name: 0
        for class_name in SELECTED_CLASSES
    }
    for split in ["train", "val", "test"]
}


# 4. Create category ID to class name mapping

category_to_class = {
    class_id: class_name
    for class_name, class_id in SELECTED_CLASSES.items()
}


# 5. Use the latest multilabel-stratified splits

split_data = {
    "train": train_images,
    "val": val_images,
    "test": test_images
}


# 6. Generate classification crops

for split_name, images in split_data.items():

    print(f"\nProcessing {split_name.upper()} images...")

    for item in tqdm(images):

        # Load original image
        original_image = item["image"].convert("RGB")

        image_width, image_height = original_image.size

        # Original image ID
        image_id = item["idx"]

        # Read annotations
        annotations = item["annotations"]

        bounding_boxes = annotations["bbox"]
        categories = annotations["category"]

        # Process each annotated object
        for object_number, (bbox, category_id) in enumerate(
            zip(bounding_boxes, categories)
        ):

            # Identify selected class
            class_name = category_to_class.get(category_id)

            # Ignore unselected classes
            if class_name is None:
                continue

            # Read bounding box
            x, y, width, height = bbox

            # Keep coordinates within image boundaries
            x1 = max(0, int(x))
            y1 = max(0, int(y))

            x2 = min(image_width, int(x + width))
            y2 = min(image_height, int(y + height))

            # Skip invalid bounding boxes
            if x2 <= x1 or y2 <= y1:
                continue

            # Crop individual object
            cropped_image = original_image.crop(
                (x1, y1, x2, y2)
            )

            # Resize while preserving original aspect ratio
            # Add padding to create a square 224 x 224 image

            cropped_image = ImageOps.pad(
                cropped_image,
                (224, 224),
                method=Image.Resampling.LANCZOS,
                color=(0, 0, 0)
            )

            # Destination class folder
            class_folder = os.path.join(
                CLASSIFICATION_DIR,
                split_name,
                class_name
            )

            # Unique filename
            filename = f"{image_id}_{object_number:03d}.jpg"

            save_path = os.path.join(
                class_folder,
                filename
            )

            # Save classification crop
            cropped_image.save(
                save_path,
                format="JPEG",
                quality=95
            )

            # Update crop count
            crop_counts[split_name][class_name] += 1


print("\nClassification dataset generation completed!")

In [ ]:
# Verify final classification dataset

# Create final dataset summary

final_results = []

for class_name in SELECTED_CLASSES:

    train_count = crop_counts["train"][class_name]
    val_count = crop_counts["val"][class_name]
    test_count = crop_counts["test"][class_name]

    final_results.append({
        "Class": class_name,
        "Train": train_count,
        "Validation": val_count,
        "Test": test_count,
        "Total": train_count + val_count + test_count
    })


# Convert results into DataFrame
final_df = pd.DataFrame(final_results)

display(final_df)

# Display overall statistics
print("Total Classes:", len(final_df))
print("Training Crops:", final_df["Train"].sum())
print("Validation Crops:", final_df["Validation"].sum())
print("Testing Crops:", final_df["Test"].sum())

print("Total Classification Crops:", final_df["Total"].sum())

# Important observation: Class imbalance on above

In [ ]:
# Final leakage verification

# Get original image IDs from each split

train_ids = {item["idx"] for item in train_images}
val_ids = {item["idx"] for item in val_images}
test_ids = {item["idx"] for item in test_images}


# Check overlap between splits

print("Train vs Validation:", len(train_ids & val_ids))

print("Train vs Test:", len(train_ids & test_ids))

print("Validation vs Test:", len(val_ids & test_ids))


# Verify no leakage exists

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("\nAll data leakage checks passed!")

In [ ]:
# Verify that original images do not overlap between splits

train_ids = {item["idx"] for item in train_images}
val_ids = {item["idx"] for item in val_images}
test_ids = {item["idx"] for item in test_images}

print("Train-Val overlap:", len(train_ids & val_ids))
print("Train-Test overlap:", len(train_ids & test_ids))
print("Val-Test overlap:", len(val_ids & test_ids))

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("\nDataset leakage check passed!")

# Step 1.4: Data Augmentation

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

In [ ]:
# Step 1.4 - Data Augmentation

import tensorflow as tf
from tensorflow.keras import layers


# Register custom color jittering function
@tf.keras.utils.register_keras_serializable()
def color_jitter(images):

    # Convert pixel values from 0-255 to 0-1
    images = images / 255.0

    # Randomly adjust hue
    images = tf.image.random_hue(
        images,
        max_delta=0.05
    )

    # Randomly adjust saturation
    images = tf.image.random_saturation(
        images,
        lower=0.8,
        upper=1.2
    )

    # Keep pixel values within valid range
    images = tf.clip_by_value(
        images,
        0.0,
        1.0
    )

    # Convert back to 0-255
    images = images * 255.0

    return images


# Complete augmentation pipeline

data_augmentation = tf.keras.Sequential([

    # Horizontal flipping
    layers.RandomFlip("horizontal"),

    # Rotation approximately +/- 15 degrees
    layers.RandomRotation(
        0.0417,
        fill_mode="nearest"
    ),

    # Random zoom
    layers.RandomZoom(
        0.1,
        fill_mode="nearest"
    ),

    # Brightness adjustment +/- 20%
    layers.RandomBrightness(
        factor=0.2,
        value_range=(0, 255)
    ),

    # Contrast adjustment
    layers.RandomContrast(0.2),

    # Hue and saturation adjustment
    layers.Lambda(color_jitter)

], name="data_augmentation")


print("Data augmentation pipeline created successfully!")

In [ ]:
# Visualize augmented images

import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

# Select one training image
sample_class = "person"

sample_folder = os.path.join(
    CLASSIFICATION_DIR,
    "train",
    sample_class
)

sample_image_name = os.listdir(sample_folder)[0]
sample_image_path = os.path.join(sample_folder, sample_image_name)

# Load image
image = load_img(sample_image_path, target_size=(224, 224))
image_array = img_to_array(image)

# Generate augmented versions
plt.figure(figsize=(12, 6))

for i in range(6):

    augmented_image = data_augmentation(
        tf.expand_dims(image_array, 0),
        training=True
    )

    plt.subplot(2, 3, i + 1)
    plt.imshow(
        tf.clip_by_value(augmented_image[0], 0, 255).numpy().astype("uint8")
    )
    plt.axis("off")
    plt.title(f"Augmented Image {i + 1}")

plt.tight_layout()
plt.show()

# augmented images look extremely distorted. That is not what we want. The object should remain recognisable after augmentation.

In [ ]:
# Check the original image

import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img

# Load the original, unaugmented image
original_image = load_img(
    sample_image_path,
    target_size=(224, 224)
)

plt.figure(figsize=(5, 5))
plt.imshow(original_image)
plt.title("Original Training Image")
plt.axis("off")
plt.show()

In [ ]:
# Inspect the original image and bounding box

from PIL import Image
import matplotlib.pyplot as plt

# Get the first image collected for the person class
item = class_images["person"][0]

# Original image
original_image = item["image"].convert("RGB")

# Image information
print("Original Image ID:", item["idx"])
print("Original Image Size:", original_image.size)

# Display annotations
annotations = item["annotations"]

print("\nBounding Boxes:")
print(annotations["bbox"][:5])

print("\nCategory IDs:")
print(annotations["category"][:5])

# Display original image
plt.figure(figsize=(8, 6))
plt.imshow(original_image)
plt.title("Original COCO Image")
plt.axis("off")
plt.show()

In [ ]:
# Draw the bounding box on the original image

from PIL import ImageDraw
import matplotlib.pyplot as plt

# Get the original image
item = class_images["person"][0]
original_image = item["image"].convert("RGB")

# Create a copy for visualization
display_image = original_image.copy()
draw = ImageDraw.Draw(display_image)

# Get annotations
annotations = item["annotations"]

# Person category ID
person_id = SELECTED_CLASSES["person"]

# Draw person bounding boxes
for bbox, category_id in zip(
    annotations["bbox"],
    annotations["category"]
):

    if category_id != person_id:
        continue

    x, y, width, height = bbox

    x1 = int(x)
    y1 = int(y)
    x2 = int(x + width)
    y2 = int(y + height)

    draw.rectangle(
        (x1, y1, x2, y2),
        outline="red",
        width=4
    )

    print("Bounding box:", (x1, y1, x2, y2))

# Display image
plt.figure(figsize=(8, 6))
plt.imshow(display_image)
plt.title("Person Bounding Box")
plt.axis("off")
plt.show()

In [ ]:
# Display the exact crop from this same image

from PIL import Image, ImageOps
import matplotlib.pyplot as plt

# Get the same original image
item = class_images["person"][0]
original_image = item["image"].convert("RGB")

# Get annotations
annotations = item["annotations"]

# Find the person bounding box
person_id = SELECTED_CLASSES["person"]

for bbox, category_id in zip(
    annotations["bbox"],
    annotations["category"]
):
    if category_id == person_id:

        x, y, width, height = bbox

        # Crop the person
        crop = original_image.crop((
            int(x),
            int(y),
            int(x + width),
            int(y + height)
        ))

        print("Original crop size:", crop.size)

        # Resize without stretching
        crop = ImageOps.pad(
            crop,
            (224, 224),
            method=Image.Resampling.LANCZOS,
            color=(0, 0, 0)
        )

        print("Final crop size:", crop.size)

        # Display crop
        plt.figure(figsize=(5, 5))
        plt.imshow(crop)
        plt.title("Person Crop - Direct from COCO Image")
        plt.axis("off")
        plt.show()

        break

direct crop is correct. The person is clearly visible, and the original aspect ratio is preserved. The black areas on the left and right are padding added to make the image square (224 × 224). This is expected and is not distortion.

In [ ]:
# STEP 1: MOUNT GOOGLE DRIVE

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

print("Google Drive mounted successfully!")


In [ ]:
# STEP 1: CREATE PROJECT FOLDERS

import os

PROJECT_DIR = "/content/drive/MyDrive/SmartVision_AI"

MODEL_DIR = os.path.join(PROJECT_DIR, "saved_models")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
HISTORY_DIR = os.path.join(RESULT_DIR, "training_history")
GRAPH_DIR = os.path.join(RESULT_DIR, "graphs")
REPORT_DIR = os.path.join(RESULT_DIR, "evaluation_reports")
CONFUSION_DIR = os.path.join(RESULT_DIR, "confusion_matrices")

# Create all directories
folders = [
    MODEL_DIR,
    RESULT_DIR,
    HISTORY_DIR,
    GRAPH_DIR,
    REPORT_DIR,
    CONFUSION_DIR
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("All project folders created successfully!")
print("Project location:", PROJECT_DIR)

In [ ]:
# CREATE DATASET FOLDERS

import os

DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")

CLASSIFICATION_DIR = os.path.join(DATASET_DIR, "classification")
DETECTION_DIR = os.path.join(DATASET_DIR, "detection")

os.makedirs(CLASSIFICATION_DIR, exist_ok=True)
os.makedirs(DETECTION_DIR, exist_ok=True)

print("Dataset folders created successfully!")
print("Dataset directory:", DATASET_DIR)
print("Classification directory:", CLASSIFICATION_DIR)
print("Detection directory:", DETECTION_DIR)

# Phase 2: Transfer Learning - Image Classification

## VGG16

In [ ]:
# PHASE 2 - VGG16
# Step 1: Load Classification Dataset

import tensorflow as tf
import os

# Dataset directory
TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

# Image settings
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# Load training dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

# Load validation dataset
val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

# Load testing dataset
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

# Get class names
class_names = train_dataset.class_names

# Display dataset information
print("Number of classes:", len(class_names))
print("Classes:", class_names)

print("\nTraining batches:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_dataset).numpy())
print("Testing batches:", tf.data.experimental.cardinality(test_dataset).numpy())

In [ ]:
# STEP 2: BUILD VGG16 MODEL

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

# Load pretrained VGG16 model
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze all pretrained layers
base_model.trainable = False

# Build the classification model
vgg16_model = models.Sequential([

    # Input layer
    layers.Input(shape=(224, 224, 3)),

    # VGG16 preprocessing
    layers.Lambda(
        tf.keras.applications.vgg16.preprocess_input
    ),

    # Pretrained VGG16 feature extractor
    base_model,

    # Classification layers
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),

    layers.Dropout(0.5),

    # Output layer for 25 classes
    layers.Dense(25, activation="softmax")
])

# Display model architecture
vgg16_model.summary()

In [ ]:
# STEP 3: COMPILE VGG16 MODEL

vgg16_model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]
)

print("VGG16 model compiled successfully!")

In [ ]:
# STEP 4: TRAIN VGG16 MODEL

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# Create directory to save models
os.makedirs("saved_models", exist_ok=True)

# Save the best model
checkpoint = ModelCheckpoint(
    "saved_models/vgg16_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

# Stop training if validation loss does not improve
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate when validation loss stops improving
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

# Train the model
history_vgg16 = vgg16_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)

In [ ]:
# SAVE TRAINED VGG16 MODEL TO GOOGLE DRIVE

vgg16_model.save(
    os.path.join(MODEL_DIR, "vgg16_best.keras")
)

print("VGG16 model saved successfully!")

In [ ]:
# SAVE TRAINING HISTORY

import json

with open(
    os.path.join(HISTORY_DIR, "vgg16_history.json"), "w"
) as file:
    json.dump(history_vgg16.history, file, indent=4)

print("VGG16 training history saved!")

In [ ]:
# VGG16 TEST EVALUATION

test_loss, test_accuracy = vgg16_model.evaluate(test_dataset)

print("========== VGG16 PERFORMANCE ==========")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")

# Compare with GUVI expected performance
if 80 <= test_accuracy * 100 <= 85:
    print("Status: Within expected accuracy range")
elif test_accuracy * 100 < 80:
    print("Status: Below expected accuracy range")
else:
    print("Status: Above expected accuracy range")

In [ ]:
# Evaluate VGG16
# VGG16 MODEL EVALUATION

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    top_k_accuracy_score
)

# Get actual labels from test dataset
y_true = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in test_dataset
])

# Get model predictions
y_prob = vgg16_model.predict(test_dataset)

# Convert probabilities into predicted class labels
y_pred = np.argmax(y_prob, axis=1)

# Calculate evaluation metrics
accuracy = accuracy_score(y_true, y_pred)

precision_macro = precision_score(
    y_true, y_pred, average="macro", zero_division=0
)

precision_weighted = precision_score(
    y_true, y_pred, average="weighted", zero_division=0
)

recall_macro = recall_score(
    y_true, y_pred, average="macro", zero_division=0
)

recall_weighted = recall_score(
    y_true, y_pred, average="weighted", zero_division=0
)

f1_macro = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

f1_weighted = f1_score(
    y_true, y_pred, average="weighted", zero_division=0
)

top5_accuracy = top_k_accuracy_score(
    y_true,
    y_prob,
    k=5,
    labels=np.arange(len(class_names))
)

# Print overall results
print("=" * 55)
print("             VGG16 MODEL PERFORMANCE")
print("=" * 55)

print(f"Test Accuracy       : {accuracy * 100:.2f}%")
print(f"Precision (Macro)   : {precision_macro * 100:.2f}%")
print(f"Precision (Weighted): {precision_weighted * 100:.2f}%")
print(f"Recall (Macro)      : {recall_macro * 100:.2f}%")
print(f"Recall (Weighted)   : {recall_weighted * 100:.2f}%")
print(f"F1-Score (Macro)    : {f1_macro * 100:.2f}%")
print(f"F1-Score (Weighted) : {f1_weighted * 100:.2f}%")
print(f"Top-5 Accuracy      : {top5_accuracy * 100:.2f}%")

print("=" * 55)

In [ ]:
# Classification report for 25 classes

# CLASSIFICATION REPORT

report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0
)

print("VGG16 CLASSIFICATION REPORT")
print("=" * 75)
print(report)


In [ ]:
# CONFUSION MATRIX

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(class_names))
)

plt.figure(figsize=(18, 16))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    cmap="Blues",
    xticks_rotation=90,
    values_format="d",
    ax=plt.gca(),
    colorbar=False
)

plt.title("VGG16 Confusion Matrix")
plt.tight_layout()
plt.show()

VGG16 evaluation is complete, but the model has not achieved the expected performance in the GUVI project

Possible reasons for low accuracy :
  - A. Class imbalance
  - B. Data augmentation may not be applied
  - C. Training may be insufficient or overfitting

In [ ]:
# Check VGG16 training history

# VGG16 TRAINING HISTORY ANALYSIS

train_acc = history_vgg16.history["accuracy"]
val_acc = history_vgg16.history["val_accuracy"]

train_loss = history_vgg16.history["loss"]
val_loss = history_vgg16.history["val_loss"]

print("========== VGG16 TRAINING SUMMARY ==========")

print(f"Total epochs trained: {len(train_acc)}")

print(f"Final Training Accuracy: {train_acc[-1] * 100:.2f}%")
print(f"Final Validation Accuracy: {val_acc[-1] * 100:.2f}%")

print(f"Best Training Accuracy: {max(train_acc) * 100:.2f}%")
print(f"Best Validation Accuracy: {max(val_acc) * 100:.2f}%")

print(f"Final Training Loss: {train_loss[-1]:.4f}")
print(f"Final Validation Loss: {val_loss[-1]:.4f}")

In [ ]:
# display the accuracy and loss curves

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_acc, label="Training Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.title("VGG16 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.title("VGG16 Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Integrate augmentation into VGG16

# REBUILD VGG16 WITH DATA AUGMENTATION

import tensorflow as tf
from tensorflow.keras import layers, models

# Load pretrained VGG16
base_model = tf.keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained convolutional layers
base_model.trainable = False

# Define model input
inputs = layers.Input(shape=(224, 224, 3))

# Apply data augmentation
x = data_augmentation(inputs)

# Apply VGG16 preprocessing
x = layers.Lambda(
    tf.keras.applications.vgg16.preprocess_input
)(x)

# Extract image features
x = base_model(x)

# Classification head
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, activation="relu")(x)

x = layers.Dropout(0.5)(x)

# Output layer for 25 classes
outputs = layers.Dense(
    25,
    activation="softmax"
)(x)

# Create final model
vgg16_model = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="VGG16_Augmented"
)

# Display architecture
vgg16_model.summary()

In [ ]:
# Compile the updated model

# COMPILE UPDATED VGG16

vgg16_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Augmented VGG16 compiled successfully!")

In [ ]:
# Retrain VGG16

# TRAIN VGG16 WITH DATA AUGMENTATION

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

vgg16_callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    ModelCheckpoint(
        "/content/drive/MyDrive/SmartVision_AI/saved_models/vgg16_augmented_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
]

history_vgg16_aug = vgg16_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=25,
    callbacks=vgg16_callbacks
)

In [ ]:
# Register both functions

# REGISTER CUSTOM FUNCTIONS FOR MODEL LOADING

import tensorflow as tf

# Register color jitter function
@tf.keras.utils.register_keras_serializable()
def color_jitter(images):

    images = images / 255.0

    images = tf.image.random_hue(
        images,
        max_delta=0.05
    )

    images = tf.image.random_saturation(
        images,
        lower=0.8,
        upper=1.2
    )

    images = tf.clip_by_value(images, 0.0, 1.0)

    return images * 255.0


# Register VGG16 preprocessing function
@tf.keras.utils.register_keras_serializable()
def vgg16_preprocess(images):

    return tf.keras.applications.vgg16.preprocess_input(images)


print("Both custom functions registered successfully!")

In [ ]:
# LOAD BEST AUGMENTED VGG16 MODEL

MODEL_PATH = "/content/drive/MyDrive/SmartVision_AI/saved_models/vgg16_augmented_best.keras"

best_vgg16_aug = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "color_jitter": color_jitter,
        "preprocess_input": vgg16_preprocess
    }
)

print("Augmented VGG16 model loaded successfully!")

In [ ]:
best_vgg16_aug.summary()

print("\nOutput classes:", best_vgg16_aug.output_shape[-1])

In [ ]:
# Evaluate augmented VGG16

# EVALUATE BEST AUGMENTED VGG16

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    top_k_accuracy_score
)

# Actual test labels
y_true_aug = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in test_dataset
])

# Predict test images
y_prob_aug = best_vgg16_aug.predict(test_dataset)

# Predicted class labels
y_pred_aug = np.argmax(y_prob_aug, axis=1)

# Calculate metrics
accuracy_aug = accuracy_score(y_true_aug, y_pred_aug)

precision_macro_aug = precision_score(
    y_true_aug, y_pred_aug,
    average="macro", zero_division=0
)

precision_weighted_aug = precision_score(
    y_true_aug, y_pred_aug,
    average="weighted", zero_division=0
)

recall_macro_aug = recall_score(
    y_true_aug, y_pred_aug,
    average="macro", zero_division=0
)

recall_weighted_aug = recall_score(
    y_true_aug, y_pred_aug,
    average="weighted", zero_division=0
)

f1_macro_aug = f1_score(
    y_true_aug, y_pred_aug,
    average="macro", zero_division=0
)

f1_weighted_aug = f1_score(
    y_true_aug, y_pred_aug,
    average="weighted", zero_division=0
)

top5_aug = top_k_accuracy_score(
    y_true_aug,
    y_prob_aug,
    k=5,
    labels=np.arange(len(class_names))
)

# Display results
print("=" * 55)
print("       AUGMENTED VGG16 PERFORMANCE")
print("=" * 55)

print(f"Test Accuracy        : {accuracy_aug * 100:.2f}%")
print(f"Precision (Macro)    : {precision_macro_aug * 100:.2f}%")
print(f"Precision (Weighted) : {precision_weighted_aug * 100:.2f}%")
print(f"Recall (Macro)       : {recall_macro_aug * 100:.2f}%")
print(f"Recall (Weighted)    : {recall_weighted_aug * 100:.2f}%")
print(f"F1-Score (Macro)     : {f1_macro_aug * 100:.2f}%")
print(f"F1-Score (Weighted)  : {f1_weighted_aug * 100:.2f}%")
print(f"Top-5 Accuracy       : {top5_aug * 100:.2f}%")

print("=" * 55)

In [ ]:
# CLASSIFICATION REPORT

print(classification_report(
    y_true_aug,
    y_pred_aug,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0
))

In [ ]:
# CONFUSION MATRIX

cm_aug = confusion_matrix(
    y_true_aug,
    y_pred_aug,
    labels=np.arange(len(class_names))
)

fig, ax = plt.subplots(figsize=(18, 16))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_aug,
    display_labels=class_names
)

disp.plot(
    cmap="Blues",
    xticks_rotation=90,
    values_format="d",
    ax=ax,
    colorbar=False
)

plt.title("Augmented VGG16 Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Check the training class distribution

import numpy as np
import pandas as pd

train_labels = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in train_dataset
])

class_counts = pd.Series(train_labels).value_counts().sort_index()

distribution = pd.DataFrame({
    "Class": class_names,
    "Training Images": [
        class_counts.get(i, 0)
        for i in range(len(class_names))
    ]
})

print(distribution.to_string(index=False))

This confirms that your training dataset is highly imbalanced.

In [ ]:
# Calculate class weights

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Extract training labels
train_labels = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in train_dataset
])

# Calculate balanced class weights
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=train_labels
)

# Convert into dictionary
class_weights = {
    i: float(weight)
    for i, weight in enumerate(class_weights_array)
}

# Display weights
for i, class_name in enumerate(class_names):
    print(f"{class_name:15s}: {class_weights[i]:.3f}")

In [ ]:
# Retrain VGG16 using class weights

# Train augmented VGG16 using class weights

history_vgg16_weighted = vgg16_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=25,
    class_weight=class_weights,
    callbacks=vgg16_callbacks
)

existing checkpoint callback points to vgg16_augmented_best.keras. To avoid overwriting your previous model, first create a separate checkpoint callback for this experiment

In [ ]:
weighted_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/SmartVision_AI/saved_models/vgg16_weighted_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
]

history_vgg16_weighted = vgg16_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=25,
    class_weight=class_weights,
    callbacks=weighted_callbacks
)

The weighted model stopped early because validation performance did not improve.

In [ ]:
# Evaluate the weighted model

# Load the best weighted VGG16 checkpoint

best_vgg16_weighted = tf.keras.models.load_model(
    "/content/drive/MyDrive/SmartVision_AI/saved_models/vgg16_weighted_best.keras",
    custom_objects={
        "color_jitter": color_jitter,
        "preprocess_input": vgg16_preprocess
    }
)

# Predict test dataset
y_prob_weighted = best_vgg16_weighted.predict(test_dataset)

y_pred_weighted = np.argmax(y_prob_weighted, axis=1)

# Actual labels
y_true_weighted = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in test_dataset
])

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("WEIGHTED VGG16 TEST PERFORMANCE")
print("=" * 45)

print(f"Accuracy          : {accuracy_score(y_true_weighted, y_pred_weighted)*100:.2f}%")
print(f"Precision (Macro) : {precision_score(y_true_weighted, y_pred_weighted, average='macro', zero_division=0)*100:.2f}%")
print(f"Recall (Macro)    : {recall_score(y_true_weighted, y_pred_weighted, average='macro', zero_division=0)*100:.2f}%")
print(f"F1 Score (Macro)  : {f1_score(y_true_weighted, y_pred_weighted, average='macro', zero_division=0)*100:.2f}%")

In [ ]:
import os
import json

HISTORY_DIR = "/content/drive/MyDrive/SmartVision_AI/results/training_history"

os.makedirs(HISTORY_DIR, exist_ok=True)

histories = {
    "vgg16_augmented": history_vgg16_aug,
    "vgg16_weighted": history_vgg16_weighted,
    "resnet50_augmented": history_resnet50,
    "resnet50_finetuned": history_resnet50_finetune
}

for name, history in histories.items():

    with open(
        os.path.join(HISTORY_DIR, f"{name}_history.json"),
        "w"
    ) as f:
        json.dump(history.history, f, indent=4)

    print(f"Saved: {name}_history.json")

print("\nAll training histories saved successfully!")

## ResNet50

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

# Load pretrained ResNet50
base_model_resnet = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained layers initially
base_model_resnet.trainable = False

# Build ResNet50 classification model
inputs = layers.Input(shape=(224, 224, 3))

# Apply data augmentation
x = data_augmentation(inputs)

# ResNet50 preprocessing
x = layers.Lambda(
    tf.keras.applications.resnet50.preprocess_input
)(x)

# Feature extraction
x = base_model_resnet(x, training=False)

# Classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)

# Output layer – 25 classes
outputs = layers.Dense(25, activation="softmax")(x)

# Create model
resnet50_model = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="ResNet50_Augmented"
)

# Display model summary
resnet50_model.summary()

In [ ]:
# Compile ResNet50

# Compile ResNet50 model

resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("ResNet50 compiled successfully!")

In [ ]:
# Configure callbacks

# ResNet50 training callbacks

resnet50_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath="/content/drive/MyDrive/SmartVision_AI/saved_models/resnet50_augmented_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
]

print("ResNet50 callbacks configured successfully!")

In [ ]:
# Train the model

# Train ResNet50

history_resnet50 = resnet50_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=25,
    callbacks=resnet50_callbacks
)

ResNet50 fine-tuning

In [ ]:
# Load the best ResNet50 checkpoint

# Load the best ResNet50 model

best_resnet50 = tf.keras.models.load_model(
    "/content/drive/MyDrive/SmartVision_AI/saved_models/resnet50_augmented_best.keras",
    custom_objects={
        "color_jitter": color_jitter,
        "preprocess_input": tf.keras.applications.resnet50.preprocess_input
    }
)

print("Best ResNet50 model loaded successfully!")

In [ ]:
# Unfreeze the last 20 layers

# Get the ResNet50 base model
base_model_resnet = best_resnet50.get_layer("resnet50")

# Initially freeze all layers
base_model_resnet.trainable = True

# Freeze all except the last 20 layers
for layer in base_model_resnet.layers[:-20]:
    layer.trainable = False

for layer in base_model_resnet.layers[-20:]:
    layer.trainable = True

# Display trainable layer information
trainable_layers = [
    layer.name for layer in base_model_resnet.layers
    if layer.trainable
]

print("Total ResNet50 layers:", len(base_model_resnet.layers))
print("Trainable ResNet50 layers:", len(trainable_layers))
print("\nLast 20 layers:")
for name in trainable_layers:
    print(name)

In [ ]:
# Recompile with a lower learning rate

best_resnet50.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("ResNet50 recompiled for fine-tuning!")

In [ ]:
# Fine-tune ResNet50

# Fine-tuning callbacks

resnet50_finetune_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath="/content/drive/MyDrive/SmartVision_AI/saved_models/resnet50_finetuned_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
]

print("Fine-tuning callbacks configured!")

In [ ]:
history_resnet50_finetune = best_resnet50.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=resnet50_finetune_callbacks
)

In [ ]:
# Evaluate fine-tuned ResNet50

# Load the best fine-tuned ResNet50 model

best_resnet50_finetuned = tf.keras.models.load_model(
    "/content/drive/MyDrive/SmartVision_AI/saved_models/resnet50_finetuned_best.keras",
    custom_objects={
        "color_jitter": color_jitter,
        "preprocess_input": tf.keras.applications.resnet50.preprocess_input
    }
)

# Predict test dataset
y_prob_resnet = best_resnet50_finetuned.predict(test_dataset)

y_pred_resnet = np.argmax(y_prob_resnet, axis=1)

# Actual labels
y_true_resnet = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in test_dataset
])

# Calculate performance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("\nRESNET50 FINE-TUNED TEST PERFORMANCE")
print("=" * 50)

print(f"Accuracy          : {accuracy_score(y_true_resnet, y_pred_resnet)*100:.2f}%")
print(f"Precision (Macro) : {precision_score(y_true_resnet, y_pred_resnet, average='macro', zero_division=0)*100:.2f}%")
print(f"Recall (Macro)    : {recall_score(y_true_resnet, y_pred_resnet, average='macro', zero_division=0)*100:.2f}%")
print(f"F1 Score (Macro)  : {f1_score(y_true_resnet, y_pred_resnet, average='macro', zero_division=0)*100:.2f}%")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true_resnet,
    y_pred_resnet,
    target_names=class_names,
    zero_division=0
))

In [ ]:
import os
import json

HISTORY_DIR = "/content/drive/MyDrive/SmartVision_AI/results/training_history"

os.makedirs(HISTORY_DIR, exist_ok=True)

histories = {
    "vgg16_augmented": history_vgg16_aug,
    "vgg16_weighted": history_vgg16_weighted,
    "resnet50_augmented": history_resnet50,
    "resnet50_finetuned": history_resnet50_finetune
}

for name, history in histories.items():

    with open(
        os.path.join(HISTORY_DIR, f"{name}_history.json"),
        "w"
    ) as f:
        json.dump(history.history, f, indent=4)

    print(f"Saved: {name}_history.json")

print("\nAll training histories saved successfully!")

# MobileNetV2


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# Load pretrained MobileNetV2
base_model_mobile = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained layers initially
base_model_mobile.trainable = False

# Build classification model
inputs = layers.Input(shape=(224, 224, 3))

# Data augmentation
x = data_augmentation(inputs)

# MobileNetV2 preprocessing
x = layers.Lambda(
    tf.keras.applications.mobilenet_v2.preprocess_input
)(x)

# Feature extraction
x = base_model_mobile(x, training=False)

# Classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)

# 25-class output
outputs = layers.Dense(25, activation="softmax")(x)

# Create model
mobilenet_model = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="MobileNetV2_Augmented"
)

mobilenet_model.summary()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# Load pretrained MobileNetV2
base_model_mobile = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained base
base_model_mobile.trainable = False

# Create model input
inputs = layers.Input(shape=(224, 224, 3))

# Apply standard data augmentation
x = data_augmentation(inputs)

# MobileNetV2 preprocessing
x = layers.Lambda(
    tf.keras.applications.mobilenet_v2.preprocess_input
)(x)

# Extract image features
x = base_model_mobile(x, training=False)

# Custom classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)

# 25-class output
outputs = layers.Dense(25, activation="softmax")(x)

# Create final model
mobilenet_model = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="MobileNetV2_Augmented"
)

# Display model architecture
mobilenet_model.summary()

In [ ]:
# Compile the model

# COMPILE MOBILENETV2 MODEL

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("MobileNetV2 compiled successfully!")

In [ ]:
# Create training callbacks

# MOBILENETV2 TRAINING CALLBACKS

mobilenet_checkpoint = os.path.join(
    MODEL_DIR,
    "mobilenetv2_best.keras"
)

mobilenet_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=mobilenet_checkpoint,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("MobileNetV2 callbacks created successfully!")
print("Checkpoint path:", mobilenet_checkpoint)

In [ ]:
# Verify the classification dataset

# CHECK CLASSIFICATION DATASET

import os

for split in ["train", "val", "test"]:
    split_path = os.path.join(CLASSIFICATION_DIR, split)

    print(f"\n{split.upper()} folder exists:",
          os.path.exists(split_path))

    if os.path.exists(split_path):
        classes = os.listdir(split_path)
        print("Number of classes:", len(classes))
        print("Sample classes:", sorted(classes)[:5])

In [ ]:
# Verify image counts

# COUNT IMAGES IN EACH DATASET SPLIT

import os

for split in ["train", "val", "test"]:

    split_path = os.path.join(CLASSIFICATION_DIR, split)

    total_images = 0

    print(f"\n{split.upper()} DATASET")

    for class_name in sorted(os.listdir(split_path)):

        class_path = os.path.join(split_path, class_name)

        if os.path.isdir(class_path):
            count = len([
                f for f in os.listdir(class_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ])

            total_images += count

    print("Total images:", total_images)

In [ ]:
# Load datasets into TensorFlow

# LOAD CLASSIFICATION DATASETS

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
SEED = 42

train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLASSIFICATION_DIR, "train"),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLASSIFICATION_DIR, "val"),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLASSIFICATION_DIR, "test"),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_dataset.class_names

print("\nNumber of classes:", len(class_names))
print("Class names:", class_names)

In [ ]:
# Optimize the dataset pipeline

# OPTIMIZE DATASET PIPELINE

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

val_dataset = val_dataset.prefetch(
    buffer_size=AUTOTUNE
)

test_dataset = test_dataset.prefetch(
    buffer_size=AUTOTUNE
)

print("Dataset optimization completed!")
print("Training, validation and testing datasets are ready.")

In [ ]:
# Start MobileNetV2 training

# MOBILENETV2 MODEL TRAINING

EPOCHS = 25

history_mobilenet = mobilenet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=mobilenet_callbacks
)

In [ ]:
# SAVE MOBILENETV2 TRAINING HISTORY

import json
import os

mobilenet_history_path = os.path.join(
    HISTORY_DIR,
    "mobilenetv2_history.json"
)

with open(mobilenet_history_path, "w") as f:
    json.dump(history_mobilenet.history, f)

print("MobileNetV2 training history saved successfully!")
print("Location:", mobilenet_history_path)

In [ ]:
# Evaluate MobileNetV2

# MOBILENETV2 MODEL EVALUATION

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Generate predictions
y_prob = mobilenet_model.predict(test_dataset)

# Predicted class indices
y_pred = np.argmax(y_prob, axis=1)

# Actual class indices
y_true = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in test_dataset
])

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)

precision_macro = precision_score(
    y_true, y_pred, average="macro", zero_division=0
)

precision_weighted = precision_score(
    y_true, y_pred, average="weighted", zero_division=0
)

recall_macro = recall_score(
    y_true, y_pred, average="macro", zero_division=0
)

recall_weighted = recall_score(
    y_true, y_pred, average="weighted", zero_division=0
)

f1_macro = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

f1_weighted = f1_score(
    y_true, y_pred, average="weighted", zero_division=0
)

# Top-5 accuracy
top5_accuracy = tf.keras.metrics.top_k_categorical_accuracy(
    np.concatenate([
        labels.numpy() for images, labels in test_dataset
    ]),
    y_prob,
    k=5
).numpy().mean()

print("MOBILENETV2 TEST RESULTS")
print("-" * 40)
print(f"Accuracy:           {accuracy * 100:.2f}%")
print(f"Macro Precision:    {precision_macro * 100:.2f}%")
print(f"Weighted Precision: {precision_weighted * 100:.2f}%")
print(f"Macro Recall:       {recall_macro * 100:.2f}%")
print(f"Weighted Recall:    {recall_weighted * 100:.2f}%")
print(f"Macro F1-Score:     {f1_macro * 100:.2f}%")
print(f"Weighted F1-Score:  {f1_weighted * 100:.2f}%")
print(f"Top-5 Accuracy:     {top5_accuracy * 100:.2f}%")

print("\nCLASSIFICATION REPORT")
print("-" * 60)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

In [ ]:
# SAVE MOBILENETV2 EVALUATION REPORT

import json
import os
from sklearn.metrics import classification_report

mobilenet_results = {
    "model": "MobileNetV2",
    "test_accuracy": float(accuracy),
    "macro_precision": float(precision_macro),
    "weighted_precision": float(precision_weighted),
    "macro_recall": float(recall_macro),
    "weighted_recall": float(recall_weighted),
    "macro_f1": float(f1_macro),
    "weighted_f1": float(f1_weighted),
    "top5_accuracy": float(top5_accuracy)
}

report_path = os.path.join(
    REPORT_DIR,
    "mobilenetv2_evaluation.json"
)

with open(report_path, "w") as f:
    json.dump(mobilenet_results, f, indent=4)

print("Evaluation report saved successfully!")
print("Location:", report_path)

In [ ]:
# Load the best MobileNetV2 checkpoint

# LOAD BEST MOBILENETV2 CHECKPOINT

best_mobilenet = tf.keras.models.load_model(
    os.path.join(MODEL_DIR, "mobilenetv2_best.keras"),
    custom_objects={
        "color_jitter": color_jitter,
        "preprocess_input":
            tf.keras.applications.mobilenet_v2.preprocess_input
    }
)

print("Best MobileNetV2 checkpoint loaded successfully!")

In [ ]:
# Unfreeze the last 20 layers

# MOBILENETV2 FINE-TUNING SETUP

base_model_mobile = best_mobilenet.get_layer(
    "mobilenetv2_1.00_224"
)

# Enable fine-tuning
base_model_mobile.trainable = True

# Freeze all layers except the last 20
for layer in base_model_mobile.layers[:-20]:
    layer.trainable = False

for layer in base_model_mobile.layers[-20:]:
    layer.trainable = True

# Keep Batch Normalization layers frozen
for layer in base_model_mobile.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Total layers:", len(base_model_mobile.layers))
print(
    "Trainable layers:",
    sum(layer.trainable for layer in base_model_mobile.layers)
)

In [ ]:
# Compile the fine-tuned model

# COMPILE MOBILENETV2 FOR FINE-TUNING

best_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("MobileNetV2 fine-tuning compilation completed!")

In [ ]:
# Fine-tune MobileNetV2

# MOBILENETV2 FINE-TUNING CALLBACKS

mobilenet_finetune_checkpoint = os.path.join(
    MODEL_DIR,
    "mobilenetv2_finetuned_best.keras"
)

mobilenet_finetune_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=mobilenet_finetune_checkpoint,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("Fine-tuning callbacks created successfully!")
print("Checkpoint:", mobilenet_finetune_checkpoint)

In [ ]:
# Start fine-tuning

# MOBILENETV2 FINE-TUNING

FINE_TUNE_EPOCHS = 15

history_mobilenet_finetune = best_mobilenet.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=mobilenet_finetune_callbacks
)

In [ ]:
# Save fine-tuning history

# SAVE MOBILENETV2 FINE-TUNING HISTORY

import json
import os

history_path = os.path.join(
    HISTORY_DIR,
    "mobilenetv2_finetuned_history.json"
)

with open(history_path, "w") as f:
    json.dump(history_mobilenet_finetune.history, f, indent=4)

print("Fine-tuning history saved successfully!")
print("Location:", history_path)

In [ ]:
# Load the best fine-tuned checkpoint

# LOAD BEST FINE-TUNED MOBILENETV2

best_mobilenet_finetuned = tf.keras.models.load_model(
    os.path.join(
        MODEL_DIR,
        "mobilenetv2_finetuned_best.keras"
    ),
    custom_objects={
        "color_jitter": color_jitter,
        "preprocess_input":
            tf.keras.applications.mobilenet_v2.preprocess_input
    }
)

print("Best fine-tuned MobileNetV2 loaded successfully!")

In [ ]:
# Test evaluation

# FINE-TUNED MOBILENETV2 EVALUATION

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Generate predictions
y_prob_finetuned = best_mobilenet_finetuned.predict(test_dataset)

# Predicted labels
y_pred_finetuned = np.argmax(y_prob_finetuned, axis=1)

# Actual labels
y_true = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in test_dataset
])

# Calculate metrics
accuracy_finetuned = accuracy_score(y_true, y_pred_finetuned)

precision_macro_finetuned = precision_score(
    y_true, y_pred_finetuned,
    average="macro", zero_division=0
)

precision_weighted_finetuned = precision_score(
    y_true, y_pred_finetuned,
    average="weighted", zero_division=0
)

recall_macro_finetuned = recall_score(
    y_true, y_pred_finetuned,
    average="macro", zero_division=0
)

recall_weighted_finetuned = recall_score(
    y_true, y_pred_finetuned,
    average="weighted", zero_division=0
)

f1_macro_finetuned = f1_score(
    y_true, y_pred_finetuned,
    average="macro", zero_division=0
)

f1_weighted_finetuned = f1_score(
    y_true, y_pred_finetuned,
    average="weighted", zero_division=0
)

# Top-5 accuracy
top5_finetuned = tf.keras.metrics.top_k_categorical_accuracy(
    np.concatenate([
        labels.numpy() for images, labels in test_dataset
    ]),
    y_prob_finetuned,
    k=5
).numpy().mean()

# Display results
print("FINE-TUNED MOBILENETV2 TEST RESULTS")
print("-" * 45)

print(f"Accuracy:           {accuracy_finetuned * 100:.2f}%")
print(f"Macro Precision:    {precision_macro_finetuned * 100:.2f}%")
print(f"Weighted Precision: {precision_weighted_finetuned * 100:.2f}%")
print(f"Macro Recall:       {recall_macro_finetuned * 100:.2f}%")
print(f"Weighted Recall:    {recall_weighted_finetuned * 100:.2f}%")
print(f"Macro F1-Score:     {f1_macro_finetuned * 100:.2f}%")
print(f"Weighted F1-Score:  {f1_weighted_finetuned * 100:.2f}%")
print(f"Top-5 Accuracy:     {top5_finetuned * 100:.2f}%")

print("\nCLASSIFICATION REPORT")
print("-" * 60)

print(classification_report(
    y_true,
    y_pred_finetuned,
    target_names=class_names,
    zero_division=0
))

# EfficientNetB0

In [ ]:
# Build EfficientNetB0 architecture

# STEP 1: BUILD EFFICIENTNETB0 MODEL

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

# Load pretrained EfficientNetB0
base_model_eff = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained base
base_model_eff.trainable = False

# Model input
inputs = layers.Input(shape=(224, 224, 3))

# Data augmentation
x = data_augmentation(inputs)

# Extract image features
x = base_model_eff(x, training=False)

# Global Average Pooling
x = layers.GlobalAveragePooling2D()(x)

# Custom classification head
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)

# 25-class output
outputs = layers.Dense(25, activation="softmax")(x)

# Create final model
efficientnet_model = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="EfficientNetB0_Augmented"
)

# Display architecture
efficientnet_model.summary()

In [ ]:
# COMPILE EFFICIENTNETB0 MODEL

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("EfficientNetB0 compiled successfully!")

In [ ]:
# EFFICIENTNETB0 TRAINING CALLBACKS

efficientnet_checkpoint = os.path.join(
    MODEL_DIR,
    "efficientnetb0_best.keras"
)

efficientnet_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=efficientnet_checkpoint,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("EfficientNetB0 callbacks created successfully!")
print("Checkpoint:", efficientnet_checkpoint)

In [ ]:
# Train EfficientNetB0

# EFFICIENTNETB0 MODEL TRAINING

EPOCHS = 25

history_efficientnet = efficientnet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=efficientnet_callbacks
)

In [ ]:
# SAVE EFFICIENTNETB0 TRAINING HISTORY

import json
import os

efficientnet_history_path = os.path.join(
    HISTORY_DIR,
    "efficientnetb0_history.json"
)

with open(efficientnet_history_path, "w") as f:
    json.dump(history_efficientnet.history, f, indent=4)

print("EfficientNetB0 training history saved successfully!")
print("Location:", efficientnet_history_path)

In [ ]:
# EFFICIENTNETB0 MODEL EVALUATION

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Generate predictions
y_prob_eff = efficientnet_model.predict(test_dataset)

# Predicted labels
y_pred_eff = np.argmax(y_prob_eff, axis=1)

# Actual labels
y_true_eff = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in test_dataset
])

# Calculate metrics
eff_accuracy = accuracy_score(y_true_eff, y_pred_eff)

eff_precision_macro = precision_score(
    y_true_eff, y_pred_eff,
    average="macro", zero_division=0
)

eff_precision_weighted = precision_score(
    y_true_eff, y_pred_eff,
    average="weighted", zero_division=0
)

eff_recall_macro = recall_score(
    y_true_eff, y_pred_eff,
    average="macro", zero_division=0
)

eff_recall_weighted = recall_score(
    y_true_eff, y_pred_eff,
    average="weighted", zero_division=0
)

eff_f1_macro = f1_score(
    y_true_eff, y_pred_eff,
    average="macro", zero_division=0
)

eff_f1_weighted = f1_score(
    y_true_eff, y_pred_eff,
    average="weighted", zero_division=0
)

# Top-5 accuracy
y_true_onehot_eff = np.concatenate([
    labels.numpy() for images, labels in test_dataset
])

eff_top5 = tf.keras.metrics.top_k_categorical_accuracy(
    y_true_onehot_eff,
    y_prob_eff,
    k=5
).numpy().mean()

# Display results
print("EFFICIENTNETB0 TEST RESULTS")
print("-" * 45)

print(f"Accuracy:           {eff_accuracy * 100:.2f}%")
print(f"Macro Precision:    {eff_precision_macro * 100:.2f}%")
print(f"Weighted Precision: {eff_precision_weighted * 100:.2f}%")
print(f"Macro Recall:       {eff_recall_macro * 100:.2f}%")
print(f"Weighted Recall:    {eff_recall_weighted * 100:.2f}%")
print(f"Macro F1-Score:     {eff_f1_macro * 100:.2f}%")
print(f"Weighted F1-Score:  {eff_f1_weighted * 100:.2f}%")
print(f"Top-5 Accuracy:     {eff_top5 * 100:.2f}%")

print("\nCLASSIFICATION REPORT")
print("-" * 60)

print(classification_report(
    y_true_eff,
    y_pred_eff,
    target_names=class_names,
    zero_division=0
))

In [ ]:
# let's verify your saved model

from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf

MODEL_PATH = "/content/drive/MyDrive/SmartVision_AI/saved_models/efficientnetb0_best.keras"

print("GPU:", tf.config.list_physical_devices('GPU'))
print("Checkpoint exists:", os.path.exists(MODEL_PATH))

if os.path.exists(MODEL_PATH):
    print("Checkpoint size:",
          round(os.path.getsize(MODEL_PATH) / (1024**2), 2), "MB")

In [ ]:
import tensorflow as tf
import os

MODEL_PATH = "/content/drive/MyDrive/SmartVision_AI/saved_models/efficientnetb0_best.keras"

print("Loading EfficientNetB0 checkpoint...")

efficientnet_model = tf.keras.models.load_model(MODEL_PATH)

print("\nModel loaded successfully!")
print("Model name:", efficientnet_model.name)
print("Input shape:", efficientnet_model.input_shape)
print("Output shape:", efficientnet_model.output_shape)
print("Available GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
import os

CLASSIFICATION_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/classification"

for split in ["train", "val", "test"]:
    path = os.path.join(CLASSIFICATION_DIR, split)

    if os.path.exists(path):
        classes = os.listdir(path)
        image_count = sum(
            len(files)
            for _, _, files in os.walk(path)
        )

        print(f"{split.upper()}:")
        print("Classes:", len(classes))
        print("Images:", image_count)
    else:
        print(f"{split.upper()}: NOT FOUND")

In [ ]:
import os

HISTORY_DIR = "/content/drive/MyDrive/SmartVision_AI/results/training_history"

print("Available training history files:\n")

for file in os.listdir(HISTORY_DIR):
    print(file)

In [ ]:
import os

history_path = "/content/drive/MyDrive/SmartVision_AI/results/training_history/efficientnetb0_history.json"

file_size = os.path.getsize(history_path)

print("History file size:", file_size, "bytes")

if file_size == 0:
    print("The history file is empty.")
else:
    print("The file contains data.")

In [ ]:
import tensorflow as tf

CLASSIFICATION_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/classification"

test_dataset = tf.keras.utils.image_dataset_from_directory(
    CLASSIFICATION_DIR + "/test",
    image_size=(224, 224),
    batch_size=32,
    label_mode="categorical",
    shuffle=False
)

print("Test dataset loaded successfully!")
print("Number of batches:", tf.data.experimental.cardinality(test_dataset).numpy())
print("Number of classes:", len(test_dataset.class_names))

In [ ]:
# Evaluate EfficientNetB0

print("Evaluating EfficientNetB0 on test dataset...")

efficientnet_test_results = efficientnet_model.evaluate(
    test_dataset,
    verbose=1
)

print("\n========== EfficientNetB0 Test Results ==========")
print(f"Test Loss: {efficientnet_test_results[0]:.4f}")
print(f"Test Accuracy: {efficientnet_test_results[1] * 100:.2f}%")

In [ ]:
# Generate EfficientNetB0 classification report

import numpy as np
from sklearn.metrics import classification_report

# Get predictions
y_pred_probs = efficientnet_model.predict(test_dataset, verbose=1)

# Convert predictions to class labels
y_pred = np.argmax(y_pred_probs, axis=1)

# Get actual labels
y_true = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in test_dataset
])

# Generate classification report
class_names = test_dataset.class_names

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
)

print("EFFICIENTNETB0 CLASSIFICATION REPORT")
print(report)

In [ ]:
# Verify accuracy and save the report

from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import json
import os

# Calculate accuracy directly
direct_accuracy = accuracy_score(y_true, y_pred)

print("Direct Test Accuracy:", round(direct_accuracy * 100, 2), "%")

# Compare with classification report
print("Classification report accuracy:", round(
    np.mean(y_true == y_pred) * 100, 2
), "%")

# Save evaluation report
REPORT_DIR = "/content/drive/MyDrive/SmartVision_AI/results/evaluation_reports"

os.makedirs(REPORT_DIR, exist_ok=True)

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True
)

report_path = os.path.join(
    REPORT_DIR,
    "efficientnetb0_evaluation.json"
)

with open(report_path, "w") as f:
    json.dump(report_dict, f, indent=4)

print("Evaluation report saved:", report_path)

In [ ]:
# Generate the EfficientNetB0 confusion matrix

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(20, 16))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("EfficientNetB0 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

# Save confusion matrix
CONFUSION_DIR = "/content/drive/MyDrive/SmartVision_AI/results/confusion_matrices"

os.makedirs(CONFUSION_DIR, exist_ok=True)

plt.savefig(
    os.path.join(CONFUSION_DIR, "efficientnetb0_confusion_matrix.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Confusion matrix saved successfully!")

In [ ]:
# Create the comparison table

import pandas as pd
import os

comparison_data = {
    "Model": [
        "VGG16",
        "ResNet50",
        "MobileNetV2",
        "EfficientNetB0"
    ],
    "Test Accuracy (%)": [
        54.24, 60.69, 56.93, 55.96
    ],
    "Macro Precision (%)": [
        52.44, 57.53, 57.11, 57.31
    ],
    "Macro Recall (%)": [
        40.27, 50.28, 46.39, 44.98
    ],
    "Macro F1 (%)": [
        43.02, 51.29, 48.43, 47.39
    ]
}

comparison_df = pd.DataFrame(comparison_data)

display(comparison_df)

# Save comparison
RESULT_DIR = "/content/drive/MyDrive/SmartVision_AI/results"

comparison_path = os.path.join(
    RESULT_DIR,
    "four_model_comparison.csv"
)

comparison_df.to_csv(comparison_path, index=False)

print("Comparison saved successfully!")
print("Location:", comparison_path)

In [ ]:
# Save the classification results summary

import os

PROJECT_DIR = "/content/drive/MyDrive/SmartVision_AI"
RESULT_DIR = os.path.join(PROJECT_DIR, "results")

summary = """
SMARTVISION AI
CLASSIFICATION MODEL EVALUATION SUMMARY

Dataset:
- Total images: 5733
- Training images: 3948
- Validation images: 854
- Testing images: 931
- Number of classes: 25

MODEL PERFORMANCE:

1. VGG16
   Test Accuracy: 54.24%
   Macro Precision: 52.44%
   Macro Recall: 40.27%
   Macro F1: 43.02%

2. ResNet50
   Test Accuracy: 60.69%
   Macro Precision: 57.53%
   Macro Recall: 50.28%
   Macro F1: 51.29%

3. MobileNetV2
   Test Accuracy: 56.93%
   Macro Precision: 57.11%
   Macro Recall: 46.39%
   Macro F1: 48.43%

4. EfficientNetB0
   Test Accuracy: 55.96%
   Macro Precision: 57.31%
   Macro Recall: 44.98%
   Macro F1: 47.39%

CONCLUSION:

ResNet50 achieved the highest test accuracy (60.69%)
and macro F1-score (51.29%) among the four models.

ResNet50 is the best-performing classification model
in this experiment.

All results are based on evaluation using the test dataset.
"""

summary_path = os.path.join(
    RESULT_DIR,
    "classification_phase_summary.txt"
)

with open(summary_path, "w") as f:
    f.write(summary)

print("Classification summary saved successfully!")
print("Location:", summary_path)

# Phase 3: YOLOv8 Object Detection.

In [ ]:
!pip install ultralytics -q

In [ ]:
import ultralytics
import torch

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
# Download pretrained YOLOv8 weights

from ultralytics import YOLO
import os

# Load pretrained YOLOv8 Nano model
yolo_model = YOLO("yolov8n.pt")

print("YOLOv8 model loaded successfully!")
print("Model: YOLOv8n")
print("Pretrained classes:", len(yolo_model.names))
print("First 10 classes:", list(yolo_model.names.values())[:10])

Step 3.2 — Detection dataset preparation

In [ ]:
# Detection dataset preparation

import os

DETECTION_DIR = "/content/smartvision_dataset/detection"

print("Detection dataset exists:", os.path.exists(DETECTION_DIR))

if os.path.exists(DETECTION_DIR):
    for root, dirs, files in os.walk(DETECTION_DIR):
        level = root.replace(DETECTION_DIR, "").count(os.sep)

        if level <= 3:
            indent = "    " * level
            print(f"{indent}{os.path.basename(root)}/")

            for file in files[:5]:
                print(f"{indent}    {file}")
else:
    print("Detection dataset not found!")

In [ ]:
# Check images, labels and data.yaml

import os

DETECTION_DIR = "/content/smartvision_dataset/detection"

IMAGE_DIR = os.path.join(DETECTION_DIR, "images")
LABEL_DIR = os.path.join(DETECTION_DIR, "labels")
YAML_PATH = os.path.join(DETECTION_DIR, "data.yaml")

# Count images and labels
images = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

labels = [
    f for f in os.listdir(LABEL_DIR)
    if f.endswith(".txt")
]

print("Total images:", len(images))
print("Total label files:", len(labels))

# Check matching image-label pairs
image_names = {os.path.splitext(f)[0] for f in images}
label_names = {os.path.splitext(f)[0] for f in labels}

print("Images without labels:", len(image_names - label_names))
print("Labels without images:", len(label_names - image_names))

# Display data.yaml
print("\n========== DATA.YAML ==========")

with open(YAML_PATH, "r") as f:
    print(f.read())

In [ ]:
# Check the actual class IDs in your labels

import os
from collections import Counter

LABEL_DIR = "/content/smartvision_dataset/detection/labels"

class_counts = Counter()
invalid_lines = []

for filename in os.listdir(LABEL_DIR):
    if not filename.endswith(".txt"):
        continue

    filepath = os.path.join(LABEL_DIR, filename)

    with open(filepath, "r") as f:
        for line_number, line in enumerate(f, start=1):
            parts = line.strip().split()

            if not parts:
                continue

            if len(parts) != 5:
                invalid_lines.append((filename, line_number, line.strip()))
                continue

            try:
                class_id = int(parts[0])
                class_counts[class_id] += 1
            except ValueError:
                invalid_lines.append((filename, line_number, line.strip()))

print("========== CLASS ID COUNTS ==========")

for class_id in sorted(class_counts):
    print(f"Class ID {class_id}: {class_counts[class_id]} objects")

print("\nTotal classes found:", len(class_counts))
print("Invalid annotation lines:", len(invalid_lines))

if invalid_lines:
    print("\nFirst 5 invalid lines:")
    for item in invalid_lines[:5]:
        print(item)

In [ ]:
# Verify bounding-box coordinates

import os

LABEL_DIR = "/content/smartvision_dataset/detection/labels"

invalid_boxes = []
total_boxes = 0

for filename in os.listdir(LABEL_DIR):
    if not filename.endswith(".txt"):
        continue

    filepath = os.path.join(LABEL_DIR, filename)

    with open(filepath, "r") as f:
        for line_number, line in enumerate(f, start=1):
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])

            total_boxes += 1

            if not (
                0 <= class_id <= 24
                and 0 <= x_center <= 1
                and 0 <= y_center <= 1
                and 0 < width <= 1
                and 0 < height <= 1
            ):
                invalid_boxes.append(
                    (filename, line_number, line.strip())
                )

print("Total bounding boxes:", total_boxes)
print("Invalid bounding boxes:", len(invalid_boxes))

if invalid_boxes:
    print("\nFirst 5 invalid annotations:")
    for item in invalid_boxes[:5]:
        print(item)
else:
    print("All bounding-box annotations are valid!")

The x_center and y_center values are greater than 1. In standard YOLO format, all four coordinates must be normalized between 0 and 1.
Why might this happen?
Your annotations may have been generated using incorrect image dimensions, or the coordinates may have been converted incorrectly.

We should investigate the original annotation format before correcting anything. Simply clipping values to 1 would damage the bounding boxes.

In [ ]:
# Inspect the original images and annotations

import os
from PIL import Image

IMAGE_DIR = "/content/smartvision_dataset/detection/images"
LABEL_DIR = "/content/smartvision_dataset/detection/labels"

sample_files = [
    "image_000684.jpg",
    "image_000684.txt"
]

image_path = os.path.join(IMAGE_DIR, sample_files[0])
label_path = os.path.join(LABEL_DIR, sample_files[1])

# Read image dimensions
with Image.open(image_path) as img:
    print("Image dimensions:", img.size)
    print("Image width:", img.width)
    print("Image height:", img.height)

# Read annotation
print("\n========== ORIGINAL LABELS ==========")

with open(label_path, "r") as f:
    for line in f.readlines()[:10]:
        print(line.strip())

In [ ]:
# Visually inspect the annotations

import os
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

IMAGE_DIR = "/content/smartvision_dataset/detection/images"
LABEL_DIR = "/content/smartvision_dataset/detection/labels"

image_name = "image_000684.jpg"

image_path = os.path.join(IMAGE_DIR, image_name)
label_path = os.path.join(
    LABEL_DIR,
    os.path.splitext(image_name)[0] + ".txt"
)

image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)

img_width, img_height = image.size

with open(label_path, "r") as f:
    for line in f:
        parts = line.strip().split()

        if len(parts) != 5:
            continue

        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:])

        # Convert YOLO coordinates into image pixel coordinates
        x1 = (x_center - width / 2) * img_width
        y1 = (y_center - height / 2) * img_height
        x2 = (x_center + width / 2) * img_width
        y2 = (y_center + height / 2) * img_height

        valid = (
            0 <= x_center <= 1
            and 0 <= y_center <= 1
            and 0 < width <= 1
            and 0 < height <= 1
        )

        color = "green" if valid else "red"

        draw.rectangle(
            [x1, y1, x2, y2],
            outline=color,
            width=3
        )

        draw.text(
            (max(0, x1), max(0, y1)),
            f"Class {class_id}",
            fill=color
        )

plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.title("Green = valid coordinates | Red = invalid coordinates")
plt.axis("off")
plt.show()

In [ ]:
# Inspect the dataset metadata

import os
import json

METADATA_PATH = "/content/smartvision_dataset/dataset_metadata.json"

print("Metadata file exists:", os.path.exists(METADATA_PATH))

if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)

    print("\n========== DATASET METADATA ==========")
    print(json.dumps(metadata, indent=4))
else:
    print("Metadata file not found.")

In [ ]:
# Check for original annotation files

import os

DATASET_DIR = "/content/smartvision_dataset"

print("========== DATASET FILES ==========")

for root, dirs, files in os.walk(DATASET_DIR):
    # Avoid listing thousands of image and label files
    dirs[:] = [
        d for d in dirs
        if d not in ["images", "labels"]
    ]

    relative_path = os.path.relpath(root, DATASET_DIR)

    for filename in files:
        if filename.lower().endswith((
            ".json", ".xml", ".csv", ".yaml", ".yml"
        )):
            print(os.path.join(relative_path, filename))

print("\n========== TOP-LEVEL CONTENTS ==========")

for item in os.listdir(DATASET_DIR):
    print(item)

In [ ]:
# Check whether the original COCO annotations are still available

print("========== ORIGINAL DATA CHECK ==========")

print("class_images available:", "class_images" in globals())

if "class_images" in globals():

    print("\nAvailable classes:", len(class_images))

    for class_name in list(class_images.keys())[:5]:

        items = class_images[class_name]

        print(f"\nClass: {class_name}")
        print("Number of images:", len(items))

        if items:
            item = items[0]

            print("Image ID:", item["idx"])
            print("Image dimensions:", item["image"].size)
            print("Annotation keys:", item["annotations"].keys())
            print("First bounding boxes:",
                  item["annotations"]["bbox"][:3])

verify one important detail: the original annotation format.

In [ ]:
# Inspect the original image 684

from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Find original image ID 684
item_684 = None

for class_name, items in class_images.items():
    for item in items:
        if item["idx"] == 684:
            item_684 = item
            break
    if item_684 is not None:
        break

if item_684 is None:
    print("Image ID 684 not found!")
else:
    original = item_684["image"].convert("RGB")
    annotations = item_684["annotations"]

    print("Original image ID:", item_684["idx"])
    print("Original image dimensions:", original.size)
    print("Number of bounding boxes:", len(annotations["bbox"]))

    print("\nFirst 10 original bounding boxes:")
    for bbox, category in zip(
        annotations["bbox"][:10],
        annotations["category"][:10]
    ):
        print("Category:", category, "| Bounding box:", bbox)

    # Display original image
    plt.figure(figsize=(10, 7))
    plt.imshow(original)
    plt.title("Original COCO Image - ID 684")
    plt.axis("off")
    plt.show()

In [ ]:
# Verify the corrected bounding boxes visually

from PIL import ImageDraw
import matplotlib.pyplot as plt

# Use the original image we found
image = item_684["image"].convert("RGB").copy()
draw = ImageDraw.Draw(image)

img_width, img_height = image.size

boxes = item_684["annotations"]["bbox"]
categories = item_684["annotations"]["category"]

print("Image dimensions:", image.size)
print("Total annotations:", len(boxes))

for bbox, category in zip(boxes, categories):

    # Original annotations are xyxy
    x1, y1, x2, y2 = map(float, bbox)

    # Keep coordinates inside the image
    x1 = max(0, min(x1, img_width))
    y1 = max(0, min(y1, img_height))
    x2 = max(0, min(x2, img_width))
    y2 = max(0, min(y2, img_height))

    if x2 <= x1 or y2 <= y1:
        continue

    # Draw original bounding box
    draw.rectangle(
        [x1, y1, x2, y2],
        outline="lime",
        width=3
    )

    draw.text(
        (x1, max(0, y1 - 12)),
        str(category),
        fill="yellow"
    )

plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.title("Corrected Bounding Boxes - Original Image 684")
plt.axis("off")
plt.show()

In [ ]:
# Check unique images and class mapping

from collections import defaultdict

# Original class-to-COCO-category mapping
print("========== CLASS MAPPING ==========")

for yolo_id, (class_name, coco_id) in enumerate(SELECTED_CLASSES.items()):
    print(f"{yolo_id}: {class_name} (COCO ID: {coco_id})")

# Combine images using their original COCO image IDs
unique_images = {}

for class_name, items in class_images.items():
    for item in items:
        image_id = item["idx"]

        if image_id not in unique_images:
            unique_images[image_id] = {
                "image": item["image"],
                "annotations": []
            }

        for bbox, category in zip(
            item["annotations"]["bbox"],
            item["annotations"]["category"]
        ):
            unique_images[image_id]["annotations"].append(
                (bbox, category)
            )

print("\n========== DATASET SUMMARY ==========")
print("Total class entries:", sum(len(v) for v in class_images.values()))
print("Unique original images:", len(unique_images))

total_objects = sum(
    len(data["annotations"])
    for data in unique_images.values()
)

print("Total collected annotations:", total_objects)

In [ ]:
# Check duplicate annotations

from collections import Counter

print("========== DUPLICATE ANNOTATION CHECK ==========")

image_id = 684

matching_items = []

for class_name, items in class_images.items():
    for item in items:
        if item["idx"] == image_id:
            matching_items.append((class_name, item))

print("Image ID:", image_id)
print("Occurrences across classes:", len(matching_items))

all_bbox_ids = []

for class_name, item in matching_items:
    bbox_ids = item["annotations"]["bbox_id"]
    all_bbox_ids.extend(bbox_ids)

print("Total annotation entries:", len(all_bbox_ids))
print("Unique bbox IDs:", len(set(all_bbox_ids)))
print("Duplicate annotation entries:",
      len(all_bbox_ids) - len(set(all_bbox_ids)))

print("\nFirst 10 bounding-box IDs:")
print(all_bbox_ids[:10])

In [ ]:
# Build a deduplicated annotation collection

from collections import defaultdict

# COCO category ID -> YOLO class ID
coco_to_yolo = {
    coco_id: yolo_id
    for yolo_id, coco_id in enumerate(SELECTED_CLASSES.values())
}

clean_images = {}

duplicate_annotations = 0
conflicting_annotations = 0

for class_name, items in class_images.items():

    for item in items:

        image_id = item["idx"]

        if image_id not in clean_images:
            clean_images[image_id] = {
                "image": item["image"],
                "annotations": {}
            }

        annotations = item["annotations"]

        for bbox_id, bbox, category in zip(
            annotations["bbox_id"],
            annotations["bbox"],
            annotations["category"]
        ):

            bbox_id = int(bbox_id)
            category = int(category)

            if bbox_id in clean_images[image_id]["annotations"]:
                duplicate_annotations += 1

                existing = clean_images[image_id]["annotations"][bbox_id]

                if existing["bbox"] != list(bbox) or existing["category"] != category:
                    conflicting_annotations += 1

                continue

            clean_images[image_id]["annotations"][bbox_id] = {
                "bbox": list(bbox),
                "category": category
            }

print("========== CLEAN DATASET SUMMARY ==========")

print("Unique images:", len(clean_images))

total_unique_annotations = sum(
    len(data["annotations"])
    for data in clean_images.values()
)

print("Unique annotations:", total_unique_annotations)
print("Duplicate entries removed:", duplicate_annotations)
print("Conflicting duplicate annotations:", conflicting_annotations)

print("\nImage 684 unique annotations:",
      len(clean_images[684]["annotations"]))

In [ ]:
# Validate all original bounding boxes

import math

total_boxes = 0
valid_boxes = 0
boxes_needing_clipping = 0
invalid_boxes = 0
unknown_classes = set()

for image_id, data in clean_images.items():

    img_width, img_height = data["image"].size

    for annotation in data["annotations"].values():

        total_boxes += 1

        category = annotation["category"]
        bbox = annotation["bbox"]

        if category not in coco_to_yolo:
            unknown_classes.add(category)
            invalid_boxes += 1
            continue

        try:
            x1, y1, x2, y2 = map(float, bbox)

            if not all(math.isfinite(v) for v in [x1, y1, x2, y2]):
                invalid_boxes += 1
                continue

            # Check whether clipping is required
            if (
                x1 < 0 or y1 < 0
                or x2 > img_width or y2 > img_height
            ):
                boxes_needing_clipping += 1

            # Clip to image boundaries
            x1 = max(0, min(x1, img_width))
            y1 = max(0, min(y1, img_height))
            x2 = max(0, min(x2, img_width))
            y2 = max(0, min(y2, img_height))

            if x2 <= x1 or y2 <= y1:
                invalid_boxes += 1
                continue

            valid_boxes += 1

        except (ValueError, TypeError):
            invalid_boxes += 1

print("========== BOUNDING BOX VALIDATION ==========")
print("Total annotations:", total_boxes)
print("Valid annotations:", valid_boxes)
print("Boxes needing clipping:", boxes_needing_clipping)
print("Invalid annotations:", invalid_boxes)
print("Unknown COCO class IDs:", unknown_classes)

In [ ]:
# Separate selected and unselected annotations

from collections import Counter

selected_counts = Counter()
excluded_counts = Counter()

for image_id, data in clean_images.items():

    for annotation in data["annotations"].values():

        category = annotation["category"]

        if category in coco_to_yolo:
            selected_counts[category] += 1
        else:
            excluded_counts[category] += 1

print("========== SELECTED CLASS ANNOTATIONS ==========")

for coco_id, yolo_id in sorted(
    [(coco_id, yolo_id) for coco_id, yolo_id in coco_to_yolo.items()],
    key=lambda x: x[1]
):
    class_name = list(SELECTED_CLASSES.keys())[yolo_id]
    print(f"{yolo_id}: {class_name}: {selected_counts[coco_id]}")

print("\nSelected annotations:", sum(selected_counts.values()))
print("Excluded annotations:", sum(excluded_counts.values()))
print("Total annotations:",
      sum(selected_counts.values()) + sum(excluded_counts.values()))

In [ ]:
# Prepare the train, validation and test split

import random

# Keep only images containing selected classes
filtered_images = {}

for image_id, data in clean_images.items():

    selected_annotations = {}

    for bbox_id, annotation in data["annotations"].items():

        if annotation["category"] in coco_to_yolo:
            selected_annotations[bbox_id] = annotation

    if selected_annotations:
        filtered_images[image_id] = {
            "image": data["image"],
            "annotations": selected_annotations
        }

# Reproducible random split
image_ids = list(filtered_images.keys())

random.seed(42)
random.shuffle(image_ids)

total = len(image_ids)

train_end = int(total * 0.70)
val_end = train_end + int(total * 0.15)

train_ids = image_ids[:train_end]
val_ids = image_ids[train_end:val_end]
test_ids = image_ids[val_end:]

print("========== DATASET SPLIT ==========")
print("Total unique images:", total)
print("Training images:", len(train_ids))
print("Validation images:", len(val_ids))
print("Testing images:", len(test_ids))

print("\nTotal:",
      len(train_ids) + len(val_ids) + len(test_ids))

print("\n========== OVERLAP CHECK ==========")
print("Train-Val overlap:", len(set(train_ids) & set(val_ids)))
print("Train-Test overlap:", len(set(train_ids) & set(test_ids)))
print("Val-Test overlap:", len(set(val_ids) & set(test_ids)))

In [ ]:
# Generate the corrected YOLO dataset

import os
import yaml
from PIL import Image

# Destination folder
OUTPUT_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed"


# Safety check: do not overwrite an existing dataset
if os.path.exists(OUTPUT_DIR):
    raise FileExistsError(
        "Destination already exists. Please inspect it before proceeding."
    )

# Create directory structure
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DIR, "images", split))
    os.makedirs(os.path.join(OUTPUT_DIR, "labels", split))

# Prepare split mapping
splits = {
    "train": train_ids,
    "val": val_ids,
    "test": test_ids
}

total_saved_images = 0
total_saved_boxes = 0
skipped_boxes = 0

# Generate images and labels
for split, ids in splits.items():

    for image_id in ids:

        data = filtered_images[image_id]

        image = data["image"].convert("RGB")
        img_width, img_height = image.size

        filename = f"image_{image_id:012d}"

        image_path = os.path.join(
            OUTPUT_DIR, "images", split, filename + ".jpg"
        )

        label_path = os.path.join(
            OUTPUT_DIR, "labels", split, filename + ".txt"
        )

        # Save image
        image.save(image_path, quality=95)

        label_lines = []

        for annotation in data["annotations"].values():

            category = annotation["category"]

            if category not in coco_to_yolo:
                continue

            x1, y1, x2, y2 = map(
                float, annotation["bbox"]
            )

            # Clip coordinates to image boundaries
            x1 = max(0, min(x1, img_width))
            y1 = max(0, min(y1, img_height))
            x2 = max(0, min(x2, img_width))
            y2 = max(0, min(y2, img_height))

            # Skip invalid boxes
            if x2 <= x1 or y2 <= y1:
                skipped_boxes += 1
                continue

            # Convert XYXY to normalized YOLO XYWH
            x_center = ((x1 + x2) / 2) / img_width
            y_center = ((y1 + y2) / 2) / img_height

            width = (x2 - x1) / img_width
            height = (y2 - y1) / img_height

            yolo_id = coco_to_yolo[category]

            label_lines.append(
                f"{yolo_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{width:.6f} "
                f"{height:.6f}"
            )

        # Save labels
        with open(label_path, "w") as f:
            f.write("\n".join(label_lines))

        total_saved_images += 1
        total_saved_boxes += len(label_lines)

# Create YOLO configuration
names = {
    idx: name
    for idx, name in enumerate(SELECTED_CLASSES.keys())
}

data_config = {
    "path": OUTPUT_DIR,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 25,
    "names": names
}

yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")

with open(yaml_path, "w") as f:
    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False
    )

print("========== YOLO DATASET CREATED ==========")
print("Destination:", OUTPUT_DIR)
print("Images saved:", total_saved_images)
print("Bounding boxes saved:", total_saved_boxes)
print("Skipped invalid boxes:", skipped_boxes)

for split in splits:
    print(f"{split} images:", len(splits[split]))

print("\nConfiguration saved:", yaml_path)
print("Number of classes:", data_config["nc"])
print("\nDataset generation completed!")

In [ ]:
# Validate the generated dataset

import os
from collections import Counter

DATASET_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed"

total_images = 0
total_labels = 0
total_boxes = 0
invalid_boxes = 0
missing_labels = 0
class_counts = Counter()

for split in ["train", "val", "test"]:

    image_dir = os.path.join(DATASET_DIR, "images", split)
    label_dir = os.path.join(DATASET_DIR, "labels", split)

    images = [
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    labels = [
        f for f in os.listdir(label_dir)
        if f.endswith(".txt")
    ]

    total_images += len(images)
    total_labels += len(labels)

    image_stems = {os.path.splitext(f)[0] for f in images}
    label_stems = {os.path.splitext(f)[0] for f in labels}

    missing_labels += len(image_stems - label_stems)

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))

    for filename in labels:

        with open(os.path.join(label_dir, filename)) as f:
            for line in f:

                parts = line.strip().split()

                if len(parts) != 5:
                    invalid_boxes += 1
                    continue

                class_id = int(parts[0])
                x, y, w, h = map(float, parts[1:])

                total_boxes += 1
                class_counts[class_id] += 1

                if not (
                    0 <= class_id < 25
                    and 0 <= x <= 1
                    and 0 <= y <= 1
                    and 0 < w <= 1
                    and 0 < h <= 1
                ):
                    invalid_boxes += 1

print("\n========== FINAL VALIDATION ==========")
print("Total images:", total_images)
print("Total label files:", total_labels)
print("Total bounding boxes:", total_boxes)
print("Missing labels:", missing_labels)
print("Invalid annotations:", invalid_boxes)
print("Classes found:", len(class_counts))

print("\n========== CLASS DISTRIBUTION ==========")
for class_id, count in sorted(class_counts.items()):
    print(f"Class {class_id}: {count}")

# Step 3.3: YOLOv8 Training and Fine-tuning.

In [ ]:
# Configure YOLOv8n for CPU training

from ultralytics import YOLO
import torch
import os

# Dataset configuration
DATA_YAML = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed/data.yaml"

# Google Drive output directory
PROJECT_DIR = "/content/drive/MyDrive/SmartVision_AI/results/yolo_training"

os.makedirs(PROJECT_DIR, exist_ok=True)

# Confirm CPU
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Load pretrained YOLOv8 Nano model
yolo_model = YOLO("yolov8n.pt")

print("\nYOLOv8n loaded successfully!")
print("Number of pretrained COCO classes:", len(yolo_model.names))
print("Custom dataset classes: 25")
print("Dataset configuration:", DATA_YAML)
print("Training output:", PROJECT_DIR)

In [ ]:
# YOLOv8 training

# YOLOv8 training on CPU

results = yolo_model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=416,
    batch=4,
    device="cpu",
    workers=2,
    patience=7,
    optimizer="AdamW",
    lr0=0.001,
    project=PROJECT_DIR,
    name="yolov8n_25class_cpu",
    exist_ok=False,
    pretrained=True,
    save=True,
    save_period=5,
    plots=True,
    cache=False,
    verbose=True
)

print("\nYOLOv8 training completed!")
print("Training results:", PROJECT_DIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BEST_MODEL = "/content/drive/MyDrive/SmartVision_AI/results/yolo_training/yolov8n_25class_cpu/weights/best.pt"
LAST_MODEL = "/content/drive/MyDrive/SmartVision_AI/results/yolo_training/yolov8n_25class_cpu/weights/last.pt"

print("Best model exists:", os.path.exists(BEST_MODEL))
print("Last model exists:", os.path.exists(LAST_MODEL))

if os.path.exists(BEST_MODEL):
    print("Best model size:", round(os.path.getsize(BEST_MODEL) / (1024 * 1024), 2), "MB")

In [ ]:
# Test Dataset Evaluation.

from ultralytics import YOLO
import os

# Load the trained best model
BEST_MODEL = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8n_25class_cpu/"
    "weights/best.pt"
)

DATA_YAML = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/data.yaml"
)

best_model = YOLO(BEST_MODEL)

# Evaluate on the separate test dataset
test_results = best_model.val(
    data=DATA_YAML,
    split="test",
    imgsz=416,
    batch=4,
    device="cpu",
    project="/content/drive/MyDrive/SmartVision_AI/results/yolo_training",
    name="yolov8n_test_evaluation",
    plots=True
)

print("\n========== TEST RESULTS ==========")
print("Precision:", test_results.box.mp)
print("Recall:", test_results.box.mr)
print("mAP@0.5:", test_results.box.map50)
print("mAP@0.5:0.95:", test_results.box.map)

In [ ]:
# Visualise object detection predictions

import os
import glob
import random
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

BEST_MODEL = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8n_25class_cpu/"
    "weights/best.pt"
)

TEST_IMAGES = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/images/test"
)

# Load trained model
model = YOLO(BEST_MODEL)

# Get test images
image_paths = glob.glob(os.path.join(TEST_IMAGES, "*"))

# Select 5 random test images
random.seed(42)
selected_images = random.sample(image_paths, min(5, len(image_paths)))

# Generate predictions
for image_path in selected_images:

    results = model.predict(
        source=image_path,
        imgsz=416,
        conf=0.25,
        device="cpu",
        verbose=False
    )

    # Display prediction
    plotted_image = results[0].plot()

    plt.figure(figsize=(10, 7))
    plt.imshow(
        plotted_image[:, :, ::-1]
    )
    plt.title(os.path.basename(image_path))
    plt.axis("off")
    plt.show()

In [ ]:
# Analyse failure cases

# we should save examples of incorrect predictions and examine them systematically.

# Let's first generate predictions with a higher confidence threshold of 0.50. This will help us inspect stronger detections and reduce some low-confidence boxes.

import os
import glob
from ultralytics import YOLO

BEST_MODEL = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8n_25class_cpu/"
    "weights/best.pt"
)

TEST_IMAGES = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/images/test"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/failure_analysis"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

model = YOLO(BEST_MODEL)

image_paths = glob.glob(os.path.join(TEST_IMAGES, "*"))

results = model.predict(
    source=image_paths,
    imgsz=416,
    conf=0.50,
    device="cpu",
    save=True,
    project=OUTPUT_DIR,
    name="high_confidence_predictions",
    exist_ok=True,
    verbose=False
)

print("Total test images:", len(image_paths))
print("Predictions generated:", len(results))
print("Saved predictions to:", OUTPUT_DIR)

In [ ]:
# Examine the evaluation plots

import os
from IPython.display import display, Image

RESULTS_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8n_test_evaluation"
)

print("Available evaluation files:\n")

for file in sorted(os.listdir(RESULTS_DIR)):
    print(file)

print("\n========== EVALUATION PLOTS ==========")

plots = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png"
]

for plot in plots:
    path = os.path.join(RESULTS_DIR, plot)

    if os.path.exists(path):
        print("\n", plot)
        display(Image(filename=path))

In [ ]:
# Precision–Recall and F1 curves

import os
from IPython.display import display, Image

RESULTS_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8n_test_evaluation"
)

plots = [
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png"
]

for plot in plots:
    path = os.path.join(RESULTS_DIR, plot)

    if os.path.exists(path):
        print("\n==========", plot, "==========")
        display(Image(filename=path))
    else:
        print("Not found:", plot)

No problem. This doesn't indicate a model error. The plots may have different filenames or may not have been generated.


In [ ]:
import os

RESULTS_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8n_test_evaluation"
)

print("========== SAVED EVALUATION FILES ==========")

for root, dirs, files in os.walk(RESULTS_DIR):
    for file in sorted(files):
        print(os.path.join(root, file))

In [ ]:
# Display all four evaluation graphs

import os
from IPython.display import display, Image

RESULTS_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8n_test_evaluation"
)

plots = [
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png"
]

for plot in plots:
    path = os.path.join(RESULTS_DIR, plot)

    print("\n==========", plot, "==========")
    display(Image(filename=path))

The Recall–Confidence curve illustrates the relationship between the detection confidence threshold and recall. The model achieves higher recall at lower confidence thresholds. As the threshold increases, recall gradually decreases because fewer predictions are accepted. This indicates that confidence threshold selection plays an important role in balancing object detection performance.

The detection dataset currently contains 991 unique images across 25 classes, with an uneven distribution. That is a significant limitation.

# improve ResNet50 first, because it is currently your best classifier. We can then improve YOLOv8 separately.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check your saved classification models

import os

MODEL_DIR = (
    "/content/drive/MyDrive/"
    "SmartVision_AI/saved_models"
)

print("========== SAVED CLASSIFICATION MODELS ==========")

if os.path.exists(MODEL_DIR):
    for file in sorted(os.listdir(MODEL_DIR)):
        path = os.path.join(MODEL_DIR, file)

        if os.path.isfile(path):
            size = os.path.getsize(path) / (1024 * 1024)
            print(f"{file}  |  {size:.2f} MB")
else:
    print("Model directory not found!")

In [ ]:
# Check the classification dataset

import os

BASE_DIR = "/content/drive/MyDrive/SmartVision_AI"

print("========== SMARTVISION AI DATASET ==========")

for root, dirs, files in os.walk(BASE_DIR):
    dirs[:] = [d for d in dirs if d not in ["results", "saved_models"]]

    level = root.replace(BASE_DIR, "").count(os.sep)

    if level > 3:
        dirs[:] = []
        continue

    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:5]:
        print(f"{indent}  {file}")

print("\nDataset inspection completed.")

In [ ]:
# Audit classification dataset distribution

import os
import pandas as pd

DATASET_DIR = (
    "/content/drive/MyDrive/"
    "SmartVision_AI/dataset/classification"
)

splits = ["train", "val", "test"]
class_counts = {}

for split in splits:
    split_dir = os.path.join(DATASET_DIR, split)
    class_counts[split] = {}

    for class_name in sorted(os.listdir(split_dir)):
        class_path = os.path.join(split_dir, class_name)

        if os.path.isdir(class_path):
            images = [
                f for f in os.listdir(class_path)
                if f.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".bmp")
                )
            ]

            class_counts[split][class_name] = len(images)

df = pd.DataFrame(class_counts).fillna(0).astype(int)
df["Total"] = df.sum(axis=1)

print("========== CLASS DISTRIBUTION ==========")
display(df)

print("\n========== TOTAL IMAGES ==========")
print(df.sum())
print("\nNumber of classes:", len(df))

classification dataset is significantly imbalanced.

In [ ]:
# First, verify image quality

import os
from PIL import Image
import pandas as pd

DATASET_DIR = (
    "/content/drive/MyDrive/"
    "SmartVision_AI/dataset/classification"
)

splits = ["train", "val", "test"]

audit_results = []

for split in splits:

    split_dir = os.path.join(DATASET_DIR, split)

    for class_name in sorted(os.listdir(split_dir)):

        class_dir = os.path.join(split_dir, class_name)

        if not os.path.isdir(class_dir):
            continue

        for filename in os.listdir(class_dir):

            if not filename.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            ):
                continue

            path = os.path.join(class_dir, filename)

            try:
                with Image.open(path) as img:
                    width, height = img.size

                    audit_results.append({
                        "split": split,
                        "class": class_name,
                        "filename": filename,
                        "width": width,
                        "height": height,
                        "valid": True
                    })

            except Exception:
                audit_results.append({
                    "split": split,
                    "class": class_name,
                    "filename": filename,
                    "width": None,
                    "height": None,
                    "valid": False
                })

audit_df = pd.DataFrame(audit_results)

print("========== IMAGE QUALITY AUDIT ==========")
print("Total images checked:", len(audit_df))
print("Valid images:", audit_df["valid"].sum())
print("Corrupted images:", (~audit_df["valid"]).sum())

print("\nImage dimension statistics:")
print(audit_df[audit_df["valid"]][["width", "height"]].describe())

print("\nSmall images (below 64x64):")
small = audit_df[
    (audit_df["valid"]) &
    ((audit_df["width"] < 64) | (audit_df["height"] < 64))
]

print("Count:", len(small))

In [ ]:
# Check duplicate images

import os
import hashlib
from collections import defaultdict

DATASET_DIR = (
    "/content/drive/MyDrive/"
    "SmartVision_AI/dataset/classification"
)

splits = ["train", "val", "test"]

hash_records = defaultdict(list)

for split in splits:

    split_dir = os.path.join(DATASET_DIR, split)

    for class_name in os.listdir(split_dir):

        class_dir = os.path.join(split_dir, class_name)

        if not os.path.isdir(class_dir):
            continue

        for filename in os.listdir(class_dir):

            if not filename.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            ):
                continue

            path = os.path.join(class_dir, filename)

            with open(path, "rb") as f:
                file_hash = hashlib.sha256(f.read()).hexdigest()

            hash_records[file_hash].append({
                "split": split,
                "class": class_name,
                "filename": filename
            })

duplicates = {
    h: records
    for h, records in hash_records.items()
    if len(records) > 1
}

cross_split_duplicates = {
    h: records
    for h, records in duplicates.items()
    if len(set(r["split"] for r in records)) > 1
}

print("========== DUPLICATE IMAGE AUDIT ==========")
print("Unique image hashes:", len(hash_records))
print("Duplicate image groups:", len(duplicates))
print("Cross-split duplicate groups:", len(cross_split_duplicates))

print("\nSample cross-split duplicates:")

for records in list(cross_split_duplicates.values())[:10]:
    print(records)

In [ ]:
# Investigate class-label quality

import os
import random
import matplotlib.pyplot as plt
from PIL import Image

DATASET_DIR = (
    "/content/drive/MyDrive/"
    "SmartVision_AI/dataset/classification/train"
)

random.seed(42)

classes = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])

fig, axes = plt.subplots(5, 5, figsize=(16, 16))

for ax, class_name in zip(axes.flat, classes):

    class_dir = os.path.join(DATASET_DIR, class_name)

    images = [
        f for f in os.listdir(class_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    selected = random.choice(images)
    image_path = os.path.join(class_dir, selected)

    img = Image.open(image_path).convert("RGB")

    ax.imshow(img)
    ax.set_title(class_name)
    ax.axis("off")

plt.suptitle("SmartVision AI – Classification Dataset Sample", fontsize=16)
plt.tight_layout()
plt.show()

Bird image: large horizontal black regions.
Bench image: very narrow visible content.
Bowl image: only a small portion of the original image is visible.

In [ ]:
# Inspect more images before training

import os
import random
import matplotlib.pyplot as plt
from PIL import Image

DATASET_DIR = (
    "/content/drive/MyDrive/"
    "SmartVision_AI/dataset/classification/train"
)

random.seed(100)

classes = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])

fig, axes = plt.subplots(25, 5, figsize=(15, 55))

for row, class_name in enumerate(classes):

    class_dir = os.path.join(DATASET_DIR, class_name)

    images = [
        f for f in os.listdir(class_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    selected = random.sample(images, min(5, len(images)))

    for col, filename in enumerate(selected):

        image_path = os.path.join(class_dir, filename)

        img = Image.open(image_path).convert("RGB")

        axes[row, col].imshow(img)
        axes[row, col].axis("off")

        if col == 0:
            axes[row, col].set_ylabel(
                class_name,
                fontsize=11
            )

plt.suptitle(
    "SmartVision AI – Detailed Dataset Inspection",
    fontsize=18
)

plt.tight_layout()
plt.show()

In [ ]:
# First, measure black padding

import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

DATASET_DIR = (
    "/content/drive/MyDrive/"
    "SmartVision_AI/dataset/classification"
)

splits = ["train", "val", "test"]

results = []

for split in splits:

    split_dir = os.path.join(DATASET_DIR, split)

    for class_name in sorted(os.listdir(split_dir)):

        class_dir = os.path.join(split_dir, class_name)

        if not os.path.isdir(class_dir):
            continue

        for filename in os.listdir(class_dir):

            if not filename.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):
                continue

            image_path = os.path.join(class_dir, filename)

            try:
                img = np.array(
                    Image.open(image_path).convert("RGB")
                )

                # Identify nearly black pixels
                black_pixels = np.all(img <= 10, axis=2)

                black_percentage = (
                    black_pixels.mean() * 100
                )

                results.append({
                    "split": split,
                    "class": class_name,
                    "filename": filename,
                    "black_percentage": black_percentage
                })

            except Exception as e:
                print("Error:", image_path, e)

df_padding = pd.DataFrame(results)

print("========== BLACK PADDING AUDIT ==========")
print("Total images:", len(df_padding))

print("\nBlack padding distribution:")
print(df_padding["black_percentage"].describe())

print("\nImages with significant black regions:")

for threshold in [10, 20, 30, 40, 50]:

    count = (
        df_padding["black_percentage"] > threshold
    ).sum()

    print(
        f"Above {threshold}% black pixels: "
        f"{count} images"
    )

print("\nClass-wise average black percentage:")

print(
    df_padding.groupby("class")["black_percentage"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

In [ ]:
# next audit

import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

DATASET_DIR = (
    "/content/drive/MyDrive/"
    "SmartVision_AI/dataset/classification"
)

results = []

for split in ["train", "val", "test"]:

    split_dir = os.path.join(DATASET_DIR, split)

    for class_name in sorted(os.listdir(split_dir)):

        class_dir = os.path.join(split_dir, class_name)

        if not os.path.isdir(class_dir):
            continue

        for filename in os.listdir(class_dir):

            if not filename.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):
                continue

            path = os.path.join(class_dir, filename)

            try:
                img = np.array(
                    Image.open(path).convert("RGB")
                )

                # Consider a pixel black if all channels <= 10
                black = np.all(img <= 10, axis=2)

                # Percentage of black pixels in each row/column
                row_black = black.mean(axis=1)
                col_black = black.mean(axis=0)

                # Detect almost completely black border rows/columns
                top = 0
                bottom = img.shape[0] - 1
                left = 0
                right = img.shape[1] - 1

                while top < bottom and row_black[top] > 0.95:
                    top += 1

                while bottom > top and row_black[bottom] > 0.95:
                    bottom -= 1

                while left < right and col_black[left] > 0.95:
                    left += 1

                while right > left and col_black[right] > 0.95:
                    right -= 1

                crop_width = right - left + 1
                crop_height = bottom - top + 1

                retained_area = (
                    crop_width * crop_height
                ) / (img.shape[0] * img.shape[1])

                results.append({
                    "split": split,
                    "class": class_name,
                    "filename": filename,
                    "top_border": top,
                    "bottom_border": img.shape[0] - 1 - bottom,
                    "left_border": left,
                    "right_border": img.shape[1] - 1 - right,
                    "retained_area_percent": retained_area * 100
                })

            except Exception as e:
                print("Error:", path, e)

df_borders = pd.DataFrame(results)

print("========== BLACK BORDER AUDIT ==========")
print("Total images:", len(df_borders))

print("\nRetained image area:")
print(df_borders["retained_area_percent"].describe().round(2))

print("\nImages with detected border padding:")

for threshold in [90, 80, 70, 60]:

    count = (
        df_borders["retained_area_percent"] < threshold
    ).sum()

    print(
        f"Less than {threshold}% retained area: {count}"
    )

print("\nClass-wise average retained area:")

print(
    df_borders.groupby("class")["retained_area_percent"]
    .mean()
    .sort_values()
    .round(2)
)

In [ ]:
# visually verify the proposed crops

import os
import matplotlib.pyplot as plt
from PIL import Image

# Select images with the smallest retained area
sample_images = df_borders.sort_values(
    "retained_area_percent"
).head(4)

# Add four images with moderate padding
moderate_samples = df_borders[
    (df_borders["retained_area_percent"] >= 50) &
    (df_borders["retained_area_percent"] <= 70)
].sample(
    n=min(4, len(df_borders[
        (df_borders["retained_area_percent"] >= 50) &
        (df_borders["retained_area_percent"] <= 70)
    ])),
    random_state=42
)

samples = pd.concat([sample_images, moderate_samples])

fig, axes = plt.subplots(
    len(samples), 2,
    figsize=(10, len(samples) * 3)
)

for i, (_, row) in enumerate(samples.iterrows()):

    image_path = os.path.join(
        DATASET_DIR,
        row["split"],
        row["class"],
        row["filename"]
    )

    img = Image.open(image_path).convert("RGB")

    left = int(row["left_border"])
    right = img.width - int(row["right_border"])

    top = int(row["top_border"])
    bottom = img.height - int(row["bottom_border"])

    cropped = img.crop((left, top, right, bottom))

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(
        f"Original: {row['class']}"
    )

    axes[i, 1].imshow(cropped)
    axes[i, 1].set_title(
        f"Proposed crop: {row['retained_area_percent']:.1f}% retained"
    )

    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# identify severely affected images

print("========== SEVERELY PADDED IMAGE AUDIT ==========")

thresholds = [5, 10, 20, 30, 40, 50]

for threshold in thresholds:

    count = (
        df_borders["retained_area_percent"] < threshold
    ).sum()

    print(
        f"Retained area below {threshold}%: {count}"
    )

print("\n========== CLASS-WISE SEVERE IMAGES ==========")

severe_images = df_borders[
    df_borders["retained_area_percent"] < 30
]

summary = (
    severe_images.groupby("class")
    .size()
    .rename("severe_images")
    .to_frame()
)

summary["total_images"] = (
    df_borders.groupby("class").size()
)

summary["percentage"] = (
    summary["severe_images"] /
    summary["total_images"] * 100
).round(2)

print(summary.sort_values(
    "percentage",
    ascending=False
))

In [ ]:
# create a visual review collection

import os
import math
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

AUDIT_DIR = (
    "/content/drive/MyDrive/"
    "SmartVision_AI/dataset_quality_audit"
)

os.makedirs(AUDIT_DIR, exist_ok=True)

# Select images retaining less than 10% area
severe = df_borders[
    df_borders["retained_area_percent"] < 10
].copy()

# Save the complete audit list
csv_path = os.path.join(
    AUDIT_DIR,
    "severely_padded_images.csv"
)

severe.to_csv(csv_path, index=False)

# Generate contact sheets: 20 images per sheet
images_per_sheet = 20
total_sheets = math.ceil(len(severe) / images_per_sheet)

for sheet_num in range(total_sheets):

    batch = severe.iloc[
        sheet_num * images_per_sheet:
        (sheet_num + 1) * images_per_sheet
    ]

    fig, axes = plt.subplots(
        4, 5,
        figsize=(16, 12)
    )

    axes = axes.flatten()

    for ax, (_, row) in zip(axes, batch.iterrows()):

        image_path = os.path.join(
            DATASET_DIR,
            row["split"],
            row["class"],
            row["filename"]
        )

        img = Image.open(image_path).convert("RGB")

        ax.imshow(img)
        ax.set_title(
            f"{row['class']}\n"
            f"Retained: {row['retained_area_percent']:.1f}%",
            fontsize=9
        )
        ax.axis("off")

    for ax in axes[len(batch):]:
        ax.axis("off")

    plt.suptitle(
        f"Severely Padded Images - Sheet {sheet_num + 1}",
        fontsize=16
    )

    plt.tight_layout()

    output_path = os.path.join(
        AUDIT_DIR,
        f"severe_images_sheet_{sheet_num + 1}.png"
    )

    plt.savefig(
        output_path,
        dpi=120,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

print("========== REVIEW COLLECTION SAVED ==========")
print("Images:", len(severe))
print("Contact sheets:", total_sheets)
print("CSV:", csv_path)
print("Folder:", AUDIT_DIR)

In [ ]:
# check how many images will be affected

# Count severely padded images by split and class

severe_counts = (
    severe.groupby(["split", "class"])
    .size()
    .reset_index(name="images_to_review")
)

print("========== SEVERE IMAGE DISTRIBUTION ==========")
print(severe_counts.to_string(index=False))

print("\nImages to exclude initially:", len(severe))
print("\nDistribution by split:")
print(severe.groupby("split").size())

print("\nDistribution by class:")
print(severe.groupby("class").size().sort_values(ascending=False))


In [ ]:
# inspect the audit data

print("========== SEVERE DATAFRAME DETAILS ==========")

print("Columns:")
print(severe.columns.tolist())

print("\nFirst 5 records:")
display(severe.head())

print("\nData types:")
print(severe.dtypes)

In [ ]:
# Create the cleaned dataset

import os
import shutil
from pathlib import Path

# Original and cleaned dataset paths
SOURCE_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/dataset/classification"
)

CLEAN_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/dataset/classification_cleaned_v1"
)

# Safety checks
if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Original dataset not found: {SOURCE_DIR}")

if CLEAN_DIR.exists():
    raise FileExistsError(
        f"Cleaned dataset already exists: {CLEAN_DIR}. "
        "Stopping to avoid overwriting files."
    )

# Identify only severely padded training images
severe_train = severe[severe["split"] == "train"]

files_to_exclude = set(
    zip(severe_train["class"], severe_train["filename"])
)

print("Training images to exclude:", len(files_to_exclude))

# Copy dataset while excluding selected training images
copied = 0
excluded = 0

for split in ["train", "val", "test"]:

    split_source = SOURCE_DIR / split
    split_destination = CLEAN_DIR / split

    for class_dir in split_source.iterdir():

        if not class_dir.is_dir():
            continue

        destination_class = split_destination / class_dir.name
        destination_class.mkdir(parents=True, exist_ok=True)

        for image_path in class_dir.iterdir():

            if not image_path.is_file():
                continue

            if split == "train" and (
                class_dir.name, image_path.name
            ) in files_to_exclude:
                excluded += 1
                continue

            shutil.copy2(
                image_path,
                destination_class / image_path.name
            )

            copied += 1

print("\n========== CLEANED DATASET CREATED ==========")
print("Images copied:", copied)
print("Training images excluded:", excluded)
print("Location:", CLEAN_DIR)

In [ ]:
# verify the cleaned dataset

from pathlib import Path

CLEAN_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/dataset/classification_cleaned_v1"
)

print("========== CLEANED DATASET VERIFICATION ==========")

total = 0

for split in ["train", "val", "test"]:

    split_dir = CLEAN_DIR / split
    split_total = 0

    print(f"\n{split.upper()}")

    for class_dir in sorted(split_dir.iterdir()):

        if not class_dir.is_dir():
            continue

        count = len([
            f for f in class_dir.iterdir()
            if f.is_file()
        ])

        split_total += count
        print(f"{class_dir.name}: {count}")

    total += split_total
    print("Split total:", split_total)

print("\nTOTAL CLEANED IMAGES:", total)
print("EXPECTED TOTAL: 5689")
print("VERIFICATION:", "PASSED" if total == 5689 else "FAILED")

In [ ]:
# inspect the existing ResNet50 model

import tensorflow as tf
import os

MODEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_finetuned_best.keras"
)

print("TensorFlow version:", tf.__version__)

# Load existing trained model
original_model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False,
    custom_objects={
        "color_jitter": color_jitter,
        "preprocess_input":
            tf.keras.applications.resnet50.preprocess_input
    }
)

print("\n========== ORIGINAL RESNET50 ==========")

print("Model name:", original_model.name)
print("Input shape:", original_model.input_shape)
print("Output shape:", original_model.output_shape)
print("Total parameters:", original_model.count_params())

print("\n========== MODEL LAYERS ==========")

for layer in original_model.layers:
    print(
        layer.name,
        "|",
        layer.__class__.__name__,
        "| Trainable:", layer.trainable
    )

In [ ]:
# Inspect the original model configuration

print("========== DATA AUGMENTATION CONFIGURATION ==========")

augmentation_layer = original_model.get_layer("data_augmentation")

for layer in augmentation_layer.layers:
    print(f"\nLayer: {layer.name}")
    print(f"Type: {layer.__class__.__name__}")
    print("Config:", layer.get_config())


print("\n========== RESNET50 FINE-TUNING CONFIGURATION ==========")

base_model = original_model.get_layer("resnet50")

print("Total ResNet50 layers:", len(base_model.layers))
print("Trainable ResNet50 layers:",
      sum(layer.trainable for layer in base_model.layers))

print("\nLast 25 ResNet50 layers:")

for layer in base_model.layers[-25:]:
    print(
        layer.name,
        "|",
        layer.__class__.__name__,
        "| Trainable:",
        layer.trainable
    )

In [ ]:
# Check file sizes and contents

import os

HISTORY_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/training_history"
)

for filename in sorted(os.listdir(HISTORY_DIR)):
    if filename.endswith(".json"):
        file_path = os.path.join(HISTORY_DIR, filename)
        size = os.path.getsize(file_path)

        print(f"\nFile: {filename}")
        print(f"Size: {size} bytes")

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read(500)

        print("Content preview:", repr(content))

In [ ]:
# Extract the complete ResNet50 training history

import os
import json
import numpy as np

HISTORY_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/training_history"
)

files = [
    "resnet50_augmented_history.json",
    "resnet50_finetuned_history.json"
]

for filename in files:

    path = os.path.join(HISTORY_DIR, filename)

    with open(path, "r") as f:
        history = json.load(f)

    val_acc = history.get("val_accuracy", [])
    train_acc = history.get("accuracy", [])

    print("\n" + "=" * 55)
    print(filename)
    print("=" * 55)

    if not val_acc:
        print("Validation accuracy not available.")
        continue

    best_epoch = int(np.argmax(val_acc))

    print("Epochs completed:", len(val_acc))
    print("Best validation epoch:", best_epoch + 1)
    print("Best validation accuracy:",
          round(val_acc[best_epoch] * 100, 2), "%")
    print("Training accuracy at best epoch:",
          round(train_acc[best_epoch] * 100, 2), "%")
    print("Final validation accuracy:",
          round(val_acc[-1] * 100, 2), "%")
    print("Final training accuracy:",
          round(train_acc[-1] * 100, 2), "%")

In [ ]:
# First, verify the cleaned dataset

import os

CLEAN_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/classification_cleaned_v1"
)

ORIGINAL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/classification"
)

for split in ["train", "val", "test"]:

    original_count = sum(
        len(files)
        for _, _, files in os.walk(os.path.join(ORIGINAL_DIR, split))
    )

    cleaned_count = sum(
        len(files)
        for _, _, files in os.walk(os.path.join(CLEAN_DIR, split))
    )

    print(f"\n{split.upper()}")
    print("Original images:", original_count)
    print("Cleaned images:", cleaned_count)
    print("Difference:", original_count - cleaned_count)

In [ ]:
# Prepare the training environment

print("========== CURRENT DATASET CONFIGURATION ==========")

for variable in [
    "BATCH_SIZE",
    "IMG_SIZE",
    "train_dataset",
    "val_dataset",
    "test_dataset",
    "class_names",
    "data_augmentation"
]:
    if variable in globals():
        value = globals()[variable]

        if variable.endswith("_dataset"):
            print(variable, "exists")
            print("Element spec:", value.element_spec)

        elif variable == "data_augmentation":
            print(variable, "exists")
            print("Type:", type(value).__name__)

        else:
            print(variable, "=", value)
    else:
        print(variable, "not found")

In [ ]:
# Recreate the dataset configuration

import tensorflow as tf
import os

ORIGINAL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/classification"
)

CLEAN_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/classification_cleaned_v1"
)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

class_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'truck', 'traffic light', 'stop sign', 'bench',
    'bird', 'cat', 'dog', 'horse', 'cow', 'elephant',
    'bottle', 'cup', 'bowl', 'pizza', 'cake', 'chair',
    'couch', 'potted plant', 'bed'
]

print("TensorFlow:", tf.__version__)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", len(class_names))

# Verify that folder names match the original model's class order
folder_classes = sorted(
    os.listdir(os.path.join(CLEAN_DIR, "train"))
)

print("\nAlphabetical folder order:")
print(folder_classes)

print("\nOriginal model class order:")
print(class_names)

assert set(folder_classes) == set(class_names), \
    "Dataset folders do not match the model classes!"

print("\nDataset class verification passed.")

In [ ]:
# Load the cleaned training and validation datasets

# Load cleaned training dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLEAN_DIR, "train"),
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

# Load original validation dataset
val_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(ORIGINAL_DIR, "val"),
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

print("\nTraining batches:",
      tf.data.experimental.cardinality(train_dataset).numpy())

print("Validation batches:",
      tf.data.experimental.cardinality(val_dataset).numpy())

print("\nClass mapping:")
print(train_dataset.class_names)

In [ ]:
# Verify image and label formats

for images, labels in train_dataset.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

    print("Image minimum:", tf.reduce_min(images).numpy())
    print("Image maximum:", tf.reduce_max(images).numpy())

    print("Number of classes:", labels.shape[-1])
    print("First image label index:",
          tf.argmax(labels[0]).numpy())

    print("First image class:",
          class_names[tf.argmax(labels[0]).numpy()])

In [ ]:
# Prepare the original ResNet50 model for the cleaned experiment

import tensorflow as tf

# Load a fresh copy of the original ResNet50 model
cleaned_resnet = tf.keras.models.load_model(
    "/content/drive/MyDrive/SmartVision_AI/saved_models/resnet50_finetuned_best.keras",
    compile=False,
    custom_objects={
        "preprocess_input":
            tf.keras.applications.resnet50.preprocess_input
    },
    safe_mode=False
)

print("Model loaded successfully!")
print("Model name:", cleaned_resnet.name)
print("Input shape:", cleaned_resnet.input_shape)
print("Output shape:", cleaned_resnet.output_shape)

# Check augmentation layer
print("\nAugmentation layer:",
      cleaned_resnet.get_layer("data_augmentation").name)

# Check backbone
backbone = cleaned_resnet.get_layer("resnet50")

print("Backbone:", backbone.name)
print("Total backbone layers:", len(backbone.layers))
print("Currently trainable backbone layers:",
      sum(layer.trainable for layer in backbone.layers))

In [ ]:
# Freeze the ResNet50 backbone

# Freeze the ResNet50 backbone
backbone = cleaned_resnet.get_layer("resnet50")
backbone.trainable = False

# Verify trainable layers
print("Backbone trainable:", backbone.trainable)

print("Trainable model layers:")

for layer in cleaned_resnet.layers:
    print(layer.name, "| Trainable:", layer.trainable)

# Compile model
cleaned_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nModel compiled successfully!")
print("Total parameters:", cleaned_resnet.count_params())
print(
    "Trainable parameters:",
    sum(
        tf.keras.backend.count_params(w)
        for w in cleaned_resnet.trainable_weights
    )
)

In [ ]:
# Configure callbacks and save paths

import os
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

MODEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models"
)

os.makedirs(MODEL_DIR, exist_ok=True)

STAGE1_PATH = os.path.join(
    MODEL_DIR,
    "resnet50_cleaned_v1_augmented_best.keras"
)

callbacks_stage1 = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=STAGE1_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
]

print("Callbacks configured successfully.")
print("Checkpoint path:", STAGE1_PATH)
print("Existing checkpoint:", os.path.exists(STAGE1_PATH))

In [ ]:
# Start Stage 1 training

history_cleaned_stage1 = cleaned_resnet.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=callbacks_stage1
)

In [ ]:
# drive connect

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf

@tf.keras.utils.register_keras_serializable()
def color_jitter(images):

    images = images / 255.0

    images = tf.image.random_hue(
        images,
        max_delta=0.05
    )

    images = tf.image.random_saturation(
        images,
        lower=0.8,
        upper=1.2
    )

    images = tf.clip_by_value(
        images,
        0.0,
        1.0
    )

    images = images * 255.0

    return images

print("Original color_jitter function registered!")

In [ ]:
# Load the saved Stage 1 model
import tensorflow as tf
import os

STAGE1_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_cleaned_v1_augmented_best.keras"
)

print("Checkpoint exists:", os.path.exists(STAGE1_PATH))

cleaned_resnet = tf.keras.models.load_model(
    STAGE1_PATH,
    compile=False,
    custom_objects={
        "preprocess_input":
            tf.keras.applications.resnet50.preprocess_input
    },
    safe_mode=False
)

print("\nStage 1 model restored successfully!")
print("Model:", cleaned_resnet.name)
print("Output:", cleaned_resnet.output_shape)

In [ ]:
# Prepare ResNet50 fine-tuning

# Get the ResNet50 backbone
backbone = cleaned_resnet.get_layer("resnet50")

# Unfreeze the backbone
backbone.trainable = True

# Freeze all layers except the final 20
for layer in backbone.layers[:-20]:
    layer.trainable = False

for layer in backbone.layers[-20:]:
    layer.trainable = True

# Compile with a lower learning rate
cleaned_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Fine-tuning model prepared!")
print(
    "Trainable backbone layers:",
    sum(layer.trainable for layer in backbone.layers)
)

print(
    "Trainable parameters:",
    sum(
        tf.keras.backend.count_params(w)
        for w in cleaned_resnet.trainable_weights
    )
)

In [ ]:
# Restore the datasets

import os
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

class_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'truck', 'traffic light', 'stop sign', 'bench',
    'bird', 'cat', 'dog', 'horse', 'cow', 'elephant',
    'bottle', 'cup', 'bowl', 'pizza', 'cake', 'chair',
    'couch', 'potted plant', 'bed'
]

ORIGINAL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/classification"
)

CLEAN_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/classification_cleaned_v1"
)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLEAN_DIR, "train"),
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(ORIGINAL_DIR, "val"),
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

print("\nTraining batches:",
      tf.data.experimental.cardinality(train_dataset).numpy())

print("Validation batches:",
      tf.data.experimental.cardinality(val_dataset).numpy())

In [ ]:
# Configure Stage 2 callbacks

import os

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

STAGE2_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_cleaned_v1_finetuned_best.keras"
)

callbacks_stage2 = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=STAGE2_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
]

print("Stage 2 callbacks configured!")
print("Checkpoint:", STAGE2_PATH)
print("Already exists:", os.path.exists(STAGE2_PATH))

In [ ]:
# Start ResNet50 fine-tuning

history_cleaned_stage2 = cleaned_resnet.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=callbacks_stage2
)

In [ ]:
import os

STAGE2_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_cleaned_v1_finetuned_best.keras"
)

print("Checkpoint exists:", os.path.exists(STAGE2_PATH))

if os.path.exists(STAGE2_PATH):
    size = os.path.getsize(STAGE2_PATH) / (1024 * 1024)
    print("Checkpoint size:", round(size, 2), "MB")
    print("Checkpoint is available in Google Drive.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

MODEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models"
)

print("Folder exists:", os.path.exists(MODEL_DIR))

if os.path.exists(MODEL_DIR):
    print("\nAvailable model files:")

    for filename in sorted(os.listdir(MODEL_DIR)):
        path = os.path.join(MODEL_DIR, filename)

        if os.path.isfile(path):
            size = os.path.getsize(path) / (1024 * 1024)
            print(f"{filename} — {size:.2f} MB")

In [ ]:
import os

STAGE2_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_cleaned_v1_finetuned_best.keras"
)

print("Stage 2 checkpoint exists:", os.path.exists(STAGE2_PATH))
print(
    "Checkpoint size:",
    round(os.path.getsize(STAGE2_PATH) / (1024 * 1024), 2),
    "MB"
)

In [ ]:
# Register color_jitter and load Stage 2

import tensorflow as tf
import os

@tf.keras.utils.register_keras_serializable()
def color_jitter(images):
    images = images / 255.0
    images = tf.image.random_hue(images, max_delta=0.05)
    images = tf.image.random_saturation(
        images, lower=0.8, upper=1.2
    )
    images = tf.clip_by_value(images, 0.0, 1.0)
    return images * 255.0


STAGE2_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_cleaned_v1_finetuned_best.keras"
)

resumed_resnet = tf.keras.models.load_model(
    STAGE2_PATH,
    compile=False,
    custom_objects={
        "color_jitter": color_jitter,
        "Custom>color_jitter": color_jitter,
        "preprocess_input":
            tf.keras.applications.resnet50.preprocess_input
    },
    safe_mode=False
)

print("Stage 2 model loaded successfully!")
print("Output shape:", resumed_resnet.output_shape)

In [ ]:
# Verify the fine-tuning configuration

backbone = resumed_resnet.get_layer("resnet50")

print("Total ResNet50 layers:", len(backbone.layers))

print(
    "Trainable backbone layers:",
    sum(layer.trainable for layer in backbone.layers)
)

print(
    "Trainable parameters:",
    sum(
        tf.keras.backend.count_params(w)
        for w in resumed_resnet.trainable_weights
    )
)

In [ ]:
# Configure resumed fine-tuning

backbone = resumed_resnet.get_layer("resnet50")

backbone.trainable = True

for layer in backbone.layers[:-20]:
    layer.trainable = False

for layer in backbone.layers[-20:]:
    layer.trainable = True

resumed_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.000005
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Fine-tuning configuration restored.")
print(
    "Trainable backbone layers:",
    sum(layer.trainable for layer in backbone.layers)
)
print(
    "Trainable parameters:",
    sum(
        tf.keras.backend.count_params(w)
        for w in resumed_resnet.trainable_weights
    )
)

In [ ]:
# Restore training and validation datasets

import os
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

class_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'truck', 'traffic light', 'stop sign', 'bench',
    'bird', 'cat', 'dog', 'horse', 'cow', 'elephant',
    'bottle', 'cup', 'bowl', 'pizza', 'cake', 'chair',
    'couch', 'potted plant', 'bed'
]

ORIGINAL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/classification"
)

CLEAN_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/classification_cleaned_v1"
)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLEAN_DIR, "train"),
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(ORIGINAL_DIR, "val"),
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

print("Training batches:",
      tf.data.experimental.cardinality(train_dataset).numpy())

print("Validation batches:",
      tf.data.experimental.cardinality(val_dataset).numpy())

In [ ]:
# Configure resumed training

import os

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

RESUME_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_cleaned_v1_resumed_best.keras"
)

callbacks_resume = [
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=RESUME_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
]

print("Resume checkpoint:", RESUME_PATH)
print("Already exists:", os.path.exists(RESUME_PATH))

In [ ]:
# Start resumed fine-tuning

history_cleaned_resume = resumed_resnet.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=8,
    callbacks=callbacks_resume
)

Best result: 60.07%
Your resumed model achieved:
- Training accuracy: 75.49% at Epoch 4
- Validation accuracy: 60.07%
- Validation loss: 1.4898
- Best checkpoint saved at Epoch 4.
The training stopped early at Epoch 6. This is expected because of the early-stopping callback.

Important: The callback restored weights from Epoch 2 because it monitors val_loss, while the checkpoint saved the best val_accuracy model from Epoch 4. For final evaluation, we should load the checkpoint file, not use the in-memory model.

In [ ]:
# Evaluate on the test dataset

# Load the best resumed model

RESUME_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_cleaned_v1_resumed_best.keras"
)

final_resnet = tf.keras.models.load_model(
    RESUME_PATH,
    compile=False,
    custom_objects={
        "color_jitter": color_jitter,
        "Custom>color_jitter": color_jitter,
        "preprocess_input":
            tf.keras.applications.resnet50.preprocess_input
    },
    safe_mode=False
)

print("Best resumed model loaded!")
print("Model output:", final_resnet.output_shape)

In [ ]:
# Load the original test dataset

test_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(ORIGINAL_DIR, "test"),
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

print("Test dataset loaded successfully!")
print("Number of test images:", test_dataset.cardinality().numpy() * BATCH_SIZE)

In [ ]:
# Evaluate the resumed ResNet50 model

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Actual labels
y_true = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for images, labels in test_dataset
])

# Model predictions
y_prob = final_resnet.predict(test_dataset, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

# Evaluation metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("\n===== CLEANED RESNET50 - TEST RESULTS =====")
print(f"Test Images       : {len(y_true)}")
print(f"Accuracy          : {accuracy * 100:.2f}%")
print(f"Macro Precision   : {precision * 100:.2f}%")
print(f"Macro Recall      : {recall * 100:.2f}%")
print(f"Macro F1-Score    : {f1 * 100:.2f}%")

Removing 44 severely padded training images did not improve overall accuracy.
Macro precision improved, meaning the model's predictions were more precise on average.
Macro recall and F1-score decreased slightly.
The original ResNet50 remains our best model overall.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Classification report
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(18, 15))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Cleaned ResNet50 - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# class-weighted training

import pandas as pd
import numpy as np

# Count actual and predicted samples for each class
actual_counts = np.bincount(y_true, minlength=len(class_names))
predicted_counts = np.bincount(y_pred, minlength=len(class_names))

distribution_df = pd.DataFrame({
    "Class": class_names,
    "Actual Images": actual_counts,
    "Predicted Images": predicted_counts,
    "Difference": predicted_counts - actual_counts
})

print(distribution_df.to_string(index=False))

In [ ]:
# calculate class weights

from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os

# Count training images for each class
train_counts = np.array([
    len([
        f for f in os.listdir(
            os.path.join(CLEAN_DIR, "train", class_name)
        )
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    for class_name in class_names
])

# Calculate balanced class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=np.repeat(np.arange(len(class_names)), train_counts)
)

class_weight_dict = {
    i: float(weight)
    for i, weight in enumerate(class_weights)
}

for name, count, weight in zip(
    class_names, train_counts, class_weights
):
    print(f"{name:15s} Images: {count:4d} | Weight: {weight:.3f}")

In [ ]:
# Calculate moderated weights

import numpy as np

# Moderated weights using square-root inverse frequency
moderated_weights = np.sqrt(
    train_counts.sum() / (len(class_names) * train_counts)
)

# Normalize weights so their average is 1
moderated_weights = moderated_weights / moderated_weights.mean()

# Keep weights within a reasonable range
moderated_weights = np.clip(
    moderated_weights,
    0.5,
    2.0
)

class_weight_dict = {
    i: float(weight)
    for i, weight in enumerate(moderated_weights)
}

print("MODERATED CLASS WEIGHTS\n")

for name, count, weight in zip(
    class_names, train_counts, moderated_weights
):
    print(f"{name:15s} Images: {count:4d} | Weight: {weight:.3f}")

print("\nMinimum weight:", round(moderated_weights.min(), 3))
print("Maximum weight:", round(moderated_weights.max(), 3))
print("Average weight:", round(moderated_weights.mean(), 3))

In [ ]:
# Prepare a separate ResNet50 model

import os
import tensorflow as tf

# Original best ResNet50 model
SOURCE_MODEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_finetuned_best.keras"
)

# New checkpoint path
WEIGHTED_MODEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_classweighted_best.keras"
)

# Safety checks
assert os.path.exists(SOURCE_MODEL_PATH), "Original model not found!"

assert not os.path.exists(WEIGHTED_MODEL_PATH), (
    "Weighted checkpoint already exists. "
    "Stop to avoid overwriting it."
)

# Load original model
weighted_resnet = tf.keras.models.load_model(
    SOURCE_MODEL_PATH,
    compile=False,
    custom_objects={
        "color_jitter": color_jitter,
        "Custom>color_jitter": color_jitter,
        "preprocess_input": tf.keras.applications.resnet50.preprocess_input
    },
    safe_mode=False
)

print("Original ResNet50 loaded successfully!")
print("Model name:", weighted_resnet.name)
print("Output shape:", weighted_resnet.output_shape)
print("New checkpoint:", WEIGHTED_MODEL_PATH)

In [ ]:
# Configure fine-tuning

# Find the ResNet50 backbone
base_model = weighted_resnet.get_layer("resnet50")

# Enable fine-tuning
base_model.trainable = True

# Freeze the first 155 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Unfreeze the final 20 layers
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Compile the model
weighted_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.000005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Fine-tuning configuration completed!")
print("Total backbone layers:", len(base_model.layers))
print("Trainable backbone layers:", sum(
    layer.trainable for layer in base_model.layers
))
print("Learning rate: 0.000005")
print("Trainable parameters:", weighted_resnet.trainable_parameters if hasattr(weighted_resnet, "trainable_parameters") else sum(
    tf.size(v).numpy() for v in weighted_resnet.trainable_weights
))

In [ ]:
# Prepare training and validation datasets

# Create cleaned training dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLEAN_DIR, "train"),
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

# Use original validation dataset
val_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(ORIGINAL_DIR, "val"),
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

print("\nDatasets prepared successfully!")
print("Training images: 3904")
print("Validation images: 854")

In [ ]:
# Configure training callbacks

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=WEIGHTED_MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )
]

print("Training callbacks configured successfully!")
print("Checkpoint:", WEIGHTED_MODEL_PATH)

In [ ]:
# Start class-weighted fine-tuning

history_weighted = weighted_resnet.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\nClass-weighted fine-tuning completed!")

Our previous ResNet50 achieved approximately 58–60% validation accuracy.

The class-weighted model started at just 3.63% validation accuracy and reached only 44.03%.

This is a substantial performance drop.

In [ ]:
# verify the saved checkpoint

import os

print("Checkpoint exists:", os.path.exists(WEIGHTED_MODEL_PATH))

if os.path.exists(WEIGHTED_MODEL_PATH):
    size_mb = os.path.getsize(WEIGHTED_MODEL_PATH) / (1024 * 1024)
    print(f"Checkpoint size: {size_mb:.2f} MB")

print("\nBest validation accuracy:",
      max(history_weighted.history["val_accuracy"]) * 100)

print("Final training accuracy:",
      history_weighted.history["accuracy"][-1] * 100)

print("Final validation accuracy:",
      history_weighted.history["val_accuracy"][-1] * 100)

In [ ]:
# Plot training and validation accuracy

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    history_weighted.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_weighted.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("Class-Weighted ResNet50 - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history_weighted.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_weighted.history["val_loss"],
    label="Validation Loss"
)

plt.title("Class-Weighted ResNet50 - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

we'll load the original model and generate the report. No training required.

In [ ]:
# Load the original best model

import tensorflow as tf
import os

ORIGINAL_MODEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "saved_models/resnet50_finetuned_best.keras"
)

original_resnet = tf.keras.models.load_model(
    ORIGINAL_MODEL_PATH,
    compile=False,
    custom_objects={
        "color_jitter": color_jitter,
        "Custom>color_jitter": color_jitter,
        "preprocess_input": tf.keras.applications.resnet50.preprocess_input
    },
    safe_mode=False
)

print("Original ResNet50 loaded successfully!")
print("Model:", original_resnet.name)
print("Output:", original_resnet.output_shape)

In [ ]:
# Identify incorrectly classified images

import numpy as np
import pandas as pd

# Generate predictions using original ResNet50
original_probabilities = original_resnet.predict(
    test_dataset,
    verbose=1
)

original_predictions = np.argmax(
    original_probabilities,
    axis=1
)

# Identify incorrect predictions
misclassified_indices = np.where(
    y_true != original_predictions
)[0]

print("\n===== MISCLASSIFICATION ANALYSIS =====")
print("Total test images:", len(y_true))
print("Correct predictions:", len(y_true) - len(misclassified_indices))
print("Incorrect predictions:", len(misclassified_indices))
print(
    "Test accuracy:",
    round((len(y_true) - len(misclassified_indices)) / len(y_true) * 100, 2),
    "%"
)

# Create a report
error_report = pd.DataFrame({
    "Image Index": misclassified_indices,
    "Actual Class": [class_names[i] for i in y_true[misclassified_indices]],
    "Predicted Class": [
        class_names[i] for i in original_predictions[misclassified_indices]
    ],
    "Confidence": [
        round(float(np.max(original_probabilities[i])) * 100, 2)
        for i in misclassified_indices
    ]
})

print("\nFirst 20 misclassified images:")
display(error_report.head(20))

The result of 0.64% accuracy is suspicious because this same original ResNet50 previously achieved 60.69% test accuracy.

In [ ]:
#Verify the class mapping

print("Dataset class names:")
print(test_dataset.class_names)

print("\nModel class names:")
print(class_names)

print("\nActual class distribution:")
print(np.bincount(y_true, minlength=25))

print("\nPredicted class distribution:")
print(np.bincount(original_predictions, minlength=25))

print("\nFirst 10 actual labels:")
print(y_true[:10])

print("\nFirst 10 predicted labels:")
print(original_predictions[:10])

print("\nModel output shape:")
print(original_probabilities.shape)

In [ ]:
# Check the model's preprocessing and predictions

# Inspect original model architecture
print("MODEL:", original_resnet.name)

print("\nFirst 15 model layers:")
for layer in original_resnet.layers[:15]:
    print(layer.name, layer.__class__.__name__)

print("\nPreprocessing layer:")
for layer in original_resnet.layers:
    if "preprocess" in layer.name.lower():
        print(layer.name, layer.get_config())

# Inspect one test batch
images, labels = next(iter(test_dataset))

print("\nInput image shape:", images.shape)
print("Input pixel minimum:", float(tf.reduce_min(images)))
print("Input pixel maximum:", float(tf.reduce_max(images)))

# Compare predictions from original and cleaned models
print("\nOriginal model predictions:")
print(np.argmax(original_resnet.predict(images[:5], verbose=0), axis=1))

print("\nCleaned model predictions:")
print(np.argmax(final_resnet.predict(images[:5], verbose=0), axis=1))

print("\nActual labels:")
print(np.argmax(labels[:5].numpy(), axis=1))

In [ ]:
# Check the augmentation behaviour

# Inspect the model's augmentation pipeline
augmentation = original_resnet.get_layer("data_augmentation")

print("Augmentation layers:")

for layer in augmentation.layers:
    print(
        layer.name,
        "|",
        layer.__class__.__name__
    )

print("\nTesting repeated predictions on the same images:")

images, labels = next(iter(test_dataset))

for i in range(3):
    predictions = original_resnet(
        images[:5],
        training=False
    ).numpy()

    print(
        f"Run {i+1}:",
        np.argmax(predictions, axis=1)
    )

print("\nActual labels:")
print(np.argmax(labels[:5].numpy(), axis=1))

The predictions are identical across all three runs, so there is no obvious inference-time randomness.

However, the original model predicts:
- Actual: Person
- Predicted: Bench or Pizza
for all five initial test images. This is clearly inconsistent with the previously reported 60.69% test accuracy.

In [ ]:
# Verify the original model checkpoint

import os
import hashlib

def get_file_hash(path):
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

print("Original model file:")
print(ORIGINAL_MODEL_PATH)

print("\nFile size:",
      round(os.path.getsize(ORIGINAL_MODEL_PATH) / (1024 * 1024), 2),
      "MB")

print("\nSHA256:")
print(get_file_hash(ORIGINAL_MODEL_PATH))

print("\nCleaned model SHA256:")
print(get_file_hash(
    "/content/drive/MyDrive/SmartVision_AI/saved_models/"
    "resnet50_cleaned_v1_resumed_best.keras"
))

In [ ]:
# Verify the original model's preprocessing


for model_name, model in [
    ("Original ResNet50", original_resnet),
    ("Cleaned ResNet50", final_resnet)
]:
    print("\n==============================")
    print(model_name)
    print("==============================")

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Lambda):
            print("\nLayer:", layer.name)
            print("Function:", layer.function)
            print("Config:", layer.get_config())

    print("\nFinal output layer:")
    print(model.layers[-1].get_config())

Your earlier 60.69% result may have been calculated using a different evaluation pipeline. We need to reproduce the result independently.

Let's first compare the original model and cleaned model on the same five images.

In [ ]:
import numpy as np
import tensorflow as tf

images, labels = next(iter(test_dataset))

original_output = original_resnet.predict(
    images,
    verbose=0
)

cleaned_output = final_resnet.predict(
    images,
    verbose=0
)

actual = np.argmax(labels.numpy(), axis=1)

original_pred = np.argmax(original_output, axis=1)
cleaned_pred = np.argmax(cleaned_output, axis=1)

comparison = pd.DataFrame({
    "Actual": [class_names[i] for i in actual[:10]],
    "Original Prediction": [
        class_names[i] for i in original_pred[:10]
    ],
    "Original Confidence": [
        round(float(np.max(p)) * 100, 2)
        for p in original_output[:10]
    ],
    "Cleaned Prediction": [
        class_names[i] for i in cleaned_pred[:10]
    ],
    "Cleaned Confidence": [
        round(float(np.max(p)) * 100, 2)
        for p in cleaned_output[:10]
    ]
})

display(comparison)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, dataset, model_name):
    probabilities = model.predict(dataset, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(
        y_true, predictions, average="macro", zero_division=0
    )
    recall = recall_score(
        y_true, predictions, average="macro", zero_division=0
    )
    f1 = f1_score(
        y_true, predictions, average="macro", zero_division=0
    )

    return {
        "Model": model_name,
        "Accuracy": round(accuracy * 100, 2),
        "Precision": round(precision * 100, 2),
        "Recall": round(recall * 100, 2),
        "F1-score": round(f1 * 100, 2)
    }

results = [
    evaluate_model(original_resnet, test_dataset, "Original ResNet50"),
    evaluate_model(final_resnet, test_dataset, "Cleaned ResNet50")
]

display(pd.DataFrame(results))

# yolo improve

In [ ]:
# Let's start with one diagnostic

import torch
import os

print("===== HARDWARE CHECK =====")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("GPU not available. Current training will use CPU.")

# Check existing YOLO model
YOLO_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8n_25class_cpu/"
    "weights/best.pt"
)

print("\nExisting YOLO model:", os.path.exists(YOLO_PATH))

if os.path.exists(YOLO_PATH):
    print("Model size:",
          round(os.path.getsize(YOLO_PATH) / (1024 * 1024), 2),
          "MB")

In [ ]:
# Verify the YOLO dataset configuration

import os
import yaml

DATASET_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed"
)

yaml_path = os.path.join(DATASET_DIR, "data.yaml")

print("Dataset YAML exists:", os.path.exists(yaml_path))

if os.path.exists(yaml_path):
    with open(yaml_path, "r") as file:
        data_config = yaml.safe_load(file)

    print("\n===== DATASET CONFIGURATION =====")
    print(yaml.dump(data_config, sort_keys=False))
else:
    print("data.yaml not found. We need to locate it.")


In [ ]:
pip install ultralytics

In [ ]:
# GPU memory and dataset integrity

import os
import torch
from ultralytics import YOLO

# Check dataset directories
for split in ["train", "val", "test"]:
    image_dir = os.path.join(DATASET_DIR, "images", split)
    label_dir = os.path.join(DATASET_DIR, "labels", split)

    print(f"\n{split.upper()}")
    print("Images directory:", os.path.exists(image_dir))
    print("Labels directory:", os.path.exists(label_dir))

    if os.path.exists(image_dir):
        print("Image count:", len(os.listdir(image_dir)))

    if os.path.exists(label_dir):
        print("Label count:", len(os.listdir(label_dir)))

# GPU check
print("\nGPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

# Load pretrained YOLOv8s
yolo_model = YOLO("yolov8s.pt")

print("\nYOLOv8s pretrained model loaded successfully!")

In [ ]:
#  Configure YOLOv8s training

import os
from ultralytics import YOLO

# Training output directory
PROJECT_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training"
)

RUN_NAME = "yolov8s_25class_gpu"

SAVE_DIR = os.path.join(PROJECT_DIR, RUN_NAME)

# Safety check
if os.path.exists(SAVE_DIR):
    raise FileExistsError(
        f"Training directory already exists: {SAVE_DIR}"
    )

# Confirm GPU
import torch
assert torch.cuda.is_available(), "CUDA GPU is not available!"

# Load pretrained YOLOv8s
yolo_model = YOLO("yolov8s.pt")

print("Training configuration ready!")
print("Model: YOLOv8s")
print("Image size: 640")
print("Batch size: 16")
print("Maximum epochs: 50")
print("Device: Tesla T4")
print("Output directory:", SAVE_DIR)

In [ ]:
# Start YOLOv8s training

results = yolo_model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,

    # Training control
    patience=10,
    cos_lr=True,
    amp=True,
    pretrained=True,
    seed=42,
    deterministic=True,

    # Data augmentation
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=0.5,
    close_mosaic=10,
    mixup=0.0,

    # Saving
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=False,
    save=True,
    save_period=5,
    plots=True,
    verbose=True
)

print("\nYOLOv8s training completed!")
print("Training directory:", SAVE_DIR)

Our mAP@50 improved by 7.3 percentage points.

In [ ]:
# Evaluate YOLOv8s on the test dataset

from ultralytics import YOLO
import os

BEST_MODEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8s_25class_gpu/"
    "weights/best.pt"
)

# Load the best checkpoint
best_yolo = YOLO(BEST_MODEL_PATH)

# Evaluate on the untouched test split
test_results = best_yolo.val(
    data=yaml_path,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project=PROJECT_DIR,
    name="yolov8s_25class_test_evaluation",
    exist_ok=False
)

print("\nYOLOv8s TEST EVALUATION COMPLETED!")
print("Test evaluation saved successfully.")

In [ ]:
# Visualize YOLOv8s predictions

from ultralytics import YOLO
from IPython.display import display
from PIL import Image
import os
import glob
import random

MODEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8s_25class_gpu/"
    "weights/best.pt"
)

TEST_IMAGES = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/images/test"
)

model = YOLO(MODEL_PATH)

image_paths = glob.glob(os.path.join(TEST_IMAGES, "*"))
selected_images = random.sample(image_paths, min(6, len(image_paths)))

for image_path in selected_images:
    results = model.predict(
        source=image_path,
        imgsz=640,
        conf=0.25,
        device=0,
        verbose=False
    )

    annotated_image = results[0].plot()
    display(Image.fromarray(annotated_image[:, :, ::-1]))

Test confidence thresholds

In [ ]:
from ultralytics import YOLO
from IPython.display import display
from PIL import Image
import glob
import os

MODEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8s_25class_gpu/"
    "weights/best.pt"
)

TEST_IMAGES = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/images/test"
)

model = YOLO(MODEL_PATH)

image_paths = glob.glob(os.path.join(TEST_IMAGES, "*"))

# Use the same image for all three comparisons
image_path = image_paths[0]

for confidence in [0.25, 0.40, 0.50]:

    results = model.predict(
        source=image_path,
        imgsz=640,
        conf=confidence,
        device=0,
        verbose=False
    )

    print(f"\nConfidence threshold: {confidence}")

    annotated = results[0].plot()
    display(Image.fromarray(annotated[:, :, ::-1]))

In [ ]:
# Investigate this particular image

from ultralytics import YOLO
from IPython.display import display
from PIL import Image
import os
import glob

MODEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8s_25class_gpu/"
    "weights/best.pt"
)

TEST_IMAGES = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/images/test"
)

TEST_LABELS = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/test"
)

model = YOLO(MODEL_PATH)

# Find the same image used in your previous comparison
image_paths = glob.glob(os.path.join(TEST_IMAGES, "*"))

image_path = image_paths[0]
image_name = os.path.basename(image_path)
label_path = os.path.join(
    TEST_LABELS,
    os.path.splitext(image_name)[0] + ".txt"
)

print("Image:", image_name)

print("\nGROUND-TRUTH ANNOTATIONS:")
if os.path.exists(label_path):
    with open(label_path, "r") as f:
        labels = f.readlines()

    for line in labels:
        class_id = int(line.split()[0])
        print(class_id, model.names[class_id], line.strip())
else:
    print("No label file found.")

print("\nMODEL PREDICTIONS (confidence >= 0.01):")

results = model.predict(
    source=image_path,
    imgsz=640,
    conf=0.01,
    device=0,
    verbose=False
)

for box in results[0].boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])

    print(
        model.names[class_id],
        f"{confidence:.4f}"
    )

display(
    Image.fromarray(
        results[0].plot()[:, :, ::-1]
    )
)

In [ ]:
# Check dog bounding-box accuracy

from ultralytics import YOLO
import os
import glob

MODEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "results/yolo_training/yolov8s_25class_gpu/"
    "weights/best.pt"
)

TEST_IMAGES = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/images/test"
)

TEST_LABELS = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/test"
)

model = YOLO(MODEL_PATH)

image_path = os.path.join(TEST_IMAGES, "image_000000001008.jpg")
label_path = os.path.join(TEST_LABELS, "image_000000001008.txt")

# Read image dimensions
import cv2
image = cv2.imread(image_path)
height, width = image.shape[:2]

# Read ground-truth dog box
gt_box = None

with open(label_path, "r") as f:
    for line in f:
        values = line.strip().split()
        if int(values[0]) == 12:
            xc, yc, w, h = map(float, values[1:5])

            gt_box = [
                (xc - w / 2) * width,
                (yc - h / 2) * height,
                (xc + w / 2) * width,
                (yc + h / 2) * height
            ]
            break

# Predict
results = model.predict(
    image_path,
    imgsz=640,
    conf=0.01,
    device=0,
    verbose=False
)

print("GROUND-TRUTH DOG BOX:", gt_box)
print("\nDOG PREDICTIONS:")

for box in results[0].boxes:
    if int(box.cls[0]) == 12:

        pred_box = box.xyxy[0].cpu().numpy().tolist()

        # Calculate IoU
        x1 = max(gt_box[0], pred_box[0])
        y1 = max(gt_box[1], pred_box[1])
        x2 = min(gt_box[2], pred_box[2])
        y2 = min(gt_box[3], pred_box[3])

        intersection = max(0, x2-x1) * max(0, y2-y1)

        gt_area = (
            (gt_box[2]-gt_box[0]) *
            (gt_box[3]-gt_box[1])
        )

        pred_area = (
            (pred_box[2]-pred_box[0]) *
            (pred_box[3]-pred_box[1])
        )

        iou = intersection / (
            gt_area + pred_area - intersection
        )

        print("Confidence:", round(float(box.conf[0]), 4))
        print("Predicted box:", pred_box)
        print("IoU:", round(iou, 4))

In [ ]:
# investigate the dataset imbalance

import os
from collections import Counter

BASE_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels"
)

class_id = 12
class_name = "dog"

for split in ["train", "val", "test"]:

    label_dir = os.path.join(BASE_DIR, split)

    total_images = 0
    dog_images = 0
    dog_instances = 0

    for filename in os.listdir(label_dir):

        if not filename.endswith(".txt"):
            continue

        total_images += 1

        file_path = os.path.join(label_dir, filename)

        with open(file_path, "r") as f:
            labels = f.readlines()

        count = sum(
            1 for line in labels
            if int(line.split()[0]) == class_id
        )

        if count > 0:
            dog_images += 1
            dog_instances += count

    print(f"\n{split.upper()} DATASET")
    print("Total images:", total_images)
    print("Images containing dogs:", dog_images)
    print("Total dog instances:", dog_instances)

In [ ]:
# Check all 25 classes

import os
from collections import Counter

BASE_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels"
)

class_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'truck', 'traffic light', 'stop sign', 'bench',
    'bird', 'cat', 'dog', 'horse', 'cow', 'elephant',
    'bottle', 'cup', 'bowl', 'pizza', 'cake', 'chair',
    'couch', 'potted plant', 'bed'
]

for split in ["train", "val", "test"]:

    counts = Counter()

    label_dir = os.path.join(BASE_DIR, split)

    for filename in os.listdir(label_dir):

        if not filename.endswith(".txt"):
            continue

        with open(os.path.join(label_dir, filename), "r") as f:

            for line in f:
                class_id = int(line.split()[0])
                counts[class_id] += 1

    print(f"\n{'='*45}")
    print(f"{split.upper()} CLASS DISTRIBUTION")
    print(f"{'='*45}")

    for class_id, name in enumerate(class_names):
        print(f"{name:18s}: {counts[class_id]}")

Class imbalance is a significant issue in your YOLOv8s project.

Your training dataset contains 4,000+ object annotations, but they are not distributed evenly across the 25 classes.

In [ ]:
# Check image-level distribution

import os
from collections import defaultdict

LABEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/train"
)

class_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'truck', 'traffic light', 'stop sign', 'bench',
    'bird', 'cat', 'dog', 'horse', 'cow', 'elephant',
    'bottle', 'cup', 'bowl', 'pizza', 'cake', 'chair',
    'couch', 'potted plant', 'bed'
]

image_counts = defaultdict(int)
instance_counts = defaultdict(int)

for filename in os.listdir(LABEL_DIR):

    if not filename.endswith(".txt"):
        continue

    image_classes = set()

    with open(os.path.join(LABEL_DIR, filename), "r") as f:

        for line in f:

            class_id = int(line.split()[0])

            image_classes.add(class_id)
            instance_counts[class_id] += 1

    for class_id in image_classes:
        image_counts[class_id] += 1

print(f"{'CLASS':18s} {'IMAGES':>10s} {'INSTANCES':>12s}")
print("-" * 43)

for class_id, name in enumerate(class_names):

    print(
        f"{name:18s} "
        f"{image_counts[class_id]:10d} "
        f"{instance_counts[class_id]:12d}"
    )

Bird has 96 instances but only 22 images. This means several birds appear together in the same images. Having more instances does not necessarily mean having enough visual diversity.

In [ ]:
# Display annotated images

import os
import glob
import random
import cv2
import matplotlib.pyplot as plt

IMAGE_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed/images/train"
LABEL_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed/labels/train"

class_names = {
    9: "bench",
    10: "bird",
    20: "cake",
    23: "potted plant",
    18: "bowl"
}

# Find images containing these classes
class_images = {name: [] for name in class_names.values()}

for label_path in glob.glob(os.path.join(LABEL_DIR, "*.txt")):

    with open(label_path, "r") as f:
        labels = f.readlines()

    present_classes = {
        int(line.split()[0])
        for line in labels if line.strip()
    }

    for class_id, class_name in class_names.items():
        if class_id in present_classes:
            image_name = os.path.splitext(
                os.path.basename(label_path)
            )[0]

            image_path = os.path.join(
                IMAGE_DIR, image_name + ".jpg"
            )

            if os.path.exists(image_path):
                class_images[class_name].append(
                    (image_path, label_path)
                )

# Display two random images per class
for class_name, samples in class_images.items():

    print(f"\n{class_name.upper()}: {len(samples)} images")

    selected = random.sample(samples, min(2, len(samples)))

    for image_path, label_path in selected:

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = image.shape[:2]

        with open(label_path, "r") as f:
            for line in f:

                values = line.split()
                class_id = int(values[0])

                if class_id not in class_names:
                    continue

                xc, yc, bw, bh = map(float, values[1:5])

                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)

                cv2.rectangle(
                    image, (x1, y1), (x2, y2), (255, 0, 0), 2
                )

                cv2.putText(
                    image,
                    class_names[class_id],
                    (x1, max(20, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 0, 0),
                    2
                )

        plt.figure(figsize=(8, 5))
        plt.imshow(image)
        plt.title(class_name)
        plt.axis("off")
        plt.show()

In [ ]:
# Audit all bench images

import os
import glob
import cv2
import matplotlib.pyplot as plt
import math

IMAGE_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed/images/train"
LABEL_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed/labels/train"

CLASS_ID = 9
CLASS_NAME = "bench"

# Find images containing benches
samples = []

for label_path in glob.glob(os.path.join(LABEL_DIR, "*.txt")):

    with open(label_path, "r") as f:
        labels = f.readlines()

    if any(int(line.split()[0]) == CLASS_ID for line in labels if line.strip()):

        image_path = None
        stem = os.path.splitext(os.path.basename(label_path))[0]

        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = os.path.join(IMAGE_DIR, stem + ext)
            if os.path.exists(candidate):
                image_path = candidate
                break

        if image_path:
            samples.append((image_path, label_path))

print(f"Total {CLASS_NAME} images found: {len(samples)}")

# Display 12 images per page
PAGE_SIZE = 12

for page in range(math.ceil(len(samples) / PAGE_SIZE)):

    batch = samples[page * PAGE_SIZE:(page + 1) * PAGE_SIZE]

    fig, axes = plt.subplots(3, 4, figsize=(18, 12))
    axes = axes.flatten()

    for ax, (image_path, label_path) in zip(axes, batch):

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = image.shape[:2]

        with open(label_path, "r") as f:
            for line in f:

                values = line.split()
                class_id = int(values[0])

                xc, yc, bw, bh = map(float, values[1:5])

                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)

                cv2.rectangle(
                    image, (x1, y1), (x2, y2), (255, 0, 0), 2
                )

                cv2.putText(
                    image,
                    f"class {class_id}",
                    (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 0, 0),
                    2
                )

        ax.imshow(image)
        ax.set_title(os.path.basename(image_path), fontsize=8)
        ax.axis("off")

    for ax in axes[len(batch):]:
        ax.axis("off")

    plt.suptitle(
        f"BENCH ANNOTATION AUDIT - PAGE {page + 1}",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# verify the suspicious annotation

import os

LABEL_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/train/"
    "image_000000001147.txt"
)

class_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'truck', 'traffic light', 'stop sign', 'bench',
    'bird', 'cat', 'dog', 'horse', 'cow', 'elephant',
    'bottle', 'cup', 'bowl', 'pizza', 'cake', 'chair',
    'couch', 'potted plant', 'bed'
]

with open(LABEL_PATH, "r") as f:
    labels = f.readlines()

print("ANNOTATIONS FOR IMAGE 000000001147")
print("-" * 45)

for line in labels:
    values = line.split()
    class_id = int(values[0])

    print(
        f"Class: {class_names[class_id]} "
        f"| ID: {class_id} "
        f"| Coordinates: {values[1:]}"
    )

This is a clear incorrect annotation:
- Wrong class: bench.
- Incorrect bounding box: almost the full image.
- It can teach the model to identify unrelated objects as benches.
We should remove this incorrect annotation from the cleaned dataset, not from the original.

In [ ]:
# Check how many similar suspicious bench annotations exist

import os
import glob

LABEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/train"
)

suspicious = []

for label_path in glob.glob(os.path.join(LABEL_DIR, "*.txt")):

    with open(label_path, "r") as f:
        for line in f:

            values = line.split()

            if int(values[0]) != 9:
                continue

            xc, yc, w, h = map(float, values[1:5])
            area = w * h

            if area > 0.80:
                suspicious.append({
                    "image": os.path.basename(label_path),
                    "width": w,
                    "height": h,
                    "area": area
                })

print("Suspicious bench annotations:", len(suspicious))

for item in suspicious:
    print(item)

In [ ]:
# Check whether there are other incorrect bench labels

import os
import glob

LABEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/train"
)

small_boxes = []

for label_path in glob.glob(os.path.join(LABEL_DIR, "*.txt")):

    with open(label_path, "r") as f:
        for line in f:

            values = line.split()

            if int(values[0]) != 9:
                continue

            xc, yc, w, h = map(float, values[1:5])
            area = w * h

            if area < 0.005:
                small_boxes.append({
                    "image": os.path.basename(label_path),
                    "width": round(w, 4),
                    "height": round(h, 4),
                    "area": round(area, 6)
                })

print("Very small bench annotations:", len(small_boxes))

for item in small_boxes:
    print(item)

In [ ]:
# Inspect the 23 small boxes closely

import os
import glob
import cv2
import matplotlib.pyplot as plt
import math

IMAGE_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/images/train"
)

LABEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/train"
)

CLASS_ID = 9
small_boxes = []

for label_path in glob.glob(os.path.join(LABEL_DIR, "*.txt")):

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:

        values = line.split()

        if int(values[0]) != CLASS_ID:
            continue

        xc, yc, bw, bh = map(float, values[1:5])

        if bw * bh < 0.005:

            stem = os.path.splitext(
                os.path.basename(label_path)
            )[0]

            for ext in [".jpg", ".jpeg", ".png"]:
                image_path = os.path.join(
                    IMAGE_DIR, stem + ext
                )

                if os.path.exists(image_path):
                    small_boxes.append(
                        (image_path, xc, yc, bw, bh)
                    )
                    break

print("Small bench annotations:", len(small_boxes))

for page in range(math.ceil(len(small_boxes) / 8)):

    batch = small_boxes[page * 8:(page + 1) * 8]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for ax, (path, xc, yc, bw, bh) in zip(axes, batch):

        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = image.shape[:2]

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        margin_x = max(40, int((x2 - x1) * 1.5))
        margin_y = max(40, int((y2 - y1) * 1.5))

        crop = image[
            max(0, y1-margin_y):min(h, y2+margin_y),
            max(0, x1-margin_x):min(w, x2+margin_x)
        ]

        ax.imshow(crop)
        ax.set_title(os.path.basename(path), fontsize=8)
        ax.axis("off")

    for ax in axes[len(batch):]:
        ax.axis("off")

    plt.suptitle(f"SMALL BENCH BOXES - PAGE {page+1}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Bird annotation audit

import os
import glob
import cv2
import matplotlib.pyplot as plt
import math

IMAGE_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed/images/train"
LABEL_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed/labels/train"

CLASS_ID = 10
CLASS_NAME = "bird"

samples = []

for label_path in glob.glob(os.path.join(LABEL_DIR, "*.txt")):

    with open(label_path, "r") as f:
        labels = f.readlines()

    if any(int(line.split()[0]) == CLASS_ID for line in labels if line.strip()):

        stem = os.path.splitext(os.path.basename(label_path))[0]

        for ext in [".jpg", ".jpeg", ".png"]:
            image_path = os.path.join(IMAGE_DIR, stem + ext)

            if os.path.exists(image_path):
                samples.append((image_path, label_path))
                break

print(f"Total bird images: {len(samples)}")

PAGE_SIZE = 12

for page in range(math.ceil(len(samples) / PAGE_SIZE)):

    batch = samples[page * PAGE_SIZE:(page + 1) * PAGE_SIZE]

    fig, axes = plt.subplots(3, 4, figsize=(18, 12))
    axes = axes.flatten()

    for ax, (image_path, label_path) in zip(axes, batch):

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = image.shape[:2]

        with open(label_path, "r") as f:
            for line in f:

                values = line.split()
                class_id = int(values[0])

                if class_id != CLASS_ID:
                    continue

                xc, yc, bw, bh = map(float, values[1:5])

                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)

                cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)

        ax.imshow(image)
        ax.set_title(os.path.basename(image_path), fontsize=8)
        ax.axis("off")

    for ax in axes[len(batch):]:
        ax.axis("off")

    plt.suptitle(f"BIRD ANNOTATION AUDIT - PAGE {page + 1}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Inspect suspicious bird annotations numerically

import os
import glob

LABEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/train"
)

bird_boxes = []

for label_path in glob.glob(os.path.join(LABEL_DIR, "*.txt")):

    with open(label_path, "r") as f:
        for line in f:

            values = line.split()

            if int(values[0]) != 10:
                continue

            xc, yc, w, h = map(float, values[1:5])

            bird_boxes.append({
                "image": os.path.basename(label_path),
                "width": round(w, 4),
                "height": round(h, 4),
                "area": round(w * h, 6)
            })

bird_boxes.sort(key=lambda x: x["area"], reverse=True)

print("TOTAL BIRD BOXES:", len(bird_boxes))
print("\nLARGEST BIRD BOXES:")

for item in bird_boxes[:15]:
    print(item)

print("\nSMALLEST BIRD BOXES:")

for item in bird_boxes[-15:]:
    print(item)

# # reconnecting and checking dataset in drive


In [ ]:
# reconnecting and checking dataset in drive

import os

PROJECT_DIR = "/content/drive/MyDrive/SmartVision_AI"

from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = os.path.join(
    PROJECT_DIR, "dataset/detection_fixed"
)

MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "results/yolo_training/yolov8s_25class_gpu/weights/best.pt"
)

print("Dataset exists:", os.path.exists(DATASET_DIR))
print("YOLOv8s checkpoint exists:", os.path.exists(MODEL_PATH))

In [ ]:
import os
import glob

LABEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/train"
)

bird_boxes = []

for label_path in glob.glob(os.path.join(LABEL_DIR, "*.txt")):

    with open(label_path, "r") as f:
        for line in f:

            values = line.split()

            if int(values[0]) != 10:
                continue

            xc, yc, w, h = map(float, values[1:5])

            bird_boxes.append({
                "image": os.path.basename(label_path),
                "width": round(w, 4),
                "height": round(h, 4),
                "area": round(w * h, 6)
            })

bird_boxes.sort(key=lambda x: x["area"], reverse=True)

print("TOTAL BIRD BOXES:", len(bird_boxes))

print("\nLARGEST BIRD BOXES:")
for item in bird_boxes[:15]:
    print(item)

print("\nSMALLEST BIRD BOXES:")
for item in bird_boxes[-15:]:
    print(item)

In [ ]:
# Inspect the suspicious annotations

import os
import cv2
import matplotlib.pyplot as plt

IMAGE_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed/images/train"
LABEL_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_fixed/labels/train"

target_ids = [
    "000000000307",
    "000000004966",
    "000000000805",
    "000000000397",
    "000000000233",
    "000000000443"
]

for image_id in target_ids:

    image_path = os.path.join(IMAGE_DIR, f"image_{image_id}.jpg")
    label_path = os.path.join(LABEL_DIR, f"image_{image_id}.txt")

    if not os.path.exists(image_path):
        print("Image not found:", image_id)
        continue

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    print(f"\nIMAGE: {image_id}")
    print("Bird annotations:")

    with open(label_path, "r") as f:
        for line in f:

            values = line.split()

            if int(values[0]) != 10:
                continue

            xc, yc, bw, bh = map(float, values[1:5])

            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)

            print(f"Box: ({x1}, {y1}, {x2}, {y2}) | Area: {bw*bh:.6f}")

            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)

    plt.figure(figsize=(12, 7))
    plt.imshow(image)
    plt.title(f"Bird annotation audit: {image_id}")
    plt.axis("off")
    plt.show()

In [ ]:
# Before modifying anything

import os

LABEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed/labels/train"
)

target_images = [
    "000000000307",
    "000000004966",
    "000000000805",
    "000000000397"
]

for image_id in target_images:

    label_path = os.path.join(
        LABEL_DIR, f"image_{image_id}.txt"
    )

    print("\n" + "=" * 65)
    print("IMAGE:", image_id)
    print("=" * 65)

    with open(label_path, "r") as f:
        lines = f.readlines()

    for index, line in enumerate(lines):
        values = line.split()

        if int(values[0]) == 10:
            print(f"Line {index}: {line.strip()}")

In [ ]:
# Create a separate cleaned dataset

import os
import shutil

SOURCE_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_fixed"
)

CLEAN_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1"
)

if os.path.exists(CLEAN_DIR):
    print("Cleaned dataset already exists.")
    print("Not overwriting existing files.")

else:
    shutil.copytree(SOURCE_DIR, CLEAN_DIR)
    print("Dataset copied successfully!")

print("Original dataset:", SOURCE_DIR)
print("Cleaned dataset:", CLEAN_DIR)

In [ ]:
  # Apply the three confirmed corrections

  import os

LABEL_DIR = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1/labels/train"
)

corrections = {
    "image_000000000307.txt": [0],
    "image_000000004966.txt": [12],
    "image_000000000397.txt": [12]
}

for filename, remove_lines in corrections.items():

    path = os.path.join(LABEL_DIR, filename)

    with open(path, "r") as f:
        lines = f.readlines()

    updated_lines = [
        line for index, line in enumerate(lines)
        if index not in remove_lines
    ]

    with open(path, "w") as f:
        f.writelines(updated_lines)

    print(
        filename,
        "| Removed:", len(remove_lines),
        "| Remaining:", len(updated_lines)
    )

print("\nThree annotation corrections completed!")

In [ ]:
# Correct image 000000000805.jpg

import os
import cv2
import matplotlib.pyplot as plt

IMAGE_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1/images/train/"
    "image_000000000805.jpg"
)

image = cv2.imread(IMAGE_PATH)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

height, width = image.shape[:2]

print("Image width:", width)
print("Image height:", height)

plt.figure(figsize=(16, 10))
plt.imshow(image)
plt.xticks(range(0, width, 50))
plt.yticks(range(0, height, 50))
plt.grid(True)
plt.title("Image 805 - Individual Bird Bounding Boxes")
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

IMAGE_PATH = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1/images/train/"
    "image_000000000805.jpg"
)

image = cv2.imread(IMAGE_PATH)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Proposed boxes: x1, y1, x2, y2
boxes = [
    (88, 90, 270, 330),
    (310, 130, 590, 350)
]

preview = image.copy()

for i, (x1, y1, x2, y2) in enumerate(boxes, 1):
    cv2.rectangle(preview, (x1, y1), (x2, y2), (255, 0, 0), 3)
    cv2.putText(
        preview, f"Bird {i}", (x1, y1 - 8),
        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2
    )

plt.figure(figsize=(14, 9))
plt.imshow(preview)
plt.title("Proposed Bird Annotation Corrections")
plt.axis("on")
plt.show()

In [ ]:
# Save corrected annotations

from pathlib import Path
import shutil

DATASET = Path(
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1"
)

LABEL_NAME = "image_000000000805.txt"

# Find the annotation file in the dataset splits
matches = list(DATASET.glob(f"labels/*/{LABEL_NAME}"))

if len(matches) != 1:
    print("Expected exactly one annotation file.")
    print("Found:", matches)
else:
    label_path = matches[0]

    # Backup the original annotation
    backup_path = label_path.with_suffix(".txt.backup")
    shutil.copy2(label_path, backup_path)

    # Original pixel coordinates: x1, y1, x2, y2
    boxes = [
        (88, 90, 270, 330),
        (310, 130, 590, 350)
    ]

    IMAGE_WIDTH = 640
    IMAGE_HEIGHT = 415
    BIRD_CLASS_ID = 10

    annotations = []

    for x1, y1, x2, y2 in boxes:
        x_center = ((x1 + x2) / 2) / IMAGE_WIDTH
        y_center = ((y1 + y2) / 2) / IMAGE_HEIGHT
        width = (x2 - x1) / IMAGE_WIDTH
        height = (y2 - y1) / IMAGE_HEIGHT

        annotations.append(
            f"{BIRD_CLASS_ID} {x_center:.6f} "
            f"{y_center:.6f} {width:.6f} {height:.6f}"
        )

    label_path.write_text("\n".join(annotations) + "\n")

    print("Annotations corrected successfully!")
    print("Label file:", label_path)
    print("Backup:", backup_path)
    print("\nNew annotations:")
    print(label_path.read_text())

In [ ]:
# Preview remaining bird annotations

from pathlib import Path
import cv2
import matplotlib.pyplot as plt

DATASET = Path(
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1"
)

image_ids = ["000000000233", "000000000443"]

for image_id in image_ids:
    image_matches = list(DATASET.glob(f"images/*/image_{image_id}.jpg"))

    if not image_matches:
        print(f"Image not found: {image_id}")
        continue

    image_path = image_matches[0]
    label_path = (
        DATASET / "labels" / image_path.parent.name /
        f"image_{image_id}.txt"
    )

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    print(f"\nIMAGE: {image_id}")
    print("Image dimensions:", w, "x", h)

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            values = line.split()
            if len(values) != 5 or int(values[0]) != 10:
                continue

            _, xc, yc, bw, bh = map(float, values)

            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)

    plt.figure(figsize=(12, 7))
    plt.imshow(image)
    plt.title(f"Bird annotation audit: {image_id}")
    plt.axis("on")
    plt.show()

In [ ]:
# Correct images 233 and 443

from pathlib import Path
import shutil

DATASET = Path(
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1"
)

BIRD_CLASS_ID = 10

def find_label(image_id):
    matches = list(
        DATASET.glob(f"labels/*/image_{image_id}.txt")
    )
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one label for {image_id}, found {len(matches)}"
        )
    return matches[0]


# --------------------------------------------------
# IMAGE 000000000233
# Remove the false bird annotation in empty sky.
# --------------------------------------------------

path_233 = find_label("000000000233")
lines_233 = path_233.read_text().splitlines()

remaining_233 = [
    line for line in lines_233
    if not (
        line.strip()
        and int(line.split()[0]) == BIRD_CLASS_ID
    )
]

if len(remaining_233) == len(lines_233):
    print("No bird annotation found in image 233.")
else:
    shutil.copy2(path_233, path_233.with_suffix(".txt.backup"))
    path_233.write_text(
        "\n".join(remaining_233) + ("\n" if remaining_233 else "")
    )
    print("Image 233: false bird annotation removed.")


# --------------------------------------------------
# IMAGE 000000000443
# Remove only the isolated false box near (289, 267).
# Original image dimensions: 500 x 299.
# --------------------------------------------------

path_443 = find_label("000000000443")
lines_443 = path_443.read_text().splitlines()

target_x = 289.5
target_y = 266.5

kept_lines = []
removed_lines = []

for line in lines_443:
    values = line.split()

    if len(values) != 5:
        kept_lines.append(line)
        continue

    class_id = int(values[0])

    if class_id != BIRD_CLASS_ID:
        kept_lines.append(line)
        continue

    xc, yc, bw, bh = map(float, values[1:])

    # Convert normalized YOLO coordinates to pixel center.
    pixel_x = xc * 500
    pixel_y = yc * 299

    if abs(pixel_x - target_x) <= 2 and abs(pixel_y - target_y) <= 2:
        removed_lines.append(line)
    else:
        kept_lines.append(line)

if len(removed_lines) != 1:
    raise RuntimeError(
        f"Expected exactly one isolated box, found {len(removed_lines)}. "
        "Image 443 was not modified."
    )

shutil.copy2(path_443, path_443.with_suffix(".txt.backup"))

path_443.write_text("\n".join(kept_lines) + "\n")

print("Image 443: isolated false annotation removed.")
print("Removed annotation:", removed_lines[0])

print("\nCORRECTION SUMMARY")
print("Image 233 backup:", path_233.with_suffix(".txt.backup"))
print("Image 443 backup:", path_443.with_suffix(".txt.backup"))
print("Image 443 remaining bird boxes:", sum(
    1 for line in kept_lines
    if line.strip() and int(line.split()[0]) == BIRD_CLASS_ID
))

Both corrections were completed successfully, and backup files were created.

In [ ]:
# Verify the cleaned dataset

# Bird annotation audit

from pathlib import Path

DATASET = Path(
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1"
)

LABEL_DIR = DATASET / "labels" / "train"

BIRD_CLASS_ID = 10

bird_boxes = []
bird_images = set()

for label_path in LABEL_DIR.glob("*.txt"):

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        class_id, xc, yc, width, height = map(float, values)

        if int(class_id) != BIRD_CLASS_ID:
            continue

        area = width * height

        bird_images.add(label_path.stem)

        bird_boxes.append({
            "image": label_path.stem,
            "width": width,
            "height": height,
            "area": area
        })

print("BIRD ANNOTATION AUDIT")
print("---------------------")
print("Images containing birds:", len(bird_images))
print("Total bird bounding boxes:", len(bird_boxes))

small_boxes = [
    box for box in bird_boxes
    if box["area"] < 0.001
]

large_boxes = [
    box for box in bird_boxes
    if box["area"] > 0.60
]

print("\nVery small boxes:", len(small_boxes))
for box in small_boxes:
    print(box)

print("\nUnusually large boxes:", len(large_boxes))
for box in large_boxes:
    print(box)

In [ ]:
# Inspect the suspicious annotations closely

from pathlib import Path
import cv2
import matplotlib.pyplot as plt

DATASET = Path(
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1"
)

def inspect_bird_boxes(image_id):
    image_path = list(
        DATASET.glob(f"images/*/image_{image_id}.jpg")
    )[0]

    label_path = (
        DATASET / "labels" / image_path.parent.name /
        f"image_{image_id}.txt"
    )

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    boxes = []

    for line in label_path.read_text().splitlines():
        values = line.split()

        if len(values) != 5 or int(values[0]) != 10:
            continue

        _, xc, yc, bw, bh = map(float, values)

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        boxes.append((x1, y1, x2, y2))

    print(f"\nImage {image_id}: {len(boxes)} bird boxes")

    fig, axes = plt.subplots(
        1, 2, figsize=(15, 6)
    )

    axes[0].imshow(image)
    axes[0].set_title("Original image with annotations")

    for x1, y1, x2, y2 in boxes:
        cv2.rectangle(
            image, (x1, y1), (x2, y2),
            (255, 0, 0), 2
        )

    axes[0].imshow(image)
    axes[0].axis("off")

    # Crop around all annotated bird boxes
    if boxes:
        x1 = max(0, min(b[0] for b in boxes) - 30)
        y1 = max(0, min(b[1] for b in boxes) - 30)
        x2 = min(w, max(b[2] for b in boxes) + 30)
        y2 = min(h, max(b[3] for b in boxes) + 30)

        crop = image[y1:y2, x1:x2]
        axes[1].imshow(crop)
        axes[1].set_title("Zoomed annotation region")

    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


for image_id in ["000000004966", "000000000443", "000000000875"]:
    inspect_bird_boxes(image_id)

In [ ]:
# Next, let's audit the bench class

# Bench annotation audit

from pathlib import Path

DATASET = Path(
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1"
)

LABEL_DIR = DATASET / "labels" / "train"

BENCH_CLASS_ID = 9

bench_boxes = []
bench_images = set()

for label_path in LABEL_DIR.glob("*.txt"):

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        class_id, xc, yc, width, height = map(float, values)

        if int(class_id) != BENCH_CLASS_ID:
            continue

        bench_images.add(label_path.stem)

        bench_boxes.append({
            "image": label_path.stem,
            "width": width,
            "height": height,
            "area": width * height
        })

small_boxes = [b for b in bench_boxes if b["area"] < 0.001]
large_boxes = [b for b in bench_boxes if b["area"] > 0.60]

print("BENCH ANNOTATION AUDIT")
print("----------------------")
print("Images containing benches:", len(bench_images))
print("Total bench boxes:", len(bench_boxes))

print("\nVery small boxes:", len(small_boxes))
for box in small_boxes:
    print(box)

print("\nUnusually large boxes:", len(large_boxes))
for box in large_boxes:
    print(box)

In [ ]:
# Visual inspection of the three images

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

DATASET = Path(
    "/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1"
)

IMAGE_DIR = DATASET / "images" / "train"
LABEL_DIR = DATASET / "labels" / "train"

suspicious_ids = [
    "image_000000001045",
    "image_000000000056",
    "image_000000001147"
]

for image_id in suspicious_ids:

    image_path = None

    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = IMAGE_DIR / (image_id + ext)
        if candidate.exists():
            image_path = candidate
            break

    label_path = LABEL_DIR / (image_id + ".txt")

    if image_path is None or not label_path.exists():
        print("Missing image or label:", image_id)
        continue

    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(image)

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        class_id, xc, yc, bw, bh = map(float, values)

        if int(class_id) == 9:

            x1 = (xc - bw / 2) * width
            y1 = (yc - bh / 2) * height
            box_width = bw * width
            box_height = bh * height

            rect = patches.Rectangle(
                (x1, y1),
                box_width,
                box_height,
                linewidth=3,
                edgecolor="red",
                facecolor="none"
            )

            ax.add_patch(rect)

    ax.set_title(f"Bench annotation audit: {image_id}")
    ax.axis("off")
    plt.show()

In [ ]:
# Remove the false annotation safely

from pathlib import Path
import shutil

LABEL_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1/labels/train"
)

label_path = LABEL_DIR / "image_000000001147.txt"
backup_path = LABEL_DIR / "image_000000001147.txt.backup"

# Create backup without overwriting an existing backup
if not backup_path.exists():
    shutil.copy2(label_path, backup_path)

lines = label_path.read_text().splitlines()

# Class 9 = bench
updated_lines = [
    line for line in lines
    if not line.split() or line.split()[0] != "9"
]

removed = len(lines) - len(updated_lines)

label_path.write_text(
    "\n".join(updated_lines) + "\n"
)

print("Correction completed!")
print("Removed false bench annotations:", removed)
print("Remaining annotations:", len(updated_lines))
print("Backup:", backup_path)

In [ ]:
# Visualize the 7 small bench annotations

import os
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

DATASET_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1"
)

image_ids = [
    "000000001112",
    "000000000332",
    "000000000879",
    "000000000966",
    "000000000956",
    "000000000725"
]

for image_id in image_ids:

    image_path = DATASET_DIR / "images/train" / f"image_{image_id}.jpg"
    label_path = DATASET_DIR / "labels/train" / f"image_{image_id}.txt"

    if not image_path.exists() or not label_path.exists():
        print(f"Missing image or label: {image_id}")
        continue

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]

    plt.figure(figsize=(12, 7))
    plt.imshow(image)

    bench_count = 0

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        class_id = int(values[0])

        if class_id != 9:
            continue

        xc, yc, w, h = map(float, values[1:])

        x1 = (xc - w / 2) * width
        y1 = (yc - h / 2) * height

        rect = plt.Rectangle(
            (x1, y1),
            w * width,
            h * height,
            fill=False,
            edgecolor="red",
            linewidth=2
        )

        plt.gca().add_patch(rect)

        bench_count += 1

    plt.title(f"Bench Audit: {image_id} | Boxes: {bench_count}")
    plt.axis("off")
    plt.show()

inspect the 7 flagged boxes individually at higher zoom before modifying anything.

In [ ]:
# Inspect small bench boxes individually

import cv2
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1"
)

IMAGE_DIR = BASE / "images/train"
LABEL_DIR = BASE / "labels/train"

MAX_AREA = 0.001

for label_path in sorted(LABEL_DIR.glob("*.txt")):

    image_path = IMAGE_DIR / (label_path.stem + ".jpg")

    if not image_path.exists():
        continue

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    H, W = image.shape[:2]

    for line_no, line in enumerate(label_path.read_text().splitlines()):

        values = line.split()

        if len(values) != 5 or int(values[0]) != 9:
            continue

        xc, yc, bw, bh = map(float, values[1:])
        area = bw * bh

        if area >= MAX_AREA:
            continue

        x1 = int((xc - bw / 2) * W)
        y1 = int((yc - bh / 2) * H)
        x2 = int((xc + bw / 2) * W)
        y2 = int((yc + bh / 2) * H)

        # Crop with surrounding context
        pad_x = max(40, int((x2 - x1) * 2))
        pad_y = max(40, int((y2 - y1) * 2))

        cx1 = max(0, x1 - pad_x)
        cy1 = max(0, y1 - pad_y)
        cx2 = min(W, x2 + pad_x)
        cy2 = min(H, y2 + pad_y)

        crop = image[cy1:cy2, cx1:cx2].copy()

        # Draw original annotation
        cv2.rectangle(
            crop,
            (x1 - cx1, y1 - cy1),
            (x2 - cx1, y2 - cy1),
            (255, 0, 0),
            2
        )

        plt.figure(figsize=(8, 5))
        plt.imshow(crop)
        plt.title(
            f"{label_path.stem} | Line {line_no} | Area: {area:.6f}"
        )
        plt.axis("off")
        plt.show()

In [ ]:
# Display the original image and all annotations

import cv2
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1"
)

image_path = BASE / "images/train/image_000000000956.jpg"
label_path = BASE / "labels/train/image_000000000956.txt"

image = cv2.imread(str(image_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

H, W = image.shape[:2]

print("Image dimensions:", W, "x", H)

plt.figure(figsize=(16, 10))
plt.imshow(image)

for i, line in enumerate(label_path.read_text().splitlines()):

    values = line.split()

    if len(values) != 5:
        continue

    class_id = int(values[0])
    xc, yc, bw, bh = map(float, values[1:])

    x1 = int((xc - bw / 2) * W)
    y1 = int((yc - bh / 2) * H)
    x2 = int((xc + bw / 2) * W)
    y2 = int((yc + bh / 2) * H)

    color = "red" if class_id == 9 else "lime"

    rect = plt.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        fill=False, edgecolor=color, linewidth=2
    )

    plt.gca().add_patch(rect)

    plt.text(
        x1, max(0, y1-5),
        f"Class {class_id} | Line {i}",
        color=color, fontsize=8
    )

plt.title("Image 956 - Complete Annotation Inspection")
plt.axis("off")
plt.show()

print("\nComplete label file:\n")
print(label_path.read_text())

In [ ]:
# Zoom in before deciding

import cv2
import matplotlib.pyplot as plt

image_path = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1/images/train/"
    "image_000000000956.jpg"
)

img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Region containing the two suspicious annotations
crop = img[180:250, 260:330]

# Enlarge for inspection
zoom = cv2.resize(
    crop, None,
    fx=6, fy=6,
    interpolation=cv2.INTER_CUBIC
)

plt.figure(figsize=(12, 8))
plt.imshow(zoom)
plt.title("Image 956 - Suspicious Bench Region")
plt.axis("off")
plt.show()

removing both bench annotations.

In [ ]:
import os
import shutil

label_path = (
    "/content/drive/MyDrive/SmartVision_AI/"
    "dataset/detection_cleaned_v1/labels/train/"
    "image_000000000956.txt"
)

backup_path = label_path + ".backup"

# Create backup without overwriting an existing backup
if not os.path.exists(backup_path):
    shutil.copy2(label_path, backup_path)

with open(label_path, "r") as f:
    lines = f.readlines()

# Remove the two confirmed suspicious bench annotations
remove_lines = {10, 11}

updated_lines = [
    line for index, line in enumerate(lines)
    if index not in remove_lines
]

with open(label_path, "w") as f:
    f.writelines(updated_lines)

print("Correction completed!")
print("Removed bench annotations:", len(remove_lines))
print("Remaining annotations:", len(updated_lines))
print("Backup:", backup_path)

inspect the remaining unusually small bench annotations, especially images 332, 725, 879, 966, and 1112.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

DATASET_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1"

image_dir = os.path.join(DATASET_DIR, "images/train")
label_dir = os.path.join(DATASET_DIR, "labels/train")

image_ids = ["000000001112", "000000000332", "000000000879",
             "000000000966", "000000000956", "000000000725"]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for ax, image_id in zip(axes, image_ids):
    image_path = os.path.join(image_dir, f"image_{image_id}.jpg")
    label_path = os.path.join(label_dir, f"image_{image_id}.txt")

    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width = img.shape[:2]

    ax.imshow(img)
    ax.set_title(f"Image {image_id}")

    with open(label_path, "r") as f:
        for line in f:
            values = line.strip().split()
            if len(values) != 5:
                continue

            cls, xc, yc, bw, bh = map(float, values)

            if int(cls) == 9:  # Bench
                x1 = (xc - bw / 2) * width
                y1 = (yc - bh / 2) * height

                rect = Rectangle(
                    (x1, y1),
                    bw * width,
                    bh * height,
                    linewidth=2,
                    edgecolor="red",
                    facecolor="none"
                )
                ax.add_patch(rect)

    ax.axis("off")

plt.tight_layout()
plt.show()

# inspect the remaining image, 000000000725, before making any corrections.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

DATASET_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1"

image_id = "000000000725"

image_path = os.path.join(
    DATASET_DIR, "images/train", f"image_{image_id}.jpg"
)
label_path = os.path.join(
    DATASET_DIR, "labels/train", f"image_{image_id}.txt"
)

img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

height, width = img.shape[:2]

fig, ax = plt.subplots(figsize=(16, 10))
ax.imshow(img)
ax.set_title(f"Image {image_id} – Complete Bench Annotation Audit")

with open(label_path, "r") as f:
    for line_number, line in enumerate(f):
        values = line.strip().split()
        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) == 9:
            x1 = (xc - bw / 2) * width
            y1 = (yc - bh / 2) * height

            rect = Rectangle(
                (x1, y1), bw * width, bh * height,
                linewidth=2, edgecolor="red",
                facecolor="none"
            )
            ax.add_patch(rect)

            ax.text(x1, y1 - 5, f"Bench {line_number}",
                    color="yellow", fontsize=10)

ax.axis("off")
plt.show()

print("Image dimensions:", width, "x", height)
print("\nComplete label file:")
with open(label_path, "r") as f:
    print(f.read())

In [ ]:
# couch annotation audit

import os

DATASET_DIR = "/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1"
LABEL_DIR = os.path.join(DATASET_DIR, "labels/train")

COUCH_CLASS_ID = 22

couch_boxes = []
couch_images = set()

for filename in os.listdir(LABEL_DIR):
    if not filename.endswith(".txt"):
        continue

    label_path = os.path.join(LABEL_DIR, filename)

    with open(label_path, "r") as f:
        for line_number, line in enumerate(f):
            values = line.strip().split()

            if len(values) != 5:
                continue

            cls, xc, yc, bw, bh = map(float, values)

            if int(cls) == COUCH_CLASS_ID:
                area = bw * bh

                couch_images.add(filename)
                couch_boxes.append({
                    "image": filename.replace(".txt", ""),
                    "line": line_number,
                    "width": round(bw, 6),
                    "height": round(bh, 6),
                    "area": round(area, 6)
                })

small_boxes = [b for b in couch_boxes if b["area"] < 0.001]
large_boxes = [b for b in couch_boxes if b["area"] > 0.70]

print("COUCH ANNOTATION AUDIT")
print("----------------------")
print("Images containing couches:", len(couch_images))
print("Total couch boxes:", len(couch_boxes))
print("Very small boxes:", len(small_boxes))
print("Unusually large boxes:", len(large_boxes))

print("\nSMALL BOXES:")
for box in small_boxes:
    print(box)

print("\nLARGE BOXES:")
for box in large_boxes:
    print(box)

In [ ]:
# Visual inspection

import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Rectangle

BASE = Path(
    "/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1"
)

image_ids = [
    "000000000114",
    "000000000445",
    "000000000318",
    "000000000288",
    "000000001135"
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for ax, image_id in zip(axes, image_ids):

    image_path = BASE / "images/train" / f"image_{image_id}.jpg"
    label_path = BASE / "labels/train" / f"image_{image_id}.txt"

    image = cv2.imread(str(image_path))

    if image is None:
        ax.set_title(f"Missing: {image_id}")
        ax.axis("off")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    ax.imshow(image)
    ax.set_title(image_id)

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5 or int(values[0]) != 22:
            continue

        _, xc, yc, bw, bh = map(float, values)

        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h

        rect = Rectangle(
            (x1, y1),
            bw * w,
            bh * h,
            linewidth=2,
            edgecolor="red",
            facecolor="none"
        )

        ax.add_patch(rect)

    ax.axis("off")

axes[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# remove the incorrect couch annotation from 000000000318

from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
label_path = BASE / "labels/train/image_000000000318.txt"
backup_path = label_path.with_suffix(".txt.backup")

# Create backup only if one does not already exist
if not backup_path.exists():
    shutil.copy2(label_path, backup_path)

lines = label_path.read_text().splitlines()
updated_lines = [
    line for line in lines
    if not (line.strip() and int(line.split()[0]) == 22)
]

removed = len(lines) - len(updated_lines)
label_path.write_text("\n".join(updated_lines) + ("\n" if updated_lines else ""))

print("Removed couch annotations:", removed)
print("Backup preserved:", backup_path.exists())
print("Remaining annotations:", len(updated_lines))

let's inspect 000000001135 more closely

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
image_id = "000000001135"

image_path = BASE / "images/train" / f"image_{image_id}.jpg"
label_path = BASE / "labels/train" / f"image_{image_id}.txt"

image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
h, w = image.shape[:2]

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)

for line in label_path.read_text().splitlines():
    values = line.split()
    if len(values) != 5:
        continue

    cls, xc, yc, bw, bh = map(float, values)

    if int(cls) == 22:
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        ax.add_patch(Rectangle((x1, y1), bw*w, bh*h,
                               fill=False, edgecolor="red", linewidth=2))

ax.set_title(image_id)
ax.axis("off")
plt.show()

In [ ]:
#correct its bounding box to cover only the visible couch

from pathlib import Path
import cv2

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
image_path = BASE / "images/train/image_000000001135.jpg"
label_path = BASE / "labels/train/image_000000001135.txt"

image = cv2.imread(str(image_path))
h, w = image.shape[:2]

print("Image dimensions:", w, "x", h)
print("\nCurrent annotations:")

for i, line in enumerate(label_path.read_text().splitlines()):
    values = line.split()
    if len(values) == 5:
        print(f"Line {i}: {line}")

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_path = BASE / "images/train/image_000000001135.jpg"
image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)

# Proposed couch box
ax.add_patch(Rectangle((0, 30), 640, 270,
                       fill=False, edgecolor="red", linewidth=2))

ax.set_title("Proposed couch bounding box")
ax.axis("off")
plt.show()

In [ ]:
# preview the tighter couch bounding box

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
image_path = BASE / "images/train/image_000000001135.jpg"

image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)

# Proposed couch bounding box
ax.add_patch(Rectangle(
    (0, 55), 640, 225,
    fill=False, edgecolor="red", linewidth=2
))

ax.set_title("Tighter proposed couch bounding box")
ax.axis("off")
plt.show()

In [ ]:
# save the correction for 000000001135

from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
label_path = BASE / "labels/train/image_000000001135.txt"
backup_path = label_path.with_suffix(".txt.backup")

# Preserve original label
if not backup_path.exists():
    shutil.copy2(label_path, backup_path)

lines = label_path.read_text().splitlines()

# Remove the old couch annotation only
lines = [
    line for line in lines
    if not (line.strip() and int(line.split()[0]) == 22)
]

# Add corrected couch annotation
lines.append("22 0.500000 0.348958 1.000000 0.468750")

label_path.write_text("\n".join(lines) + "\n")

print("Corrected couch annotation saved.")
print("Dog annotations preserved:", sum(
    1 for line in lines if line.split()[0] == "12"
))
print("Couch annotations:", sum(
    1 for line in lines if line.split()[0] == "22"
))
print("Backup exists:", backup_path.exists())

In [ ]:
from pathlib import Path
from collections import defaultdict

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
LABEL_DIR = BASE / "labels/train"

CLASS_ID = 20  # cake

cake_images = 0
cake_boxes = 0
small_boxes = []
large_boxes = []

for label_path in LABEL_DIR.glob("*.txt"):
    for line_no, line in enumerate(label_path.read_text().splitlines()):
        values = line.split()

        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) != CLASS_ID:
            continue

        cake_images += 1 if cake_images == 0 else 0
        cake_boxes += 1

        area = bw * bh

        if area < 0.001:
            small_boxes.append((label_path.name, line_no, round(area, 6)))

        if area > 0.25:
            large_boxes.append((label_path.name, line_no, round(area, 4)))

print("CAKE ANNOTATION AUDIT")
print("Total cake boxes:", cake_boxes)
print("Images containing cake:", len(set(
    item[0] for item in small_boxes + large_boxes
)))

print("\nVery small boxes:", len(small_boxes))
for item in small_boxes:
    print(item)

print("\nUnusually large boxes:", len(large_boxes))
for item in large_boxes:
    print(item)

In [ ]:
# Visually inspect suspicious cake annotations

from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_ids = [
    "000000001284",
    "000000000907",
    "000000000195",
    "000000001248",
    "000000000019",
    "000000001451",
    "000000000928",
    "000000001247",
    "000000000600",
    "000000000585"
]

fig, axes = plt.subplots(5, 2, figsize=(16, 25))
axes = axes.flatten()

for ax, image_id in zip(axes, image_ids):

    image_path = BASE / "images/train" / f"image_{image_id}.jpg"
    label_path = BASE / "labels/train" / f"image_{image_id}.txt"

    image = cv2.imread(str(image_path))

    if image is None:
        ax.set_title(f"{image_id} - Image not found")
        ax.axis("off")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    ax.imshow(image)

    for line in label_path.read_text().splitlines():
        values = line.split()

        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) == 20:
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h

            ax.add_patch(Rectangle(
                (x1, y1), bw * w, bh * h,
                fill=False, edgecolor="red", linewidth=2
            ))

    ax.set_title(image_id)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Inspect the three suspicious labels

from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
image_ids = ["000000000928", "000000000585", "000000000907"]

for image_id in image_ids:
    label_path = BASE / "labels/train" / f"image_{image_id}.txt"

    print(f"\nIMAGE: {image_id}")

    for i, line in enumerate(label_path.read_text().splitlines()):
        values = line.split()
        if len(values) == 5:
            print(f"Line {i}: {line}")

In [ ]:
# Correct the three annotations

from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
LABEL_DIR = BASE / "labels/train"

corrections = {
    "image_000000000928.txt": [
        "20 0.661370 0.567533 0.407140 0.636347"
    ],
    "image_000000000585.txt": [
        "20 0.500000 0.853930 1.000000 0.265172"
    ],
    "image_000000000907.txt": [
        "20 0.562061 0.371875 0.529274 0.709375"
    ]
}

for filename, remove_lines in corrections.items():

    label_path = LABEL_DIR / filename

    # Create a separate backup without overwriting an existing one
    backup_path = label_path.with_name(
        label_path.stem + "_before_cake_cleanup.txt"
    )

    if not backup_path.exists():
        shutil.copy2(label_path, backup_path)

    lines = label_path.read_text().splitlines()

    for target in remove_lines:
        if target not in lines:
            raise ValueError(f"Expected annotation not found: {filename}")

        lines.remove(target)

    label_path.write_text(
        "\n".join(lines) + ("\n" if lines else "")
    )

    print(f"Updated: {filename}")
    print("Remaining annotations:", len(lines))
    print("Backup exists:", backup_path.exists())

print("\nCake annotation corrections completed.")

In [ ]:
# Verify the cake annotations

from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
LABEL_DIR = BASE / "labels/train"

cake_images = set()
cake_boxes = 0

for label_path in LABEL_DIR.glob("*.txt"):
    for line in label_path.read_text().splitlines():
        values = line.split()

        if len(values) != 5:
            continue

        if int(float(values[0])) == 20:
            cake_images.add(label_path.name)
            cake_boxes += 1

print("CAKE AUDIT SUMMARY")
print("Images containing cake:", len(cake_images))
print("Remaining cake annotations:", cake_boxes)

In [ ]:
from pathlib import Path
import re

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
LABEL_DIR = BASE / "labels/train"

cake_images = set()
cake_boxes = 0

# Only count original dataset label files
for label_path in LABEL_DIR.glob("image_*.txt"):

    # Exclude backup files
    if not re.fullmatch(r"image_\d+\.txt", label_path.name):
        continue

    has_cake = False

    for line in label_path.read_text().splitlines():
        values = line.split()

        if len(values) != 5:
            continue

        if int(float(values[0])) == 20:
            cake_boxes += 1
            has_cake = True

    if has_cake:
        cake_images.add(label_path.name)

print("VERIFIED CAKE AUDIT")
print("Images containing cake:", len(cake_images))
print("Remaining cake annotations:", cake_boxes)

In [ ]:
# bowl annotation audit (class ID 18).

from pathlib import Path
import re

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
LABEL_DIR = BASE / "labels/train"

CLASS_ID = 18  # Bowl

bowl_images = set()
bowl_boxes = 0
small_boxes = []
large_boxes = []

for label_path in LABEL_DIR.glob("image_*.txt"):

    # Exclude backup files
    if not re.fullmatch(r"image_\d+\.txt", label_path.name):
        continue

    has_bowl = False

    for line_no, line in enumerate(label_path.read_text().splitlines()):
        values = line.split()

        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) != CLASS_ID:
            continue

        has_bowl = True
        bowl_boxes += 1

        area = bw * bh

        if area < 0.001:
            small_boxes.append((label_path.name, line_no, round(area, 6)))

        if area > 0.25:
            large_boxes.append((label_path.name, line_no, round(area, 4)))

    if has_bowl:
        bowl_images.add(label_path.name)

print("BOWL ANNOTATION AUDIT")
print("Images containing bowls:", len(bowl_images))
print("Total bowl annotations:", bowl_boxes)

print("\nVery small boxes:", len(small_boxes))
for item in small_boxes:
    print(item)

print("\nUnusually large boxes:", len(large_boxes))
for item in large_boxes:
    print(item)

In [ ]:
# Visually inspect suspicious annotations

from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_ids = [
    "000000000456",
    "000000000272",
    "000000001023",
    "000000000465",
    "000000000001",
    "000000001250",
    "000000000250",
    "000000000552",
    "000000000601",
    "000000000305"
]

fig, axes = plt.subplots(5, 2, figsize=(16, 24))
axes = axes.flatten()

for ax, image_id in zip(axes, image_ids):

    image_path = BASE / "images/train" / f"image_{image_id}.jpg"
    label_path = BASE / "labels/train" / f"image_{image_id}.txt"

    image = cv2.imread(str(image_path))

    if image is None:
        ax.set_title(f"{image_id} - Image not found")
        ax.axis("off")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    ax.imshow(image)

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) == 18:

            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h

            ax.add_patch(Rectangle(
                (x1, y1),
                bw * w,
                bh * h,
                fill=False,
                edgecolor="red",
                linewidth=2
            ))

    ax.set_title(image_id)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Verify the questionable annotations

from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
LABEL_DIR = BASE / "labels/train"

image_ids = [
    "000000000456",
    "000000000272",
    "000000001023",
    "000000000001"
]

for image_id in image_ids:
    label_path = LABEL_DIR / f"image_{image_id}.txt"

    print(f"\nIMAGE: {image_id}")

    for i, line in enumerate(label_path.read_text().splitlines()):
        values = line.split()

        if len(values) == 5 and int(float(values[0])) == 18:
            print(f"Line {i}: {line}")

In [ ]:
# Zoom into the suspicious objects

from pathlib import Path
import cv2
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_ids = [
    "000000000456",
    "000000000272",
    "000000001023",
    "000000000001"
]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, image_id in zip(axes, image_ids):

    image_path = BASE / "images/train" / f"image_{image_id}.jpg"
    image = cv2.imread(str(image_path))

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    label_path = BASE / "labels/train" / f"image_{image_id}.txt"

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) != 18:
            continue

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        # Expand crop around the annotated object
        pad_x = max(30, int((x2 - x1) * 2))
        pad_y = max(30, int((y2 - y1) * 2))

        crop = image[
            max(0, y1-pad_y):min(h, y2+pad_y),
            max(0, x1-pad_x):min(w, x2+pad_x)
        ]

        if crop.size > 0:
            ax.imshow(crop)

    ax.set_title(image_id)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Display the complete images

from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_ids = [
    "000000000456",
    "000000000272",
    "000000001023",
    "000000000001"
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for ax, image_id in zip(axes, image_ids):

    image_path = BASE / "images/train" / f"image_{image_id}.jpg"
    label_path = BASE / "labels/train" / f"image_{image_id}.txt"

    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    ax.imshow(image)

    for line in label_path.read_text().splitlines():
        values = line.split()

        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) == 18:
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h

            ax.add_patch(Rectangle(
                (x1, y1), bw * w, bh * h,
                fill=False, edgecolor="red", linewidth=2
            ))

    ax.set_title(image_id)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Get the original image dimensions

from pathlib import Path
import cv2
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_path = BASE / "images/train/image_000000000001.jpg"

image = cv2.imread(str(image_path))
h, w = image.shape[:2]

print("Image width:", w)
print("Image height:", h)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Original image - 000000000001")
plt.axis("off")
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_path = BASE / "images/train/image_000000000001.jpg"

image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)

# Proposed bounding boxes: x1, y1, x2, y2
boxes = [
    (0, 45, 430, 360),    # Left pink container
    (300, 0, 640, 320),  # Right pink container
    (0, 235, 610, 480)   # Yellow container
]

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)

for i, (x1, y1, x2, y2) in enumerate(boxes):
    ax.add_patch(Rectangle(
        (x1, y1), x2-x1, y2-y1,
        fill=False, edgecolor="red", linewidth=2
    ))
    ax.text(x1+5, y1+15, f"Container {i+1}", color="red",
            fontsize=10, weight="bold")

ax.set_title("Proposed corrected bowl annotations")
ax.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import shutil
import cv2

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_path = BASE / "images/train/image_000000000001.jpg"
label_path = BASE / "labels/train/image_000000000001.txt"

# Read image dimensions
image = cv2.imread(str(image_path))
h, w = image.shape[:2]

# Preserve original annotation
backup_path = label_path.with_name(
    "image_000000000001_before_bowl_correction.txt"
)

if not backup_path.exists():
    shutil.copy2(label_path, backup_path)

# Proposed pixel coordinates: x1, y1, x2, y2
boxes = [
    (0, 45, 430, 360),
    (300, 0, 640, 320),
    (0, 235, 610, 480)
]

# Read existing annotations and preserve other classes
lines = label_path.read_text().splitlines()

lines = [
    line for line in lines
    if not (line.strip() and int(line.split()[0]) == 18)
]

# Convert new boxes into YOLO format
for x1, y1, x2, y2 in boxes:

    xc = ((x1 + x2) / 2) / w
    yc = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h

    lines.append(
        f"18 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
    )

label_path.write_text("\n".join(lines) + "\n")

print("Bowl annotations corrected successfully.")
print("Image dimensions:", w, "x", h)
print("New bowl boxes:", len(boxes))
print("Backup preserved:", backup_path.exists())
print("Other annotations preserved.")

In [ ]:
# Visual inspection

from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_ids = [
    "000000001250",
    "000000000250",
    "000000000552",
    "000000000601",
    "000000000305"
]

fig, axes = plt.subplots(3, 2, figsize=(16, 20))
axes = axes.flatten()

for ax, image_id in zip(axes, image_ids):

    image_path = BASE / "images/train" / f"image_{image_id}.jpg"
    label_path = BASE / "labels/train" / f"image_{image_id}.txt"

    image = cv2.imread(str(image_path))

    if image is None:
        ax.set_title(f"{image_id} - Image not found")
        ax.axis("off")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    ax.imshow(image)

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) == 18:
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h

            ax.add_patch(Rectangle(
                (x1, y1), bw * w, bh * h,
                fill=False,
                edgecolor="red",
                linewidth=2
            ))

    ax.set_title(image_id)
    ax.axis("off")

axes[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Let's correct 000000000601 first.

from pathlib import Path
import cv2

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_path = BASE / "images/train/image_000000000601.jpg"
label_path = BASE / "labels/train/image_000000000601.txt"

image = cv2.imread(str(image_path))
h, w = image.shape[:2]

print("Image dimensions:", w, "x", h)

for i, line in enumerate(label_path.read_text().splitlines()):
    if int(line.split()[0]) == 18:
        print(f"Line {i}: {line}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_path = BASE / "images/train/image_000000000601.jpg"
image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)

ax.add_patch(Rectangle(
    (280, 185), 330, 205,
    fill=False, edgecolor="red", linewidth=2
))

ax.set_title("Proposed blue bowl bounding box")
ax.axis("off")
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_path = BASE / "images/train/image_000000000601.jpg"
image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)

ax.add_patch(Rectangle(
    (280, 155), 335, 200,
    fill=False, edgecolor="red", linewidth=2
))

ax.set_title("Revised blue bowl bounding box")
ax.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

label_path = BASE / "labels/train/image_000000000601.txt"
backup_path = label_path.with_name(
    "image_000000000601_before_bowl_correction.txt"
)

# Preserve original annotation
if not backup_path.exists():
    shutil.copy2(label_path, backup_path)

# Replace only the bowl annotation
lines = label_path.read_text().splitlines()

lines = [
    line for line in lines
    if not (line.strip() and int(line.split()[0]) == 18)
]

# Revised pixel coordinates: x1=280, y1=155, x2=615, y2=355
w, h = 640, 480

xc = (280 + 615) / 2 / w
yc = (155 + 355) / 2 / h
bw = (615 - 280) / w
bh = (355 - 155) / h

lines.append(f"18 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

label_path.write_text("\n".join(lines) + "\n")

print("Bowl annotation corrected!")
print("New annotation:", lines[-1])
print("Backup exists:", backup_path.exists())

In [ ]:
# Inspect 000000000305

from pathlib import Path
import cv2

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_path = BASE / "images/train/image_000000000305.jpg"
label_path = BASE / "labels/train/image_000000000305.txt"

image = cv2.imread(str(image_path))
h, w = image.shape[:2]

print("Image dimensions:", w, "x", h)

for i, line in enumerate(label_path.read_text().splitlines()):
    values = line.split()

    if len(values) == 5 and int(values[0]) == 18:
        print(f"Line {i}: {line}")

In [ ]:
# potted plant audit (class ID 23).

from pathlib import Path
import re

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
LABEL_DIR = BASE / "labels/train"

CLASS_ID = 23  # Potted plant

plant_images = set()
plant_boxes = 0
small_boxes = []
large_boxes = []

for label_path in LABEL_DIR.glob("image_*.txt"):

    # Exclude backup files
    if not re.fullmatch(r"image_\d+\.txt", label_path.name):
        continue

    has_plant = False

    for line_no, line in enumerate(label_path.read_text().splitlines()):

        values = line.split()

        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) != CLASS_ID:
            continue

        has_plant = True
        plant_boxes += 1

        area = bw * bh

        if area < 0.001:
            small_boxes.append((label_path.name, line_no, round(area, 6)))

        if area > 0.25:
            large_boxes.append((label_path.name, line_no, round(area, 4)))

    if has_plant:
        plant_images.add(label_path.name)

print("POTTED PLANT ANNOTATION AUDIT")
print("Images containing plants:", len(plant_images))
print("Total plant annotations:", plant_boxes)

print("\nVery small boxes:", len(small_boxes))
for item in small_boxes:
    print(item)

print("\nUnusually large boxes:", len(large_boxes))
for item in large_boxes:
    print(item)

In [ ]:
# Visual inspection

from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_ids = [
    "000000000003",
    "000000000521",
    "000000000800",
    "000000001289",
    "000000000283",
    "000000000728",
    "000000001357",
    "000000001044",
    "000000001194",
    "000000001379"
]

fig, axes = plt.subplots(5, 2, figsize=(16, 24))
axes = axes.flatten()

for ax, image_id in zip(axes, image_ids):

    image_path = BASE / "images/train" / f"image_{image_id}.jpg"
    label_path = BASE / "labels/train" / f"image_{image_id}.txt"

    image = cv2.imread(str(image_path))

    if image is None:
        ax.set_title(f"{image_id} - Image not found")
        ax.axis("off")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    ax.imshow(image)

    for line in label_path.read_text().splitlines():

        values = line.split()

        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        if int(cls) == 23:

            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h

            ax.add_patch(Rectangle(
                (x1, y1),
                bw * w,
                bh * h,
                fill=False,
                edgecolor="red",
                linewidth=2
            ))

    ax.set_title(image_id)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Verify the two annotations before removal

from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")
files = [
    "image_000000000521.txt",
    "image_000000001194.txt",
    "image_000000001379.txt"
]

for filename in files:
    path = BASE / "labels/train" / filename

    print(f"\n{filename}")

    for i, line in enumerate(path.read_text().splitlines()):
        values = line.split()

        if len(values) == 5 and int(values[0]) == 23:
            print(f"Line {i}: {line}")

In [ ]:
from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

files = [
    "image_000000000521.txt",
    "image_000000001194.txt"
]

for filename in files:

    label_path = BASE / "labels/train" / filename
    backup_path = label_path.with_name(
        label_path.stem + ".backup_before_plant_cleanup"
    )

    # Preserve original label file
    shutil.copy2(label_path, backup_path)

    lines = label_path.read_text().splitlines()

    # Remove only class ID 23 (potted plant)
    updated_lines = [
        line for line in lines
        if not (line.strip() and line.split()[0] == "23")
    ]

    label_path.write_text(
        "\n".join(updated_lines) + ("\n" if updated_lines else "")
    )

    print(f"\nUpdated: {filename}")
    print("Removed plant annotations:", len(lines) - len(updated_lines))
    print("Other annotations preserved:", len(updated_lines))
    print("Backup exists:", backup_path.exists())

In [ ]:
# Check the small plant annotation

import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Rectangle

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

image_path = BASE / "images/train/image_000000001379.jpg"
label_path = BASE / "labels/train/image_000000001379.txt"

image = cv2.imread(str(image_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

h, w = image.shape[:2]

plt.figure(figsize=(10, 8))
plt.imshow(image)

for line in label_path.read_text().splitlines():
    values = line.split()

    if len(values) != 5 or int(values[0]) != 23:
        continue

    _, xc, yc, bw, bh = map(float, values)

    x1 = (xc - bw / 2) * w
    y1 = (yc - bh / 2) * h

    plt.gca().add_patch(
        Rectangle((x1, y1), bw * w, bh * h,
                  fill=False, edgecolor="red", linewidth=2)
    )

plt.title("Potted plant annotation verification")
plt.axis("off")
plt.show()

In [ ]:
# Dataset-wide integrity check

from pathlib import Path
from PIL import Image

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

CLASS_COUNT = 25
SPLITS = ["train", "val", "test"]

total_images = 0
total_labels = 0
total_boxes = 0
errors = []

for split in SPLITS:

    image_dir = BASE / "images" / split
    label_dir = BASE / "labels" / split

    images = list(image_dir.glob("*.jpg"))
    labels = list(label_dir.glob("*.txt"))

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    missing_labels = image_stems - label_stems
    missing_images = label_stems - image_stems

    split_boxes = 0

    for label_path in labels:

        for line_number, line in enumerate(
            label_path.read_text().splitlines(), start=1
        ):

            values = line.split()

            if len(values) != 5:
                errors.append(
                    f"{split}/{label_path.name}: Invalid column count at line {line_number}"
                )
                continue

            try:
                class_id = int(values[0])
                xc, yc, bw, bh = map(float, values[1:])
            except ValueError:
                errors.append(
                    f"{split}/{label_path.name}: Invalid values at line {line_number}"
                )
                continue

            if not 0 <= class_id < CLASS_COUNT:
                errors.append(
                    f"{split}/{label_path.name}: Invalid class ID {class_id}"
                )

            if not all(0 <= v <= 1 for v in [xc, yc, bw, bh]):
                errors.append(
                    f"{split}/{label_path.name}: Coordinates outside 0-1"
                )

            if bw <= 0 or bh <= 0:
                errors.append(
                    f"{split}/{label_path.name}: Zero or negative box size"
                )

            if xc - bw / 2 < 0 or xc + bw / 2 > 1:
                errors.append(
                    f"{split}/{label_path.name}: Bounding box exceeds image width"
                )

            if yc - bh / 2 < 0 or yc + bh / 2 > 1:
                errors.append(
                    f"{split}/{label_path.name}: Bounding box exceeds image height"
                )

            split_boxes += 1

    for name in missing_labels:
        errors.append(f"{split}: Missing label for {name}")

    for name in missing_images:
        errors.append(f"{split}: Label has no matching image: {name}")

    total_images += len(images)
    total_labels += len(labels)
    total_boxes += split_boxes

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Label files:", len(labels))
    print("Bounding boxes:", split_boxes)
    print("Images without labels:", len(missing_labels))
    print("Labels without images:", len(missing_images))

print("\n========== FINAL DATASET REPORT ==========")
print("Total images:", total_images)
print("Total label files:", total_labels)
print("Total bounding boxes:", total_boxes)
print("Total errors:", len(errors))

if errors:
    print("\nFirst 30 errors:")
    for error in errors[:30]:
        print(error)
else:
    print("\nSUCCESS: All annotation integrity checks passed!")

In [ ]:
# Inspect the errors

from pathlib import Path
from collections import Counter, defaultdict

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

error_counts = Counter()
class_errors = Counter()
invalid_details = []
orphan_labels = []

for split in ["train", "val", "test"]:

    image_dir = BASE / "images" / split
    label_dir = BASE / "labels" / split

    image_stems = {p.stem for p in image_dir.glob("*.jpg")}

    for label_path in label_dir.glob("*.txt"):

        if label_path.stem not in image_stems:
            orphan_labels.append(label_path.name)
            continue

        for line_no, line in enumerate(
            label_path.read_text().splitlines(), 1
        ):

            values = line.split()

            if len(values) != 5:
                continue

            cls, xc, yc, bw, bh = map(float, values)

            left = xc - bw / 2
            right = xc + bw / 2
            top = yc - bh / 2
            bottom = yc + bh / 2

            problems = []

            if left < 0:
                problems.append("left")
            if right > 1:
                problems.append("right")
            if top < 0:
                problems.append("top")
            if bottom > 1:
                problems.append("bottom")

            if problems:
                error_counts.update(problems)
                class_errors[int(cls)] += 1

                invalid_details.append({
                    "split": split,
                    "file": label_path.name,
                    "line": line_no,
                    "class": int(cls),
                    "issues": problems,
                    "coordinates": (xc, yc, bw, bh),
                    "overflow": (
                        round(max(0, -left), 5),
                        round(max(0, right - 1), 5),
                        round(max(0, -top), 5),
                        round(max(0, bottom - 1), 5)
                    )
                })

print("BOUNDARY ERROR COUNTS")
print(error_counts)

print("\nERRORS BY CLASS")
for cls, count in sorted(class_errors.items()):
    print(cls, ":", count)

print("\nFIRST 25 INVALID ANNOTATIONS")
for item in invalid_details[:25]:
    print(item)

print("\nORPHAN LABEL FILES")
for filename in orphan_labels:
    print(filename)

print("\nTOTAL INVALID ANNOTATIONS:", len(invalid_details))
print("TOTAL ORPHAN LABELS:", len(orphan_labels))

In [ ]:
# Measure the actual boundary errors

from pathlib import Path
from collections import Counter

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

thresholds = [1e-6, 1e-4, 1e-3, 1e-2]
counts = Counter()
max_overflow = 0
worst_cases = []

for split in ["train", "val", "test"]:

    label_dir = BASE / "labels" / split

    for label_path in label_dir.glob("*.txt"):

        image_path = BASE / "images" / split / f"{label_path.stem}.jpg"

        if not image_path.exists():
            continue

        for line_no, line in enumerate(label_path.read_text().splitlines(), 1):

            values = line.split()

            if len(values) != 5:
                continue

            cls, xc, yc, bw, bh = map(float, values)

            overflows = {
                "left": max(0, -(xc - bw / 2)),
                "right": max(0, xc + bw / 2 - 1),
                "top": max(0, -(yc - bh / 2)),
                "bottom": max(0, yc + bh / 2 - 1)
            }

            overflow = max(overflows.values())

            if overflow > 0:
                max_overflow = max(max_overflow, overflow)

                for threshold in thresholds:
                    if overflow > threshold:
                        counts[f">{threshold}"] += 1

                worst_cases.append((
                    overflow, split, label_path.name,
                    line_no, int(cls), overflows
                ))

print("Maximum overflow:", f"{max_overflow:.10f}")

print("\nERRORS ABOVE EACH THRESHOLD")
for threshold in thresholds:
    print(f">{threshold}: {counts[f'>{threshold}']}")

print("\n20 LARGEST OVERFLOWS")
for item in sorted(worst_cases, reverse=True)[:20]:
    print(item)

In [ ]:
from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

LABEL_DIR = BASE / "labels/train"
BACKUP_DIR = BASE / "_annotation_backups/train"

BACKUP_DIR.mkdir(parents=True, exist_ok=True)

backup_files = [
    "image_000000000928_before_cake_cleanup.txt",
    "image_000000000585_before_cake_cleanup.txt",
    "image_000000000907_before_cake_cleanup.txt",
    "image_000000000001_before_bowl_correction.txt",
    "image_000000000601_before_bowl_correction.txt"
]

for filename in backup_files:
    source = LABEL_DIR / filename
    destination = BACKUP_DIR / filename

    if source.exists():
        shutil.move(str(source), str(destination))
        print("Moved:", filename)
    else:
        print("Not found:", filename)

print("\nBackup files preserved:", len(list(BACKUP_DIR.glob("*.txt"))))

In [ ]:
# Final dataset integrity check

from pathlib import Path
import re

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

total_images = 0
total_labels = 0
total_boxes = 0
errors = []

for split in ["train", "val", "test"]:

    image_dir = BASE / "images" / split
    label_dir = BASE / "labels" / split

    images = {
        p.stem for p in image_dir.glob("*.jpg")
    }

    labels = {
        p.stem for p in label_dir.glob("*.txt")
        if re.fullmatch(r"image_\d+", p.stem)
    }

    total_images += len(images)
    total_labels += len(labels)

    for name in images - labels:
        errors.append(f"Missing label: {split}/{name}")

    for name in labels - images:
        errors.append(f"Orphan label: {split}/{name}")

    split_boxes = 0

    for label_path in label_dir.glob("*.txt"):

        if not re.fullmatch(r"image_\d+\.txt", label_path.name):
            continue

        for line in label_path.read_text().splitlines():

            values = line.split()

            if len(values) != 5:
                errors.append(f"Invalid format: {label_path.name}")
                continue

            cls, xc, yc, bw, bh = map(float, values)

            if not (0 <= int(cls) < 25):
                errors.append(f"Invalid class: {label_path.name}")

            if not all(0 <= v <= 1 for v in [xc, yc, bw, bh]):
                errors.append(f"Invalid coordinates: {label_path.name}")

            if bw <= 0 or bh <= 0:
                errors.append(f"Invalid box size: {label_path.name}")

            tolerance = 1e-6

            if (
                xc - bw/2 < -tolerance or
                xc + bw/2 > 1+tolerance or
                yc - bh/2 < -tolerance or
                yc + bh/2 > 1+tolerance
            ):
                errors.append(f"Meaningful boundary error: {label_path.name}")

            split_boxes += 1

    total_boxes += split_boxes

    print(f"\n{split.upper()}:")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Boxes:", split_boxes)

print("\n========== FINAL REPORT ==========")
print("Total images:", total_images)
print("Total labels:", total_labels)
print("Total boxes:", total_boxes)
print("Errors:", len(errors))

if errors:
    print("\nFirst 20 errors:")
    for error in errors[:20]:
        print(error)
else:
    print("\nSUCCESS: Dataset integrity check passed!")

In [ ]:
# Inspect the label file

from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

label_path = BASE / "labels/train/image_000000001147.txt"
image_path = BASE / "images/train/image_000000001147.jpg"

print("Image exists:", image_path.exists())
print("Label exists:", label_path.exists())
print("Label file size:", label_path.stat().st_size, "bytes")

print("Label content:")
print(repr(label_path.read_text()))

In [ ]:
# Create a separate YOLO dataset configuration

from pathlib import Path
import yaml

BASE = Path("/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1")

class_names = [
    "person", "bicycle", "car", "motorcycle", "airplane",
    "bus", "truck", "traffic light", "stop sign", "bench",
    "bird", "cat", "dog", "horse", "cow",
    "elephant", "bottle", "cup", "bowl", "pizza",
    "cake", "chair", "couch", "potted plant", "bed"
]

data_config = {
    "path": str(BASE),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(class_names),
    "names": class_names
}

yaml_path = BASE / "data_cleaned.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("Configuration created:", yaml_path)
print("Number of classes:", len(class_names))
print("Training path:", BASE / "images/train")
print("Validation path:", BASE / "images/val")
print("Test path:", BASE / "images/test")

In [ ]:
# Verify the configuration

from pathlib import Path
import yaml

yaml_path = Path(
    "/content/drive/MyDrive/SmartVision_AI/dataset/detection_cleaned_v1/data_cleaned.yaml"
)

with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

print("YAML validation:", "PASSED" if config["nc"] == len(config["names"]) else "FAILED")
print("Number of classes:", config["nc"])
print("First class:", config["names"][0])
print("Last class:", config["names"][-1])

for split in ["train", "val", "test"]:
    folder = Path(config["path"]) / config[split]
    print(f"{split.upper()} directory exists:", folder.exists())

In [ ]:
pip install ultralytics

In [ ]:
# Check GPU and existing checkpoint

import torch
from pathlib import Path
from ultralytics import YOLO

# Existing best checkpoint
BEST_MODEL = Path(
    "/content/drive/MyDrive/SmartVision_AI/results/yolo_training/"
    "yolov8s_25class_gpu/weights/best.pt"
)

# Cleaned dataset configuration
DATA_YAML = Path(
    "/content/drive/MyDrive/SmartVision_AI/dataset/"
    "detection_cleaned_v1/data_cleaned.yaml"
)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Checkpoint exists:", BEST_MODEL.exists())
print("Dataset YAML exists:", DATA_YAML.exists())

if not BEST_MODEL.exists():
    raise FileNotFoundError("Existing best YOLOv8s checkpoint not found!")

# Load existing trained model
model = YOLO(str(BEST_MODEL))

print("\nExisting YOLOv8s model loaded successfully.")
print("Model classes:", len(model.names))
print("Class names:", model.names)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI")

CHECKPOINT = BASE / "results/yolo_training/yolov8s_25class_gpu/weights/best.pt"

DATA_YAML = BASE / "dataset/detection_cleaned_v1/data_cleaned.yaml"

print("Checkpoint exists:", CHECKPOINT.exists())
print("Cleaned dataset exists:", DATA_YAML.exists())

In [ ]:
# Install and import YOLO

!pip install -q ultralytics

import torch
from ultralytics import YOLO
from pathlib import Path
import yaml

BASE = Path("/content/drive/MyDrive/SmartVision_AI")

CHECKPOINT = BASE / "results/yolo_training/yolov8s_25class_gpu/weights/best.pt"

DATA_YAML = BASE / "dataset/detection_cleaned_v1/data_cleaned.yaml"

print("Ultralytics installed successfully")
print("CUDA available:", torch.cuda.is_available())
print("Checkpoint exists:", CHECKPOINT.exists())
print("Dataset exists:", DATA_YAML.exists())

In [ ]:
# Load your existing model

model = YOLO(str(CHECKPOINT))

print("Model loaded successfully")
print("Number of classes:", len(model.names))

In [ ]:
# Evaluate the existing model on the cleaned validation dataset

# Evaluate existing YOLOv8s model on cleaned validation dataset

results = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=8,
    device="cpu",
    plots=True
)

print("\nCLEANED VALIDATION RESULTS")
print("Precision:", round(results.box.mp, 4))
print("Recall:", round(results.box.mr, 4))
print("mAP50:", round(results.box.map50, 4))
print("mAP50-95:", round(results.box.map, 4))

In [ ]:
# Configure CPU training

import torch
from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI")

DATA_YAML = BASE / "dataset/detection_cleaned_v1/data_cleaned.yaml"

CHECKPOINT = BASE / "results/yolo_training/yolov8s_25class_gpu/weights/best.pt"

PROJECT_DIR = BASE / "results/yolo_training"

RUN_NAME = "yolov8s_25class_cleaned_v1_cpu"

print("CUDA available:", torch.cuda.is_available())
print("Dataset exists:", DATA_YAML.exists())
print("Checkpoint exists:", CHECKPOINT.exists())
print("Training output:", PROJECT_DIR / RUN_NAME)

In [ ]:
# Start CPU training

from ultralytics import YOLO

# Load existing best checkpoint
model = YOLO(str(CHECKPOINT))

# Train on cleaned dataset
results = model.train(
    data=str(DATA_YAML),
    epochs=20,
    imgsz=640,
    batch=4,
    optimizer="AdamW",
    lr0=0.001,
    device="cpu",
    workers=2,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=False,
    patience=10,
    plots=True,
    save=True
)

print("Training completed!")
print("Results saved at:", PROJECT_DIR / RUN_NAME)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/results/yolo_training/"
    "yolov8s_25class_cleaned_v1_cpu"
)

for filename in ["best.pt", "last.pt", "results.csv"]:
    path = RUN_DIR / filename
    print(filename, "EXISTS" if path.exists() else "NOT FOUND")

In [ ]:
# Inspect saved training progress

import pandas as pd
from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/results/yolo_training/"
    "yolov8s_25class_cleaned_v1_cpu"
)

# List saved files
print("FILES IN TRAINING DIRECTORY:")
for item in RUN_DIR.rglob("*"):
    if item.is_file():
        print(item.relative_to(RUN_DIR))

# Read training history
csv_path = RUN_DIR / "results.csv"

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("\nTOTAL RECORDED EPOCHS:", len(df))
    print("\nLAST RECORDED EPOCHS:")
    print(df.tail(5).to_string(index=False))

In [ ]:
pip install ultralytics

In [ ]:
# Restore the dataset path

from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI")

DATA_YAML = (
    BASE / "dataset/detection_cleaned_v1/data_cleaned.yaml"
)

print("Dataset configuration exists:", DATA_YAML.exists())
print("Dataset path:", DATA_YAML)

In [ ]:
# Evaluate the recovered best checkpoint

from ultralytics import YOLO
from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/results/yolo_training/"
    "yolov8s_25class_cleaned_v1_cpu"
)

BEST_CHECKPOINT = RUN_DIR / "weights" / "best.pt"

print("Best checkpoint exists:", BEST_CHECKPOINT.exists())

# Load recovered best model
cleaned_model = YOLO(str(BEST_CHECKPOINT))

# Evaluate on cleaned validation dataset
cleaned_results = cleaned_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=4,
    device="cpu"
)

print("\nRECOVERED MODEL RESULTS")
print("Precision:", round(cleaned_results.box.mp, 4))
print("Recall:", round(cleaned_results.box.mr, 4))
print("mAP50:", round(cleaned_results.box.map50, 4))
print("mAP50-95:", round(cleaned_results.box.map, 4))

In [ ]:
# compete training from 18 epochs

from ultralytics import YOLO
from pathlib import Path

LAST_CHECKPOINT = Path(
    "/content/drive/MyDrive/SmartVision_AI/results/yolo_training/"
    "yolov8s_25class_cleaned_v1_cpu/weights/last.pt"
)

print("Last checkpoint exists:", LAST_CHECKPOINT.exists())

if not LAST_CHECKPOINT.exists():
    raise FileNotFoundError("last.pt checkpoint not found!")

# Load the last training checkpoint
resume_model = YOLO(str(LAST_CHECKPOINT))

# Resume the interrupted training
resume_results = resume_model.train(resume=True)

print("Training resumed and completed!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/results/yolo_training/"
    "yolov8s_25class_cleaned_v1_cpu"
)

for filename in ["best.pt", "last.pt", "results.csv"]:
    path = RUN_DIR / "weights" / filename if filename.endswith(".pt") else RUN_DIR / filename
    print(filename, "EXISTS" if path.exists() else "NOT FOUND")

In [ ]:
# Check the actual completed epoch

import pandas as pd
from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/SmartVision_AI/results/yolo_training/"
    "yolov8s_25class_cleaned_v1_cpu"
)

df = pd.read_csv(RUN_DIR / "results.csv")

print("Completed epochs:", len(df))
print("\nLatest training results:")
print(df.tail(3)[[
    "epoch",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)"
]].to_string(index=False))

print("\nBest checkpoint:", (RUN_DIR / "weights/best.pt").exists())
print("Last checkpoint:", (RUN_DIR / "weights/last.pt").exists())

In [ ]:
# Final test evaluation

from ultralytics import YOLO
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/SmartVision_AI")

DATA_YAML = BASE / "dataset/detection_cleaned_v1/data_cleaned.yaml"

# Original YOLOv8s checkpoint
ORIGINAL_PATH = (
    BASE / "results/yolo_training/"
    "yolov8s_25class_gpu/weights/best.pt"
)

# Cleaned YOLOv8s checkpoint
CLEANED_PATH = (
    BASE / "results/yolo_training/"
    "yolov8s_25class_cleaned_v1_cpu/weights/best.pt"
)

# Load both models
original_model = YOLO(str(ORIGINAL_PATH))
cleaned_model = YOLO(str(CLEANED_PATH))

# Evaluate original model
print("Evaluating ORIGINAL YOLOv8s...")

original_results = original_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=4,
    device="cpu"
)

# Evaluate cleaned model
print("\nEvaluating CLEANED YOLOv8s...")

cleaned_results = cleaned_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=4,
    device="cpu"
)

# Compare results
comparison = pd.DataFrame({
    "Metric": ["Precision", "Recall", "mAP50", "mAP50-95"],
    "Original YOLOv8s": [
        original_results.box.mp,
        original_results.box.mr,
        original_results.box.map50,
        original_results.box.map
    ],
    "Cleaned YOLOv8s": [
        cleaned_results.box.mp,
        cleaned_results.box.mr,
        cleaned_results.box.map50,
        cleaned_results.box.map
    ]
})

comparison["Original (%)"] = comparison["Original YOLOv8s"] * 100
comparison["Cleaned (%)"] = comparison["Cleaned YOLOv8s"] * 100

print("\nFINAL TEST COMPARISON")
display(comparison[["Metric", "Original (%)", "Cleaned (%)"]].round(2))

- Metric	Original YOLOv8s	Cleaned YOLOv8s	Winner
- Precision	47.50%	42.77%	Original
- Recall	37.16%	32.02%	Original
- mAP50	36.84%	32.47%	Original
- mAP50-95	22.15%	21.03%	Original

I selected Original YOLOv8s

Final test performance:
- Precision: 47.50%
- Recall: 37.16%
- mAP50: 36.84%
- mAP50-95: 22.15%
- Classes: 25

# Phase 4: Model Integration & Pipeline Development

Step 4.1: Inference Pipeline Design

In [ ]:
# Install required libraries
!pip install -q ultralytics opencv-python-headless matplotlib pillow

# Import libraries
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from ultralytics import YOLO
from pathlib import Path

# Project directory
BASE = Path("/content/drive/MyDrive/SmartVision_AI")

# Selected YOLOv8s model
MODEL_PATH = (
    BASE / "results/yolo_training/"
    "yolov8s_25class_gpu/weights/best.pt"
)

# Load model
detection_model = YOLO(str(MODEL_PATH))

print("SmartVision AI - Detection Pipeline")
print("-----------------------------------")
print("Model loaded:", MODEL_PATH.exists())
print("Architecture:", "YOLOv8s")
print("Number of classes:", len(detection_model.names))
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")
print("OpenCV version:", cv2.__version__)
print("PyTorch version:", torch.__version__)

Step 4.1.2 – Build the object detection prediction function

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def detect_objects(image, confidence_threshold=0.50):

    # Convert input to RGB
    if isinstance(image, Image.Image):
        image = np.array(image.convert("RGB"))

    elif isinstance(image, str):
        image = np.array(Image.open(image).convert("RGB"))

    # Run YOLO detection
    results = detection_model.predict(
        source=image,
        conf=confidence_threshold,
        iou=0.45,
        device="cpu",
        verbose=False
    )

    result = results[0]

    # Extract detections
    detections = []

    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        detections.append({
            "Class ID": class_id,
            "Class Name": detection_model.names[class_id],
            "Confidence": round(confidence, 4),
            "X1": x1,
            "Y1": y1,
            "X2": x2,
            "Y2": y2
        })

    # Create detection DataFrame
    detection_df = pd.DataFrame(detections)

    # Annotated image
    annotated_image = result.plot()

    annotated_image = cv2.cvtColor(
        annotated_image,
        cv2.COLOR_BGR2RGB
    )

    return annotated_image, detection_df


print("Object detection pipeline function created successfully!")

Step 4.1.3 – Test the pipeline

In [ ]:
from pathlib import Path

TEST_IMAGES = BASE / "dataset/detection_cleaned_v1/images/test"

sample_image = next(TEST_IMAGES.glob("*.jpg"))

annotated_image, detection_df = detect_objects(
    sample_image,
    confidence_threshold=0.50
)

plt.figure(figsize=(12, 8))
plt.imshow(annotated_image)
plt.axis("off")
plt.title("SmartVision AI - Object Detection")
plt.show()

print("\nDetected Objects:")
display(detection_df)

print("Total objects detected:", len(detection_df))

In [ ]:
# Debug low-confidence predictions

# Test with a lower confidence threshold
annotated_image, detection_df = detect_objects(
    sample_image,
    confidence_threshold=0.10
)

plt.figure(figsize=(12, 8))
plt.imshow(annotated_image)
plt.axis("off")
plt.title("SmartVision AI - Low Confidence Detection")
plt.show()

print("\nDetected Objects:")
display(detection_df)

print("Total objects detected:", len(detection_df))

Step 4.1.5 – Test different confidence thresholds

In [ ]:
confidence_levels = [0.10, 0.20, 0.30, 0.40, 0.50]

comparison_results = []

for threshold in confidence_levels:

    results = detection_model.predict(
        source=str(sample_image),
        conf=threshold,
        iou=0.45,
        device="cpu",
        verbose=False
    )

    boxes = results[0].boxes

    comparison_results.append({
        "Confidence Threshold": threshold,
        "Objects Detected": len(boxes),
        "Detected Classes": [
            detection_model.names[int(box.cls[0])]
            for box in boxes
        ],
        "Confidence Scores": [
            round(float(box.conf[0]), 3)
            for box in boxes
        ]
    })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

Step 4.1.6 – Improve the inference pipeline

In [ ]:
def detect_objects(image, confidence_threshold=0.25, iou_threshold=0.45):

    if isinstance(image, Image.Image):
        image = np.array(image.convert("RGB"))

    elif isinstance(image, str):
        image = np.array(Image.open(image).convert("RGB"))

    results = detection_model.predict(
        source=image,
        conf=confidence_threshold,
        iou=iou_threshold,
        imgsz=640,
        max_det=100,
        device="cpu",
        verbose=False
    )

    result = results[0]
    detections = []

    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        detections.append({
            "Class ID": class_id,
            "Class Name": detection_model.names[class_id],
            "Confidence": round(confidence, 4),
            "Bounding Box": (x1, y1, x2, y2)
        })

    detection_df = pd.DataFrame(detections)

    annotated_image = cv2.cvtColor(
        result.plot(),
        cv2.COLOR_BGR2RGB
    )

    return annotated_image, detection_df


print("Updated inference pipeline successfully!")

Step 4.1.7 – Test the updated pipeline

In [ ]:
annotated_image, detection_df = detect_objects(
    sample_image
)

# Display detected image
plt.figure(figsize=(12, 8))
plt.imshow(annotated_image)
plt.axis("off")
plt.title("SmartVision AI - Detection Results")
plt.show()

# Display detection details
print("Detection Summary")
display(detection_df)

print("\nTotal objects detected:", len(detection_df))

if len(detection_df) > 0:
    print("\nClass-wise detection count:")
    display(
        detection_df["Class Name"]
        .value_counts()
        .reset_index(name="Count")
    )

# Step 4.2 – Post-processing

In [ ]:
# Bounding-box validation

# Validate detected bounding boxes

image_width = np.array(
    Image.open(sample_image).convert("RGB")
).shape[1]

image_height = np.array(
    Image.open(sample_image).convert("RGB")
).shape[0]

print("Image dimensions:", image_width, "x", image_height)

valid_boxes = 0
invalid_boxes = 0

for _, row in detection_df.iterrows():

    x1, y1, x2, y2 = row["Bounding Box"]

    valid = (
        0 <= x1 < x2 <= image_width and
        0 <= y1 < y2 <= image_height
    )

    if valid:
        valid_boxes += 1
    else:
        invalid_boxes += 1

print("\nBounding Box Validation")
print("Valid boxes:", valid_boxes)
print("Invalid boxes:", invalid_boxes)

print(
    "Validation status:",
    "PASSED" if invalid_boxes == 0 else "FAILED"
)

In [ ]:
# Improve detection result formatting

def generate_detection_summary(detection_df):

    if detection_df.empty:
        return {
            "total_objects": 0,
            "unique_classes": 0,
            "average_confidence": 0,
            "highest_confidence": 0,
            "class_counts": {}
        }

    summary = {
        "total_objects": len(detection_df),
        "unique_classes": detection_df["Class Name"].nunique(),
        "average_confidence": round(
            detection_df["Confidence"].mean(), 4
        ),
        "highest_confidence": round(
            detection_df["Confidence"].max(), 4
        ),
        "class_counts": (
            detection_df["Class Name"]
            .value_counts()
            .to_dict()
        )
    }

    return summary


summary = generate_detection_summary(detection_df)

print("SMARTVISION AI - DETECTION SUMMARY")
print("----------------------------------")

for key, value in summary.items():
    print(f"{key}: {value}")

# Step 4.3 – Performance Optimization

In [ ]:
# Batch image processing

def batch_detect_images(image_paths, confidence_threshold=0.25):

    batch_results = []

    for image_path in image_paths:

        annotated_image, detections = detect_objects(
            image_path,
            confidence_threshold=confidence_threshold
        )

        batch_results.append({
            "Image": Path(image_path).name,
            "Total Objects": len(detections),
            "Detected Classes": (
                detections["Class Name"].tolist()
                if not detections.empty else []
            ),
            "Average Confidence": (
                round(detections["Confidence"].mean(), 4)
                if not detections.empty else 0
            ),
            "Annotated Image": annotated_image,
            "Detection Details": detections
        })

    return batch_results


print("Batch processing function created successfully!")

In [ ]:
# Test batch processing

# Select five sample test images
sample_images = list(TEST_IMAGES.glob("*.jpg"))[:5]

# Process images
batch_results = batch_detect_images(
    sample_images,
    confidence_threshold=0.25
)

# Display summary
batch_summary = pd.DataFrame([
    {
        "Image": item["Image"],
        "Objects Detected": item["Total Objects"],
        "Classes": item["Detected Classes"],
        "Average Confidence": item["Average Confidence"]
    }
    for item in batch_results
])

display(batch_summary)

print("Images processed:", len(batch_results))

In [ ]:
# Check GPU availability

import torch
import platform
import psutil

print("SMARTVISION AI - HARDWARE INFORMATION")
print("--------------------------------------")

print("Processor:", platform.processor())
print("CPU cores:", psutil.cpu_count(logical=True))
print("Available RAM:", round(
    psutil.virtual_memory().available / (1024**3), 2
), "GB")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / (1024**3), 2
    ), "GB")
else:
    print("Running inference on CPU")

In [ ]:
# INT8 Quantization

# Check the current model size

from pathlib import Path

MODEL_PATH = (
    BASE / "results/yolo_training/"
    "yolov8s_25class_gpu/weights/best.pt"
)

model_size_mb = MODEL_PATH.stat().st_size / (1024 * 1024)

print("Original YOLOv8s Model")
print("----------------------")
print("Model size:", round(model_size_mb, 2), "MB")
print("Model format: PyTorch (.pt)")
print("Quantization: Not applied")

In [ ]:
# Export YOLOv8s to ONNX

from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = (
    BASE / "results/yolo_training/"
    "yolov8s_25class_gpu/weights/best.pt"
)

EXPORT_DIR = BASE / "results/model_optimization"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load original model
export_model = YOLO(str(MODEL_PATH))

# Export to ONNX
onnx_path = export_model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=17
)

# Copy exported model to optimization directory
import shutil

onnx_path = Path(onnx_path)
final_onnx_path = EXPORT_DIR / "yolov8s_25class.onnx"

shutil.copy2(onnx_path, final_onnx_path)

print("ONNX export completed!")
print("ONNX model:", final_onnx_path)
print("Model size:", round(
    final_onnx_path.stat().st_size / (1024 * 1024), 2
), "MB")

In [ ]:
# Verify ONNX inference

import onnxruntime as ort
from pathlib import Path

ONNX_PATH = BASE / "results/model_optimization/yolov8s_25class.onnx"

# Check available execution providers
print("ONNX Runtime version:", ort.__version__)
print("Available providers:", ort.get_available_providers())

# Load ONNX model
onnx_session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"]
)

# Display model input and output information
print("\nONNX model loaded successfully!")
print("Input name:", onnx_session.get_inputs()[0].name)
print("Input shape:", onnx_session.get_inputs()[0].shape)
print("Output name:", onnx_session.get_outputs()[0].name)
print("Output shape:", onnx_session.get_outputs()[0].shape)

In [ ]:
# Compare PyTorch vs ONNX inference

import time
import numpy as np
from PIL import Image
import torch

# Select the same test image
TEST_IMAGE = TEST_IMAGES / "image_000000001234.jpg"

# Load and preprocess image using Ultralytics preprocessing
image = Image.open(TEST_IMAGE).convert("RGB")
image_array = np.array(image)

# Prepare input using the ONNX model's preprocessing
from ultralytics.data.augment import LetterBox

letterbox = LetterBox(new_shape=(640, 640))
processed = letterbox(image=image_array)

input_tensor = processed.transpose(2, 0, 1)
input_tensor = np.ascontiguousarray(input_tensor, dtype=np.float32)
input_tensor = input_tensor[None] / 255.0

# ONNX inference
onnx_times = []

for _ in range(5):
    start = time.perf_counter()
    onnx_output = onnx_session.run(
        ["output0"],
        {"images": input_tensor}
    )
    onnx_times.append(time.perf_counter() - start)

onnx_avg = np.mean(onnx_times[1:]) * 1000

# PyTorch inference
pytorch_times = []

for _ in range(5):
    start = time.perf_counter()
    pytorch_result = detection_model.predict(
        source=image_array,
        imgsz=640,
        conf=0.25,
        device="cpu",
        verbose=False
    )
    pytorch_times.append(time.perf_counter() - start)

pytorch_avg = np.mean(pytorch_times[1:]) * 1000

print("Inference comparison")
print("--------------------")
print(f"ONNX average:     {onnx_avg:.2f} ms")
print(f"PyTorch average: {pytorch_avg:.2f} ms")

if onnx_avg < pytorch_avg:
    print(f"\nONNX is {pytorch_avg / onnx_avg:.2f}x faster.")
else:
    print(f"\nPyTorch is {onnx_avg / pytorch_avg:.2f}x faster.")

print("\nONNX raw output shape:", onnx_output[0].shape)

In [ ]:
# INT8 quantization

try:
    from onnxruntime.quantization import (
        quantize_static,
        CalibrationDataReader,
        QuantType,
        QuantFormat
    )

    print("ONNX quantization tools imported successfully!")
    print("Static INT8 quantization is available.")

except Exception as e:
    print("Quantization import failed:", e)

In [ ]:
# Prepare calibration images

from pathlib import Path
import numpy as np
from PIL import Image
from ultralytics.data.augment import LetterBox

# Calibration dataset
CALIBRATION_DIR = (
    BASE / "dataset/detection_cleaned_v1/images/train"
)

calibration_paths = sorted(
    CALIBRATION_DIR.glob("*")
)

# Select up to 100 images
calibration_paths = calibration_paths[:100]

def preprocess_calibration_image(image_path):
    image = np.array(
        Image.open(image_path).convert("RGB")
    )

    image = LetterBox(new_shape=(640, 640))(image=image)

    image = image.transpose(2, 0, 1)
    image = np.ascontiguousarray(image, dtype=np.float32)

    return image / 255.0

calibration_images = np.stack([
    preprocess_calibration_image(path)
    for path in calibration_paths
])

print("Calibration images prepared!")
print("Number of images:", len(calibration_images))
print("Input shape:", calibration_images.shape)
print("Data type:", calibration_images.dtype)

- Static quantization
- INT8 activations and weights
- Per-channel weight quantization
- QDQ format (QuantizeLinear/DequantizeLinear nodes)

In [ ]:
# Create the INT8 quantized ONNX model

from onnxruntime.quantization import (
    quantize_static,
    CalibrationDataReader,
    QuantType,
    QuantFormat
)

class YOLOCalibrationReader(CalibrationDataReader):

    def __init__(self, images):
        self.images = images
        self.index = 0

    def get_next(self):
        if self.index >= len(self.images):
            return None

        image = self.images[self.index]
        self.index += 1

        return {"images": image[None, ...]}

    def rewind(self):
        self.index = 0


# Create calibration reader
calibration_reader = YOLOCalibrationReader(calibration_images)

# Define model paths
INT8_PATH = (
    BASE / "results/model_optimization/yolov8s_25class_int8.onnx"
)

# Perform static INT8 quantization
quantize_static(
    model_input=str(ONNX_PATH),
    model_output=str(INT8_PATH),
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=True
)

print("INT8 quantization completed!")

print("Original ONNX size:",
      round(ONNX_PATH.stat().st_size / (1024 * 1024), 2), "MB")

print("INT8 ONNX size:",
      round(INT8_PATH.stat().st_size / (1024 * 1024), 2), "MB")

In [ ]:
# Verify the INT8 model

import onnxruntime as ort

INT8_SESSION = ort.InferenceSession(
    str(INT8_PATH),
    providers=["CPUExecutionProvider"]
)

print("INT8 model loaded successfully!")
print("Input:", INT8_SESSION.get_inputs()[0].shape)
print("Output:", INT8_SESSION.get_outputs()[0].shape)
print("Execution provider:", INT8_SESSION.get_providers())

In [ ]:
# Compare detection quality

import numpy as np

# Run FP32 ONNX inference
fp32_output = onnx_session.run(
    ["output0"],
    {"images": input_tensor}
)[0]

# Run INT8 ONNX inference
int8_output = INT8_SESSION.run(
    ["output0"],
    {"images": input_tensor}
)[0]

# Compare raw predictions
difference = np.abs(fp32_output - int8_output)

print("FP32 output shape:", fp32_output.shape)
print("INT8 output shape:", int8_output.shape)

print("\nRaw output comparison")
print("Mean absolute difference:", round(float(difference.mean()), 6))
print("Maximum absolute difference:", round(float(difference.max()), 6))

print("\nFP32 output range:",
      float(fp32_output.min()), "to", float(fp32_output.max()))

print("INT8 output range:",
      float(int8_output.min()), "to", float(int8_output.max()))

In [ ]:
#  Check the available NMS implementation

import ultralytics
import ultralytics.utils.ops as ops

print("Ultralytics version:", ultralytics.__version__)
print("NMS-related functions:")

print([
    name for name in dir(ops)
    if "nms" in name.lower() or "suppression" in name.lower()
])

In [ ]:
# Compare FP32 and INT8 detections

from ultralytics import YOLO
import pandas as pd

# Load FP32 ONNX model
fp32_model = YOLO(str(ONNX_PATH), task="detect")

# Load INT8 ONNX model
int8_model = YOLO(str(INT8_PATH), task="detect")

# Use the same test image
test_image = str(TEST_IMAGE)

# Run predictions
fp32_results = fp32_model.predict(
    test_image,
    conf=0.25,
    iou=0.45,
    imgsz=640,
    device="cpu",
    verbose=False
)

int8_results = int8_model.predict(
    test_image,
    conf=0.25,
    iou=0.45,
    imgsz=640,
    device="cpu",
    verbose=False
)

# Extract detections
def extract_detections(results, model):
    rows = []

    for box in results[0].boxes:
        class_id = int(box.cls[0])

        rows.append({
            "Class": model.names[class_id],
            "Confidence": round(float(box.conf[0]), 4),
            "Bounding Box": tuple(
                round(float(x), 2)
                for x in box.xyxy[0].tolist()
            )
        })

    return pd.DataFrame(rows)

fp32_df = extract_detections(fp32_results, fp32_model)
int8_df = extract_detections(int8_results, int8_model)

print("FP32 ONNX detections:")
display(fp32_df)

print("\nINT8 ONNX detections:")
display(int8_df)

print("\nDetection counts:")
print("FP32:", len(fp32_df))
print("INT8:", len(int8_df))

INT8 model has a significant detection-quality problem.

In [ ]:
# Check whether INT8 predictions are below the confidence threshold

# Test both models with a very low confidence threshold

fp32_low = fp32_model.predict(
    test_image,
    conf=0.001,
    iou=0.45,
    imgsz=640,
    device="cpu",
    verbose=False
)

int8_low = int8_model.predict(
    test_image,
    conf=0.001,
    iou=0.45,
    imgsz=640,
    device="cpu",
    verbose=False
)

print("FP32 detections:", len(fp32_low[0].boxes))
print("INT8 detections:", len(int8_low[0].boxes))

print("\nINT8 predictions:")
for box in int8_low[0].boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])

    print(
        detection_model.names[class_id],
        "Confidence:", round(confidence, 4)
    )

This confirms that the current INT8 model is not suitable for deployment.

In [ ]:
# Try QUInt8 activation quantization

from onnxruntime.quantization import (
    quantize_static,
    QuantType,
    QuantFormat
)

# Reset calibration reader
calibration_reader.rewind()

# New model path
INT8_UNSIGNED_PATH = (
    BASE / "results/model_optimization/"
    "yolov8s_25class_int8_unsigned.onnx"
)

# Quantize with unsigned activations
quantize_static(
    model_input=str(ONNX_PATH),
    model_output=str(INT8_UNSIGNED_PATH),
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=True
)

print("Unsigned activation quantization completed!")
print(
    "Model size:",
    round(INT8_UNSIGNED_PATH.stat().st_size / (1024 * 1024), 2),
    "MB"
)

In [ ]:
# Load the new model

import onnxruntime as ort

INT8_UNSIGNED_SESSION = ort.InferenceSession(
    str(INT8_UNSIGNED_PATH),
    providers=["CPUExecutionProvider"]
)

print("Unsigned INT8 model loaded successfully!")
print("Input:", INT8_UNSIGNED_SESSION.get_inputs()[0].shape)
print("Output:", INT8_UNSIGNED_SESSION.get_outputs()[0].shape)

In [ ]:
# Compare all three models

from ultralytics import YOLO

# Load unsigned INT8 model
int8_unsigned_model = YOLO(
    str(INT8_UNSIGNED_PATH),
    task="detect"
)

# Run inference
unsigned_results = int8_unsigned_model.predict(
    test_image,
    conf=0.001,
    iou=0.45,
    imgsz=640,
    device="cpu",
    verbose=False
)

# Compare detection counts
print("Detection comparison")
print("--------------------")
print("FP32 ONNX:", len(fp32_low[0].boxes))
print("Signed INT8:", len(int8_low[0].boxes))
print("Unsigned INT8:", len(unsigned_results[0].boxes))

# Display unsigned INT8 predictions
print("\nUnsigned INT8 predictions:")

for box in unsigned_results[0].boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])

    print(
        detection_model.names[class_id],
        "Confidence:", round(confidence, 4)
    )

In [ ]:
# Preprocess the FP32 ONNX model

from onnxruntime.quantization.shape_inference import quant_pre_process

PREPROCESSED_PATH = (
    BASE / "results/model_optimization/"
    "yolov8s_25class_preprocessed.onnx"
)

quant_pre_process(
    input_model=str(ONNX_PATH),
    output_model_path=str(PREPROCESSED_PATH),
    skip_optimization=False,
    skip_onnx_shape=False,
    skip_symbolic_shape=False
)

print("ONNX preprocessing completed!")
print(
    "Preprocessed model size:",
    round(PREPROCESSED_PATH.stat().st_size / (1024 * 1024), 2),
    "MB"
)

In [ ]:
# Retry INT8 quantization

from onnxruntime.quantization import (
    quantize_static,
    QuantType,
    QuantFormat
)

PREPROCESSED_INT8_PATH = (
    BASE / "results/model_optimization/"
    "yolov8s_25class_preprocessed_int8.onnx"
)

calibration_reader.rewind()

quantize_static(
    model_input=str(PREPROCESSED_PATH),
    model_output=str(PREPROCESSED_INT8_PATH),
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=True
)

print("Preprocessed INT8 quantization completed!")
print(
    "Model size:",
    round(PREPROCESSED_INT8_PATH.stat().st_size / (1024 * 1024), 2),
    "MB"
)

new model is approximately 74% smaller than FP32 ONNX

In [ ]:
# Test the preprocessed INT8 model

from ultralytics import YOLO

# Load preprocessed INT8 model
preprocessed_int8_model = YOLO(
    str(PREPROCESSED_INT8_PATH),
    task="detect"
)

# Run inference
preprocessed_int8_results = preprocessed_int8_model.predict(
    test_image,
    conf=0.001,
    iou=0.45,
    imgsz=640,
    device="cpu",
    verbose=False
)

print("Detection comparison")
print("--------------------")
print("FP32 ONNX:", len(fp32_low[0].boxes))
print("Signed INT8:", len(int8_low[0].boxes))
print("Unsigned INT8:", len(unsigned_results[0].boxes))
print(
    "Preprocessed INT8:",
    len(preprocessed_int8_results[0].boxes)
)

print("\nPreprocessed INT8 predictions:")

for box in preprocessed_int8_results[0].boxes:
    class_id = int(box.cls[0])

    print(
        detection_model.names[class_id],
        "Confidence:",
        round(float(box.conf[0]), 4)
    )


# testing OpenVINO.

In [ ]:
# Export YOLO to OpenVINO

from ultralytics import YOLO

# Load the selected original YOLO model
openvino_model = YOLO(str(MODEL_PATH))

# Export to OpenVINO FP32
openvino_path = openvino_model.export(
    format="openvino",
    imgsz=640,
    half=False,
    int8=False
)

print("OpenVINO export completed!")
print("Export location:", openvino_path)

In [ ]:
# Test OpenVINO inference

from ultralytics import YOLO

# Load OpenVINO model
openvino_inference_model = YOLO(str(openvino_path))

# Run prediction on the same test image
openvino_results = openvino_inference_model.predict(
    test_image,
    conf=0.25,
    iou=0.45,
    imgsz=640,
    device="cpu",
    verbose=False
)

print("OpenVINO model loaded successfully!")
print("Total detections:", len(openvino_results[0].boxes))

print("\nDetected objects:")

for box in openvino_results[0].boxes:
    class_id = int(box.cls[0])

    print(
        detection_model.names[class_id],
        "Confidence:",
        round(float(box.conf[0]), 4)
    )

In [ ]:
# Benchmark inference speed

import time
import numpy as np

models = {
    "PyTorch": detection_model,
    "ONNX FP32": fp32_model,
    "OpenVINO FP32": openvino_inference_model
}

benchmark_results = {}

for model_name, model in models.items():

    # Warm-up
    for _ in range(3):
        model.predict(
            test_image,
            conf=0.25,
            imgsz=640,
            device="cpu",
            verbose=False
        )

    # Benchmark
    times = []

    for _ in range(10):
        start = time.perf_counter()

        model.predict(
            test_image,
            conf=0.25,
            imgsz=640,
            device="cpu",
            verbose=False
        )

        times.append(
            (time.perf_counter() - start) * 1000
        )

    benchmark_results[model_name] = {
        "Average (ms)": round(np.mean(times), 2),
        "Minimum (ms)": round(np.min(times), 2),
        "Maximum (ms)": round(np.max(times), 2),
        "FPS": round(1000 / np.mean(times), 2)
    }

import pandas as pd

benchmark_df = pd.DataFrame(benchmark_results).T

display(benchmark_df)

- PyTorch is approximately 13% faster than ONNX.
- PyTorch is approximately 15% faster than OpenVINO.
- ONNX and OpenVINO have very similar performance.
- All three formats show considerable variation between runs.
Current winner: PyTorch FP32.

In [ ]:
# Evaluate OpenVINO on the complete test dataset

from ultralytics import YOLO
from pathlib import Path

# Validate OpenVINO model on cleaned test dataset
openvino_metrics = openvino_inference_model.val(
    data=str(
        BASE / "dataset/detection_cleaned_v1/data_cleaned.yaml"
    ),
    split="test",
    imgsz=640,
    batch=1,
    device="cpu",
    workers=0,
    plots=False
)

print("\nOpenVINO Test Performance")
print("-------------------------")
print("Precision:", round(openvino_metrics.box.mp, 4))
print("Recall:", round(openvino_metrics.box.mr, 4))
print("mAP50:", round(openvino_metrics.box.map50, 4))
print("mAP50-95:", round(openvino_metrics.box.map, 4))

- Precision (49.68%): Approximately 49.68% of the model's evaluated detections were correct.
- Recall (37.55%): The model detected approximately 37.55% of the ground-truth objects under the evaluation settings.
- mAP50 (37.26%): Mean Average Precision at an IoU threshold of 0.50.
- mAP50-95 (22.19%): Average mAP across IoU thresholds from 0.50 to 0.95.

In [ ]:
# Finalize Phase 4 – Performance comparison

# Copy test images to Colab local storage

import shutil
from pathlib import Path

SOURCE_TEST = BASE / "dataset/detection_cleaned_v1/images/test"
LOCAL_TEST = Path("/content/SmartVision_test")

if LOCAL_TEST.exists():
    shutil.rmtree(LOCAL_TEST)

shutil.copytree(SOURCE_TEST, LOCAL_TEST)

test_images = sorted(LOCAL_TEST.glob("*"))

print("Test images copied:", len(test_images))
print("Local test directory:", LOCAL_TEST)
print("First image:", test_images[0].name)

In [ ]:
# Prepare the three models

from ultralytics import YOLO
from pathlib import Path

# Model paths
pytorch_path = BASE / "results/yolo_training/yolov8s_25class_gpu/weights/best.pt"

onnx_path = BASE / "results/model_optimization/yolov8s_25class.onnx"

openvino_path = BASE / "results/yolo_training/yolov8s_25class_gpu/weights/best_openvino_model"

# Load all three models
pytorch_model = YOLO(str(pytorch_path))
onnx_model = YOLO(str(onnx_path))
openvino_model = YOLO(str(openvino_path))

# Verify model files
print("PyTorch model:", pytorch_path.exists())
print("ONNX model:", onnx_path.exists())
print("OpenVINO model:", openvino_path.exists())

print("\nAll three models loaded successfully!")

print("\nModel formats:")
print("1. PyTorch FP32")
print("2. ONNX FP32")
print("3. OpenVINO FP32")

In [ ]:
# Run a controlled inference benchmark

import time
import numpy as np
import pandas as pd

CONF = 0.25
IOU = 0.45
IMG_SIZE = 640

models = {
    "PyTorch FP32": pytorch_model,
    "ONNX FP32": onnx_model,
    "OpenVINO FP32": openvino_model
}

benchmark_results = []

for model_name, model in models.items():

    print(f"\nBenchmarking: {model_name}")

    # Warm-up using first 3 images
    for image_path in test_images[:3]:
        model.predict(
            source=str(image_path),
            conf=CONF,
            iou=IOU,
            imgsz=IMG_SIZE,
            device="cpu",
            verbose=False
        )

    # Measure all 150 images
    inference_times = []

    for image_path in test_images:

        start_time = time.perf_counter()

        model.predict(
            source=str(image_path),
            conf=CONF,
            iou=IOU,
            imgsz=IMG_SIZE,
            device="cpu",
            verbose=False
        )

        end_time = time.perf_counter()

        inference_times.append((end_time - start_time) * 1000)

    avg_time = np.mean(inference_times)
    median_time = np.median(inference_times)
    min_time = np.min(inference_times)
    max_time = np.max(inference_times)
    fps = 1000 / avg_time

    benchmark_results.append({
        "Model": model_name,
        "Average Time (ms)": round(avg_time, 2),
        "Median Time (ms)": round(median_time, 2),
        "Minimum Time (ms)": round(min_time, 2),
        "Maximum Time (ms)": round(max_time, 2),
        "FPS": round(fps, 3)
    })

    print(f"Average inference time: {avg_time:.2f} ms")
    print(f"FPS: {fps:.3f}")

benchmark_df = pd.DataFrame(benchmark_results)

print("\nFINAL PERFORMANCE COMPARISON")
display(benchmark_df)

ONNX FP32 is the fastest in this experiment.

In [ ]:
# Compare model sizes

from pathlib import Path
import pandas as pd

model_files = {
    "PyTorch FP32": pytorch_path,
    "ONNX FP32": onnx_path
}

# Calculate model sizes
size_results = []

for name, path in model_files.items():
    size_mb = path.stat().st_size / (1024 * 1024)

    size_results.append({
        "Model": name,
        "Size (MB)": round(size_mb, 2)
    })

# Calculate OpenVINO directory size
openvino_size = sum(
    file.stat().st_size
    for file in openvino_path.rglob("*")
    if file.is_file()
) / (1024 * 1024)

size_results.append({
    "Model": "OpenVINO FP32",
    "Size (MB)": round(openvino_size, 2)
})

model_size_df = pd.DataFrame(size_results)

display(model_size_df)

# Streamlit

In [ ]:
# Prepare the Image Classification page

from pathlib import Path

BASE = Path("/content/drive/MyDrive/SmartVision_AI")

for path in BASE.rglob("*"):
    if path.is_file() and path.suffix.lower() in [".h5", ".keras", ".pt", ".pth", ".pkl"]:
        print(path)


In [ ]:
import os

drive_root = "/content/drive/MyDrive"

print("Google Drive folders and files:\n")

for item in os.listdir(drive_root):
    full_path = os.path.join(drive_root, item)
    if os.path.isdir(full_path):
        print("📁", item)
    else:
        print("📄", item)

In [ ]:
import os

project_path = "/content/drive/MyDrive/SmartVision_AI"

print("SmartVision_AI contents:\n")

for item in os.listdir(project_path):
    full_path = os.path.join(project_path, item)
    print("📁" if os.path.isdir(full_path) else "📄", item)

In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build

credentials, project = google.auth.default()
drive_service = build("drive", "v3", credentials=credentials)

account = drive_service.about().get(
    fields="user(displayName,emailAddress)"
).execute()

print("Colab Google account:")
print("Name:", account["user"]["displayName"])
print("Email:", account["user"]["emailAddress"])

In [ ]:
# Search Google Drive for every folder named SmartVision_AI or results

query = """
mimeType = 'application/vnd.google-apps.folder'
and trashed = false
and (name = 'SmartVision_AI' or name = 'results')
"""

response = drive_service.files().list(
    q=query,
    fields="files(id, name, parents, webViewLink)",
    pageSize=100
).execute()

for folder in response.get("files", []):
    print("Folder:", folder["name"])
    print("ID:", folder["id"])
    print("Parent IDs:", folder.get("parents", []))
    print("Link:", folder.get("webViewLink"))
    print("-" * 50)

In [ ]:
# Access the exact results folder using its Google Drive ID

results_folder_id = "1rzjjJu7J-WoHcNbQ9Tu99hRECyGRhbQd"

response = drive_service.files().list(
    q=f"'{results_folder_id}' in parents and trashed = false",
    fields="files(id, name, mimeType, size)",
    pageSize=100
).execute()

print("Files inside SmartVision_AI/results:\n")

for item in response.get("files", []):
    print(item["name"], "|", item["mimeType"], "|", item.get("size", "Folder"))

In [ ]:
# Google Drive folder IDs
folder_ids = {
    "evaluation_reports": "FOLDER_ID_TO_FIND",
    "confusion_matrices": "FOLDER_ID_TO_FIND"
}

# Find the actual IDs of both folders
for folder_name in folder_ids:
    query = (
        f"'{results_folder_id}' in parents "
        f"and name = '{folder_name}' "
        f"and trashed = false"
    )

    response = drive_service.files().list(
        q=query,
        fields="files(id, name)"
    ).execute()

    for folder in response.get("files", []):
        print(f"\n{folder_name} contents:")

        contents = drive_service.files().list(
            q=f"'{folder['id']}' in parents and trashed = false",
            fields="files(id, name, mimeType, size)",
            pageSize=100
        ).execute()

        for item in contents.get("files", []):
            print(" -", item["name"], "|", item["mimeType"])

In [ ]:
import json

# Get the evaluation_reports folder ID
query = (
    f"'{results_folder_id}' in parents "
    "and name = 'evaluation_reports' "
    "and trashed = false"
)

folder = drive_service.files().list(
    q=query,
    fields="files(id)"
).execute()["files"][0]

reports_folder_id = folder["id"]

# Retrieve and display both evaluation reports
query = f"'{reports_folder_id}' in parents and trashed = false"

files = drive_service.files().list(
    q=query,
    fields="files(id, name)"
).execute()["files"]

for file in files:
    content = drive_service.files().get_media(
        fileId=file["id"]
    ).execute()

    report = json.loads(content.decode("utf-8"))

    print("\n", "=" * 50)
    print("FILE:", file["name"])
    print("=" * 50)
    print(json.dumps(report, indent=4))

In [ ]:
# Retrieve the original four-model comparison CSV
import pandas as pd
import io

for filename in [
    "four_model_comparison.csv",
    "classification_phase_summary.txt"
]:
    query = (
        f"'{results_folder_id}' in parents "
        f"and name = '{filename}' "
        "and trashed = false"
    )

    result = drive_service.files().list(
        q=query,
        fields="files(id, name)"
    ).execute()

    file = result["files"][0]

    content = drive_service.files().get_media(
        fileId=file["id"]
    ).execute()

    print("\n" + "=" * 55)
    print(filename)
    print("=" * 55)

    if filename.endswith(".csv"):
        df = pd.read_csv(io.BytesIO(content))
        display(df)
    else:
        print(content.decode("utf-8"))